# 04. Kỹ Thuật Trích Xuất Đặc Trưng (Tuần 5–6), Lát Cắt Thời Gian & Hệ Thống Cảnh Báo Sớm
## Trích xuất Đặc trưng Toàn diện, Zero Temporal Leakage & Early Warning (OULAD)
**Dự án**: LMS Tiên tiến và Learning Analytics (Bộ dữ liệu OULAD)  
**Tác giả**: Vũ Hoà Bình (MSV: 202416137)  

---
Notebook này là **một pipeline tự chủ 100% (self-contained)** bao gồm toàn bộ mã nguồn, thực nghiệm mô hình, xuất biểu đồ (Hình 21–30) và bộ **18 kiểm thử tự động (Unit Tests)** cho cả hai giai đoạn:

### Giai đoạn Tuần 5 (76 Đặc trưng SEDL + 4 Lát cắt Snapshot + 92 Thực nghiệm Benchmark):
1. **T5.1 Click-based (Behavioural)**: `total_clicks`, `active_days`, `clicks_per_day`, `daily_click_mean`, `click_std`, `click_cv`, `active_days_in_course`, `active_days_ratio`, `pre_course_clicks`, `has_pre_course_clicks`, `first_click_day`, `last_click_day`, `active_span_days`.
2. **T5.2 Activity-based (20 tài nguyên VLE & Tỷ trọng SEDL)**: Đầy đủ 20 biến `clicks_<activity_type>` (gồm `clicks_glossary`), `cognitive_clicks`, `social_clicks`, các tỷ trọng và độ đa dạng `activity_diversity`, Shannon Entropy `activity_entropy`.
3. **T5.3 Assessment-based (Coursework-only, loại trừ `Exam`)**: `avg_score`, `weighted_avg_score`, `earned_weight_score`, `min_score`, `max_score`, `score_std`, `num_submissions`, `submission_rate`, `avg_days_early`, `late_submissions`, `late_submission_rate`, `first_score`, `tma1_score`, `tma1_days_early`, `submitted_tma1`.
4. **T5.4 Demographics & Context**: `gender`, `region`, `highest_education`, `imd_band`, `age_band`, `num_of_prev_attempts`, `studied_credits`, `disability`, `module_presentation_length`, `date_registration`, `is_repeat_student`, `credit_overload`.
5. **Snapshot & Cohort Modeling**: Lát cắt $T_{\text{cutoff}} \in \{25\%, 50\%, 75\%, 100\%\}$ trên 3 Cohort (`full`, `pre_course_active`, `dynamic_active`).

### Giai đoạn Tuần 6 (Mở rộng 16 Đặc trưng Động học $\rightarrow$ 92 Đặc trưng Catalog, 114 Biến Mã hóa & Cảnh báo Sớm):
6. **Temporal & Decay (8 biến)**: `clicks_week_1_to_4`, `clicks_last_2_weeks`, `clicks_trend`, `engagement_decay`, `days_since_last_activity`, `inactivity_ratio`, `max_inactivity_streak`, `consistency_score`.
7. **Regularity & Burst (6 biến)**: `active_weeks`, `login_regularity`, `weekend_clicks`, `weekend_ratio`, `burst_count`, `burst_ratio`.
8. **Score Trajectory & Pre-course (3 biến danh mục mới/đồng bộ)**: `last_minus_first_score`, `score_trend_slope`, `days_before_start` ($= -\text{date\_registration}$).
9. **Early Warning Threshold Tuning**: Tối ưu ngưỡng quyết định $\tau^*$ **duy nhất trên Validation** cho `LightGBM (Balanced)` trên Cohort `dynamic_active`, đạt $\text{Recall} \ge 0.75$ ngay tại mốc sớm **25%**.

In [1]:
import json
import logging
import sys
import unittest
import warnings
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from lightgbm import LGBMClassifier
from scipy import stats
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder, RobustScaler

warnings.filterwarnings("ignore")

if sys.platform == "win32":
    try:
        sys.stdout.reconfigure(encoding="utf-8")
        sys.stderr.reconfigure(encoding="utf-8")
    except Exception:
        pass

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,
)
logger = logging.getLogger("OULAD_FeatureEngineering")

plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: "%.4f" % x)

PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path.cwd()
PROJECT_DIR = PROJECT_ROOT
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
SNAPSHOTS_DIR = PROCESSED_DIR / "week5_6_snapshots"
WEEK5_PROCESSED_DIR = PROCESSED_DIR / "week5_feature_engineering"
WEEK6_PROCESSED_DIR = PROCESSED_DIR / "week6_advanced_features"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

for _d in [PROCESSED_DIR, SNAPSHOTS_DIR, WEEK5_PROCESSED_DIR, WEEK6_PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

print(f"Project Root : {PROJECT_ROOT}")
print(f"Raw Data Dir : {RAW_DIR}")
print(f"Processed Dir: {PROCESSED_DIR}")
print(f"Snapshots Dir: {SNAPSHOTS_DIR}")
print(f"Figures Dir  : {FIGURES_DIR}")

Project Root : C:\Users\mrore\Project_I
Raw Data Dir : C:\Users\mrore\Project_I\data\raw
Processed Dir: C:\Users\mrore\Project_I\data\processed
Snapshots Dir: C:\Users\mrore\Project_I\data\processed\week5_6_snapshots
Figures Dir  : C:\Users\mrore\Project_I\reports\figures


## 1. Động cơ Trích xuất Đặc trưng `OULADFeatureEngineer` & Kiểm soát Zero Leakage
Cài đặt đầy đủ lớp `OULADFeatureEngineer` (tích hợp đặc trưng Tuần 5 & Tuần 6 vectorized theo lát cắt thời gian $T_{\text{cutoff}}$), hàm tính độ lớn hiệu ứng `cohens_d`, và hàm chia tập `train_test_split_group`.

In [2]:
class OULADFeatureEngineer:
    """
    Bộ trích xuất đặc trưng chuẩn hóa Tuần 5 cho Tập dữ liệu OULAD.
    Đảm bảo Zero Temporal Leakage: tại bất kỳ lát cắt thời gian cutoff_pct nào,
    chỉ sử dụng log tương tác VLE có date <= cutoff_day và bài nộp Coursework
    (assessment_type != 'Exam') có date_submitted <= cutoff_day.
    """

    KEYS = ["code_module", "code_presentation", "id_student"]
    COURSE_KEYS = ["code_module", "code_presentation"]

    # Toàn bộ 20 loại activity_type chính thức trong bảng vle.csv của OULAD
    ALL_ACTIVITY_TYPES = [
        "dataplus",
        "dualpane",
        "externalquiz",
        "folder",
        "forumng",
        "glossary",
        "homepage",
        "htmlactivity",
        "oucollaborate",
        "oucontent",
        "ouelluminate",
        "ouwiki",
        "page",
        "questionnaire",
        "quiz",
        "repeatactivity",
        "resource",
        "sharedsubpage",
        "subpage",
        "url",
    ]

    # Phân nhóm chỉ báo gián tiếp (Proxies) theo Khung lý thuyết SEDL (Bergdahl et al., 2024; Bond & Bedenlier, 2019)
    COGNITIVE_ACTIVITIES = [
        "oucontent",
        "quiz",
        "externalquiz",
        "glossary",
        "page",
        "resource",
        "url",
        "dataplus",
    ]

    SOCIAL_ACTIVITIES = [
        "forumng",
        "ouwiki",
        "oucollaborate",
        "ouelluminate",
        "sharedsubpage",
    ]

    ORDINAL_MAPPINGS = {
        "highest_education": {
            "No Formal quals": 0,
            "Lower Than A Level": 1,
            "A Level or Equivalent": 2,
            "HE Qualification": 3,
            "Post Graduate Qualification": 4,
        },
        "imd_band": {
            "0-10%": 0,
            "10-20": 1,
            "20-30%": 2,
            "30-40%": 3,
            "40-50%": 4,
            "50-60%": 5,
            "60-70%": 6,
            "70-80%": 7,
            "80-90%": 8,
            "90-100%": 9,
            "Unknown": 4,
        },
        "age_band": {
            "0-35": 0,
            "35-55": 1,
            "55<=": 2,
        },
    }

    NOMINAL_COLS = ["gender", "disability", "region", "code_module"]
    ORDINAL_COLS = ["highest_education", "imd_band", "age_band"]

    def __init__(self, raw_dir: Optional[Path] = None, random_state: int = 42):
        self.raw_dir = raw_dir or RAW_DIR
        self.random_state = random_state

        self.courses: Optional[pd.DataFrame] = None
        self.assessments: Optional[pd.DataFrame] = None
        self.student_info: Optional[pd.DataFrame] = None
        self.student_reg: Optional[pd.DataFrame] = None
        self.student_assess: Optional[pd.DataFrame] = None
        self.vle: Optional[pd.DataFrame] = None
        self.student_vle: Optional[pd.DataFrame] = None

        # Bảng đã tiền kết nối để tăng tốc trích xuất ở nhiều lát cắt thời gian
        self._vle_enriched: Optional[pd.DataFrame] = None
        self._coursework_assessments: Optional[pd.DataFrame] = None
        self._coursework_submissions: Optional[pd.DataFrame] = None
        self._first_tma_per_course: Optional[pd.DataFrame] = None

    def load_raw_tables(self) -> "OULADFeatureEngineer":
        """Đọc 7 bảng CSV gốc của OULAD từ thư mục data/raw/ và chuẩn bị bảng liên kết."""
        logger.info(f"Đang nạp 7 bảng dữ liệu gốc từ {self.raw_dir}...")
        self.courses = pd.read_csv(self.raw_dir / "courses.csv")
        self.assessments = pd.read_csv(self.raw_dir / "assessments.csv")
        self.student_info = pd.read_csv(self.raw_dir / "studentInfo.csv")
        self.student_reg = pd.read_csv(self.raw_dir / "studentRegistration.csv")
        self.student_assess = pd.read_csv(self.raw_dir / "studentAssessment.csv")
        self.vle = pd.read_csv(self.raw_dir / "vle.csv")
        self.student_vle = pd.read_csv(self.raw_dir / "studentVle.csv")

        # Gắn activity_type và thời lượng khóa học vào studentVle
        self._vle_enriched = (
            self.student_vle.merge(
                self.vle[["id_site", "activity_type"]], on="id_site", how="left"
            ).merge(
                self.courses, on=self.COURSE_KEYS, how="left"
            )
        )

        # Lọc riêng Coursework (TMA + CMA), loại bỏ hoàn toàn Exam khỏi đặc trưng đầu vào
        self._coursework_assessments = (
            self.assessments[self.assessments["assessment_type"] != "Exam"]
            .copy()
            .merge(self.courses, on=self.COURSE_KEYS, how="left")
        )

        # Kết nối studentAssessment với Coursework assessments (tự động loại trừ Exam)
        self._coursework_submissions = (
            self.student_assess.merge(
                self._coursework_assessments, on="id_assessment", how="inner"
            )
        )
        self._coursework_submissions["days_early"] = (
            self._coursework_submissions["date"] - self._coursework_submissions["date_submitted"]
        )
        self._coursework_submissions["is_late"] = (
            self._coursework_submissions["days_early"] < 0
        ).astype(int)

        # Xác định bài TMA đầu tiên (TMA1) của mỗi khóa học
        tma_only = self._coursework_assessments[
            self._coursework_assessments["assessment_type"] == "TMA"
        ].sort_values(self.COURSE_KEYS + ["date", "id_assessment"])

        self._first_tma_per_course = (
            tma_only.groupby(self.COURSE_KEYS)
            .first()
            .reset_index()[self.COURSE_KEYS + ["id_assessment", "date"]]
            .rename(columns={"id_assessment": "tma1_id", "date": "tma1_deadline"})
        )

        logger.info(
            f"Đã nạp xong! Enrollments: {len(self.student_info):,}, "
            f"VLE logs: {len(self.student_vle):,}, "
            f"Coursework assessments: {len(self._coursework_assessments)} (đã loại bỏ {len(self.assessments) - len(self._coursework_assessments)} bài Exam)."
        )
        return self

    def extract_snapshot_features(self, cutoff_pct: float = 1.0) -> pd.DataFrame:
        """
        Trích xuất ma trận đặc trưng đầy đủ tại mốc thời gian cutoff_pct (0.25, 0.50, 0.75, 1.00).
        Quy tắc Zero Temporal Leakage:
        - Chỉ lấy log VLE có date <= module_presentation_length * cutoff_pct.
        - Chỉ lấy bài nộp Coursework (assessment_type != 'Exam') có date_submitted <= module_presentation_length * cutoff_pct.
        """
        if self.student_info is None:
            self.load_raw_tables()

        if not (0.0 < cutoff_pct <= 1.0):
            raise ValueError(f"cutoff_pct phải nằm trong (0, 1.0], nhận được: {cutoff_pct}")

        logger.info(f"Đang trích xuất đặc trưng tại lát cắt thời gian cutoff_pct = {cutoff_pct:.2f} ({int(cutoff_pct*100)}%)...")

        # =====================================================================
        # NHÓM 5.4: DEMOGRAPHICS & CONTEXT (studentInfo + studentRegistration + courses)
        # =====================================================================
        df = self.student_info.copy()
        df["at_risk"] = df["final_result"].isin(["Fail", "Withdrawn"]).astype(int)
        df["binary_label"] = df["at_risk"].map({0: "Not At-Risk (0)", 1: "At-Risk (1)"})

        df = df.merge(self.courses, on=self.COURSE_KEYS, how="left")
        df = df.merge(
            self.student_reg[self.KEYS + ["date_registration", "date_unregistration"]],
            on=self.KEYS,
            how="left",
        )

        df["cutoff_pct"] = float(cutoff_pct)
        df["cutoff_day"] = df["module_presentation_length"] * float(cutoff_pct)

        # Các đặc trưng bối cảnh học tập dẫn xuất (5.4)
        df["is_repeat_student"] = (df["num_of_prev_attempts"] > 0).astype(int)
        df["credit_overload"] = (df["studied_credits"] > 60).astype(int)
        df["days_registered_before_start"] = -df["date_registration"].fillna(-57.0)
        df["days_before_start"] = df["days_registered_before_start"]
        df["is_late_registration"] = (df["date_registration"].fillna(-57.0) > 0).astype(int)

        # Cờ xác định Cohort động tại mốc dự đoán (phục vụ lọc Cohort, KHÔNG đưa vào X)
        df["is_pre_course_active"] = (
            df["date_unregistration"].isna() | (df["date_unregistration"] >= 0)
        ).astype(int)
        df["is_snapshot_active"] = (
            df["date_unregistration"].isna() | (df["date_unregistration"] > df["cutoff_day"])
        ).astype(int)

        # =====================================================================
        # NHÓM 5.1: CLICK-BASED FEATURES (Behavioural Engagement)
        # =====================================================================
        vle_cut = self._vle_enriched[
            self._vle_enriched["date"] <= (self._vle_enriched["module_presentation_length"] * cutoff_pct)
        ]

        click_basic = (
            vle_cut.groupby(self.KEYS)
            .agg(
                total_clicks=("sum_click", "sum"),
                active_days=("date", "nunique"),
                first_click_day=("date", "min"),
                last_click_day=("date", "max"),
            )
            .reset_index()
        )

        # Tính tổng click theo từng ngày hoạt động của từng sinh viên để tính mean & std chính xác
        daily_clicks = (
            vle_cut.groupby(self.KEYS + ["date"])["sum_click"]
            .sum()
            .reset_index()
        )
        daily_stats = (
            daily_clicks.groupby(self.KEYS)["sum_click"]
            .agg(daily_click_mean="mean", daily_click_std="std")
            .reset_index()
        )
        daily_stats["daily_click_std"] = daily_stats["daily_click_std"].fillna(0.0)

        # Số ngày hoạt động trong kỳ học chính thức (0 <= date <= cutoff_day)
        in_course_vle = vle_cut[vle_cut["date"] >= 0]
        in_course_days = (
            in_course_vle.groupby(self.KEYS)["date"]
            .nunique()
            .reset_index()
            .rename(columns={"date": "active_days_in_course"})
        )

        # Lượt click chuẩn bị trước khai giảng (date < 0)
        pre_vle = vle_cut[vle_cut["date"] < 0]
        pre_clicks = (
            pre_vle.groupby(self.KEYS)["sum_click"]
            .sum()
            .reset_index()
            .rename(columns={"sum_click": "pre_course_clicks"})
        )

        df = (
            df.merge(click_basic, on=self.KEYS, how="left")
            .merge(daily_stats, on=self.KEYS, how="left")
            .merge(in_course_days, on=self.KEYS, how="left")
            .merge(pre_clicks, on=self.KEYS, how="left")
        )

        for col in [
            "total_clicks",
            "active_days",
            "daily_click_mean",
            "daily_click_std",
            "active_days_in_course",
            "pre_course_clicks",
        ]:
            df[col] = df[col].fillna(0.0)

        # Các chỉ số cường độ và mức độ duy trì tương tác (5.1)
        df["clicks_per_day"] = df["total_clicks"] / np.maximum(1.0, df["cutoff_day"])
        df["clicks_per_active_day"] = np.where(
            df["active_days"] > 0, df["total_clicks"] / df["active_days"], 0.0
        )
        df["click_std"] = df["daily_click_std"]
        df["click_cv"] = np.where(
            df["clicks_per_active_day"] > 0,
            df["click_std"] / df["clicks_per_active_day"],
            0.0,
        )
        df["active_days_ratio"] = np.clip(
            df["active_days_in_course"] / np.maximum(1.0, df["cutoff_day"]), 0.0, 1.0
        )
        df["has_pre_course_clicks"] = (df["pre_course_clicks"] > 0).astype(int)
        df["active_span_days"] = np.where(
            df["active_days"] > 0,
            df["last_click_day"] - df["first_click_day"] + 1.0,
            0.0,
        )

        # =====================================================================
        # NHÓM 5.2: ACTIVITY-BASED FEATURES (20 loại VLE & Đa chiều SEDL)
        # =====================================================================
        act_pivot = (
            vle_cut.groupby(self.KEYS + ["activity_type"])["sum_click"]
            .sum()
            .unstack(fill_value=0.0)
            .reset_index()
        )
        for act in self.ALL_ACTIVITY_TYPES:
            if act not in act_pivot.columns:
                act_pivot[act] = 0.0

        act_rename = {act: f"clicks_{act}" for act in self.ALL_ACTIVITY_TYPES}
        act_pivot = act_pivot[self.KEYS + self.ALL_ACTIVITY_TYPES].rename(columns=act_rename)

        df = df.merge(act_pivot, on=self.KEYS, how="left")
        act_click_cols = [f"clicks_{act}" for act in self.ALL_ACTIVITY_TYPES]
        for col in act_click_cols:
            df[col] = df[col].fillna(0.0)

        # Tính độ đa dạng học liệu (activity_diversity) và Shannon Entropy (activity_entropy)
        act_matrix = df[act_click_cols].values.astype(float)
        df["activity_diversity"] = (act_matrix > 0).sum(axis=1).astype(float)

        row_sums = act_matrix.sum(axis=1, keepdims=True)
        with np.errstate(divide="ignore", invalid="ignore"):
            probs = np.where(row_sums > 0, act_matrix / row_sums, 0.0)
            log_probs = np.where(probs > 0, np.log(probs), 0.0)
            entropy = -np.sum(probs * log_probs, axis=1)
        entropy = np.where(np.abs(entropy) < 1e-12, 0.0, entropy)
        df["activity_entropy"] = np.maximum(0.0, entropy)

        # Tổng hợp theo chiều Nhận thức (Cognitive proxies) và Xã hội (Social proxies) của Khung SEDL
        cog_cols = [f"clicks_{a}" for a in self.COGNITIVE_ACTIVITIES]
        soc_cols = [f"clicks_{a}" for a in self.SOCIAL_ACTIVITIES]
        df["cognitive_clicks"] = df[cog_cols].sum(axis=1)
        df["social_clicks"] = df[soc_cols].sum(axis=1)

        total_c = df["total_clicks"].values
        df["oucontent_ratio"] = np.where(total_c > 0, df["clicks_oucontent"] / total_c, 0.0)
        df["forumng_ratio"] = np.where(total_c > 0, df["clicks_forumng"] / total_c, 0.0)
        df["quiz_ratio"] = np.where(total_c > 0, df["clicks_quiz"] / total_c, 0.0)
        df["resource_ratio"] = np.where(total_c > 0, df["clicks_resource"] / total_c, 0.0)
        df["glossary_ratio"] = np.where(total_c > 0, df["clicks_glossary"] / total_c, 0.0)
        df["cognitive_ratio"] = np.where(total_c > 0, df["cognitive_clicks"] / total_c, 0.0)
        df["social_ratio"] = np.where(total_c > 0, df["social_clicks"] / total_c, 0.0)

        # =====================================================================
        # NHÓM 5.3: ASSESSMENT-BASED FEATURES (Coursework Only, Zero Leakage)
        # =====================================================================
        # 1. Tổng số bài coursework của toàn khóa và số bài có hạn nộp <= cutoff_day
        total_cw_counts = (
            self._coursework_assessments.groupby(self.COURSE_KEYS)["id_assessment"]
            .count()
            .reset_index()
            .rename(columns={"id_assessment": "total_course_assessments"})
        )
        due_cw = self._coursework_assessments[
            self._coursework_assessments["date"]
            <= (self._coursework_assessments["module_presentation_length"] * cutoff_pct)
        ]
        due_cw_counts = (
            due_cw.groupby(self.COURSE_KEYS)["id_assessment"]
            .count()
            .reset_index()
            .rename(columns={"id_assessment": "due_course_assessments"})
        )

        df = df.merge(total_cw_counts, on=self.COURSE_KEYS, how="left")
        df = df.merge(due_cw_counts, on=self.COURSE_KEYS, how="left")
        df["total_course_assessments"] = df["total_course_assessments"].fillna(0).astype(int)
        df["due_course_assessments"] = df["due_course_assessments"].fillna(0).astype(int)

        # 2. Lọc các bài coursework thực tế nộp trước hoặc tại mốc cắt (date_submitted <= cutoff_day)
        sub_cut = self._coursework_submissions[
            self._coursework_submissions["date_submitted"]
            <= (self._coursework_submissions["module_presentation_length"] * cutoff_pct)
        ].copy()

        # Đếm tổng số lượt nộp thô (bao gồm cả bài nộp chưa chấm điểm score.isna())
        raw_sub_counts = (
            sub_cut.groupby(self.KEYS)["id_assessment"]
            .count()
            .reset_index()
            .rename(columns={"id_assessment": "num_submissions_raw"})
        )

        # Thống kê trên các bài nộp đã có điểm hợp lệ (score.notna())
        graded_sub = sub_cut[sub_cut["score"].notna()].copy()
        graded_sub["weighted_score_term"] = graded_sub["score"] * graded_sub["weight"]

        assess_agg = (
            graded_sub.groupby(self.KEYS)
            .agg(
                avg_score=("score", "mean"),
                min_score=("score", "min"),
                max_score=("score", "max"),
                score_std=("score", "std"),
                num_submissions=("score", "count"),
                sum_weighted_score=("weighted_score_term", "sum"),
                sum_submitted_weight=("weight", "sum"),
            )
            .reset_index()
        )
        assess_agg["score_std"] = assess_agg["score_std"].fillna(0.0)
        # Với môn GGG (coursework weight = 0%), dùng trung bình cộng đơn giản làm điểm trung bình trọng số
        assess_agg["weighted_avg_score"] = np.where(
            assess_agg["sum_submitted_weight"] > 0,
            assess_agg["sum_weighted_score"] / assess_agg["sum_submitted_weight"],
            assess_agg["avg_score"],
        )
        assess_agg["earned_weight_score"] = assess_agg["sum_weighted_score"] / 100.0
        assess_agg = assess_agg.drop(columns=["sum_weighted_score", "sum_submitted_weight"])

        # Thống kê tính kịp thời (trên mọi bài nộp có hạn nộp xác định)
        timeliness_agg = (
            sub_cut.groupby(self.KEYS)
            .agg(
                avg_days_early=("days_early", "mean"),
                late_submissions=("is_late", "sum"),
            )
            .reset_index()
        )
        timeliness_agg["avg_days_before_deadline"] = timeliness_agg["avg_days_early"]

        # Điểm bài kiểm tra đầu tiên mà sinh viên thực tế nộp (first_score)
        graded_sorted = graded_sub.sort_values(
            self.KEYS + ["date_submitted", "date", "id_assessment"]
        )
        first_sub_df = (
            graded_sorted.groupby(self.KEYS)
            .first()
            .reset_index()[self.KEYS + ["score"]]
            .rename(columns={"score": "first_score"})
        )

        # Điểm và số ngày nộp sớm của bài TMA1 (chỉ tính nếu nộp <= cutoff_day)
        tma1_sub = sub_cut.merge(
            self._first_tma_per_course,
            left_on=self.COURSE_KEYS + ["id_assessment"],
            right_on=self.COURSE_KEYS + ["tma1_id"],
            how="inner",
        )[self.KEYS + ["score", "days_early"]].rename(
            columns={"score": "tma1_score", "days_early": "tma1_days_early"}
        )

        df = (
            df.merge(raw_sub_counts, on=self.KEYS, how="left")
            .merge(assess_agg, on=self.KEYS, how="left")
            .merge(timeliness_agg, on=self.KEYS, how="left")
            .merge(first_sub_df, on=self.KEYS, how="left")
            .merge(tma1_sub, on=self.KEYS, how="left")
        )

        df["num_submissions_raw"] = df["num_submissions_raw"].fillna(0).astype(int)
        df["num_submissions"] = df["num_submissions"].fillna(0).astype(int)
        df["num_graded_submissions"] = df["num_submissions"]
        df["late_submissions"] = df["late_submissions"].fillna(0).astype(int)
        df["score_std"] = df["score_std"].fillna(0.0)
        df["earned_weight_score"] = df["earned_weight_score"].fillna(0.0)

        # Tỷ lệ nộp bài dựa trên số bài đã đến hạn tại mốc cắt (hoặc tổng bài coursework)
        denom_due = np.maximum(1, df["due_course_assessments"])
        df["submission_rate"] = np.where(
            df["due_course_assessments"] > 0,
            np.clip(df["num_submissions"] / denom_due, 0.0, 1.0),
            np.clip(df["num_submissions"] / np.maximum(1, df["total_course_assessments"]), 0.0, 1.0),
        )
        df["course_progress_submission_rate"] = np.clip(
            df["num_submissions"] / np.maximum(1, df["total_course_assessments"]), 0.0, 1.0
        )

        df["late_submission_rate"] = np.where(
            df["num_submissions_raw"] > 0,
            np.clip(df["late_submissions"] / df["num_submissions_raw"], 0.0, 1.0),
            0.0,
        )

        df["submitted_tma1"] = df["tma1_score"].notna().astype(int)
        df["avg_score_filled"] = df["avg_score"].fillna(0.0)
        df["weighted_avg_score_filled"] = df["weighted_avg_score"].fillna(0.0)
        df["first_score_filled"] = df["first_score"].fillna(0.0)
        df["tma1_score_filled"] = df["tma1_score"].fillna(0.0)

        # =====================================================================
        # NHÓM TUẦN 6: TEMPORAL, REGULARITY & SCORE TRAJECTORY (16 đặc trưng)
        # =====================================================================
        df = self._add_week6_temporal_regularity_features(
            df=df,
            vle_cut=vle_cut,
            daily_clicks=daily_clicks,
            in_course_vle=in_course_vle,
            cutoff_pct=cutoff_pct,
        )
        df = self._add_week6_score_trajectory_features(
            df=df,
            graded_sorted=graded_sorted,
        )

        return df

    def _add_week6_temporal_regularity_features(
        self,
        df: pd.DataFrame,
        vle_cut: pd.DataFrame,
        daily_clicks: pd.DataFrame,
        in_course_vle: pd.DataFrame,
        cutoff_pct: float,
    ) -> pd.DataFrame:
        """Trích xuất 14 đặc trưng Temporal, Inactivity Gap, Regularity, Weekend & Burst (Nhóm 6.1 & 6.2)."""
        cutoff_series = in_course_vle["module_presentation_length"] * cutoff_pct
        w1_4_vle = in_course_vle[in_course_vle["date"] <= np.minimum(27.0, cutoff_series)]
        w1_4_df = (
            w1_4_vle.groupby(self.KEYS)["sum_click"]
            .sum()
            .reset_index()
            .rename(columns={"sum_click": "clicks_week_1_to_4"})
        )

        vle_cutoff_day = vle_cut["module_presentation_length"] * cutoff_pct
        last_2w_vle = vle_cut[vle_cut["date"] > (vle_cutoff_day - 14.0)]
        last_2w_df = (
            last_2w_vle.groupby(self.KEYS)["sum_click"]
            .sum()
            .reset_index()
            .rename(columns={"sum_click": "clicks_last_2_weeks"})
        )

        # Thống kê tuần trên in_course_vle (0 <= date <= cutoff_day)
        in_course_weekly = (
            in_course_vle.assign(week=(in_course_vle["date"] // 7).astype(int))
            .groupby(self.KEYS + ["week"])["sum_click"]
            .sum()
            .reset_index()
        )
        in_course_weekly["w_clicks"] = in_course_weekly["week"] * in_course_weekly["sum_click"]
        weekly_agg = (
            in_course_weekly.groupby(self.KEYS)
            .agg(
                active_weeks=("week", "nunique"),
                in_course_clicks_total=("sum_click", "sum"),
                sum_w_clicks=("w_clicks", "sum"),
            )
            .reset_index()
        )

        # Consistency score: chuẩn hóa Shannon entropy của phân phối click theo tuần [0, 1]
        in_course_w_probs = in_course_weekly.merge(
            weekly_agg[self.KEYS + ["in_course_clicks_total"]], on=self.KEYS, how="left"
        )
        pos_mask = (in_course_w_probs["sum_click"] > 0) & (in_course_w_probs["in_course_clicks_total"] > 0)
        in_course_w_probs = in_course_w_probs[pos_mask].copy()
        p_w = in_course_w_probs["sum_click"] / in_course_w_probs["in_course_clicks_total"]
        in_course_w_probs["neg_plogp"] = -p_w * np.log(p_w)
        weekly_entropy_df = (
            in_course_w_probs.groupby(self.KEYS)["neg_plogp"]
            .sum()
            .reset_index()
            .rename(columns={"neg_plogp": "_weekly_entropy"})
        )

        # Max inactivity streak trên in_course_vle [0, floor(cutoff_day)]
        unique_days = (
            in_course_vle[self.KEYS + ["date"]]
            .drop_duplicates()
            .sort_values(self.KEYS + ["date"])
        )
        prev_day = unique_days.groupby(self.KEYS)["date"].shift(1)
        unique_days["gap"] = np.where(
            prev_day.isna(),
            unique_days["date"],
            unique_days["date"] - prev_day - 1.0,
        )
        streak_agg = (
            unique_days.groupby(self.KEYS)
            .agg(
                _max_gap=("gap", "max"),
                _last_in_course_day=("date", "max"),
            )
            .reset_index()
        )

        # Weekend clicks (ngày thứ 6 & 7 trong tuần môn học: date % 7 in {5, 6})
        weekend_vle = vle_cut[(vle_cut["date"] % 7).isin([5, 6])]
        weekend_df = (
            weekend_vle.groupby(self.KEYS)["sum_click"]
            .sum()
            .reset_index()
            .rename(columns={"sum_click": "weekend_clicks"})
        )

        # Burst count: số ngày daily_clicks > mean + 2*std (khi active_days >= 3 & std > 0)
        dc_with_stats = daily_clicks.merge(
            df[self.KEYS + ["active_days", "daily_click_mean", "daily_click_std"]],
            on=self.KEYS,
            how="left",
        )
        dc_with_stats["is_burst"] = (
            (dc_with_stats["active_days"] >= 3)
            & (dc_with_stats["daily_click_std"] > 0)
            & (
                dc_with_stats["sum_click"]
                > (dc_with_stats["daily_click_mean"] + 2.0 * dc_with_stats["daily_click_std"])
            )
        ).astype(int)
        burst_df = (
            dc_with_stats.groupby(self.KEYS)["is_burst"]
            .sum()
            .reset_index()
            .rename(columns={"is_burst": "burst_count"})
        )

        df = (
            df.merge(w1_4_df, on=self.KEYS, how="left")
            .merge(last_2w_df, on=self.KEYS, how="left")
            .merge(weekly_agg, on=self.KEYS, how="left")
            .merge(weekly_entropy_df, on=self.KEYS, how="left")
            .merge(streak_agg, on=self.KEYS, how="left")
            .merge(weekend_df, on=self.KEYS, how="left")
            .merge(burst_df, on=self.KEYS, how="left")
        )

        for col in [
            "clicks_week_1_to_4",
            "clicks_last_2_weeks",
            "in_course_clicks_total",
            "sum_w_clicks",
            "_weekly_entropy",
            "weekend_clicks",
        ]:
            df[col] = df[col].fillna(0.0)
        for col in ["active_weeks", "burst_count"]:
            df[col] = df[col].fillna(0).astype(int)

        df["engagement_decay"] = df["clicks_last_2_weeks"] / np.maximum(1.0, df["clicks_week_1_to_4"])

        # OLS slope của clicks theo tuần w in [0, w_cut] (bao gồm tuần 0 click)
        w_cut = np.floor(df["cutoff_day"] / 7.0)
        n_weeks = w_cut + 1.0
        x_bar = w_cut / 2.0
        ss_xx = n_weeks * (n_weeks**2 - 1.0) / 12.0
        df["clicks_trend"] = np.where(
            n_weeks >= 2.0,
            (df["sum_w_clicks"] - x_bar * df["in_course_clicks_total"]) / np.maximum(1e-9, ss_xx),
            0.0,
        )

        df["days_since_last_activity"] = np.where(
            df["last_click_day"].notna(),
            np.maximum(0.0, df["cutoff_day"] - df["last_click_day"]),
            df["cutoff_day"] + 1.0,
        )

        total_in_course_days = np.floor(df["cutoff_day"]) + 1.0
        df["inactivity_ratio"] = np.clip(
            (total_in_course_days - df["active_days_in_course"]) / np.maximum(1.0, total_in_course_days),
            0.0,
            1.0,
        )

        tail_gap = np.floor(df["cutoff_day"]) - df["_last_in_course_day"]
        df["max_inactivity_streak"] = np.where(
            df["_last_in_course_day"].notna(),
            np.maximum(df["_max_gap"].fillna(0.0), np.maximum(0.0, tail_gap.fillna(0.0))),
            total_in_course_days,
        )

        df["consistency_score"] = np.where(
            n_weeks > 1.0,
            np.clip(df["_weekly_entropy"] / np.maximum(1e-9, np.log(n_weeks)), 0.0, 1.0),
            0.0,
        )
        df["login_regularity"] = np.clip(
            df["active_weeks"] / np.maximum(1.0, n_weeks), 0.0, 1.0
        )
        df["weekend_ratio"] = np.where(
            df["total_clicks"] > 0,
            np.clip(df["weekend_clicks"] / df["total_clicks"], 0.0, 1.0),
            0.0,
        )
        df["burst_ratio"] = np.where(
            df["active_days"] > 0,
            np.clip(df["burst_count"] / df["active_days"], 0.0, 1.0),
            0.0,
        )

        return df.drop(
            columns=["in_course_clicks_total", "sum_w_clicks", "_weekly_entropy", "_max_gap", "_last_in_course_day"]
        )

    def _add_week6_score_trajectory_features(
        self,
        df: pd.DataFrame,
        graded_sorted: pd.DataFrame,
    ) -> pd.DataFrame:
        """Trích xuất 2 đặc trưng quỹ đạo điểm số Early Warning (Nhóm 6.3)."""
        if graded_sorted.empty:
            df["last_minus_first_score"] = 0.0
            df["score_trend_slope"] = 0.0
            return df

        g = graded_sorted.copy()
        g["_idx"] = g.groupby(self.KEYS).cumcount().astype(float)
        g["_idx_score"] = g["_idx"] * g["score"]
        traj_agg = (
            g.groupby(self.KEYS)
            .agg(
                _last_score=("score", "last"),
                _n_sub=("score", "count"),
                _sum_score=("score", "sum"),
                _sum_idx_score=("_idx_score", "sum"),
            )
            .reset_index()
        )
        df = df.merge(traj_agg, on=self.KEYS, how="left")
        df["last_minus_first_score"] = np.where(
            df["_n_sub"].fillna(0) >= 2,
            df["_last_score"] - df["first_score"],
            0.0,
        )

        m = df["_n_sub"].fillna(0).astype(float)
        x_bar = (m - 1.0) / 2.0
        ss_xx = m * (m**2 - 1.0) / 12.0
        df["score_trend_slope"] = np.where(
            m >= 2.0,
            (df["_sum_idx_score"].fillna(0.0) - x_bar * df["_sum_score"].fillna(0.0)) / np.maximum(1e-9, ss_xx),
            0.0,
        )
        return df.drop(columns=["_last_score", "_n_sub", "_sum_score", "_sum_idx_score"])

    @staticmethod
    def filter_cohort(df: pd.DataFrame, cohort_type: str = "full") -> pd.DataFrame:
        """
        Lọc tập dữ liệu theo 3 định nghĩa Cohort tại mỗi mốc dự đoán:
        - 'full': Toàn bộ 32,593 lượt đăng ký.
        - 'pre_course_active': Loại bỏ 2,678 lượt đăng ký đã rút môn trước ngày khai giảng (date_unregistration < 0).
        - 'dynamic_active': Chỉ giữ các sinh viên còn đang theo học tại thời điểm dự đoán
                            (date_unregistration.isna() hoặc date_unregistration > cutoff_day).
        """
        if cohort_type == "full":
            return df.copy().reset_index(drop=True)
        elif cohort_type == "pre_course_active":
            mask = df["date_unregistration"].isna() | (df["date_unregistration"] >= 0)
            return df[mask].copy().reset_index(drop=True)
        elif cohort_type == "dynamic_active":
            mask = df["date_unregistration"].isna() | (df["date_unregistration"] > df["cutoff_day"])
            return df[mask].copy().reset_index(drop=True)
        else:
            raise ValueError(f"cohort_type không hợp lệ: {cohort_type}")

    @classmethod
    def get_feature_sets(cls) -> Dict[str, List[str]]:
        """
        Trả về danh sách các cột đặc trưng đầu vào (chưa One-Hot) cho:
        1. 'week3_4_baseline': Bộ đặc trưng cơ sở Tuần 3-4 (đã sửa lỗi loại bỏ Exam).
        2. 'week5_expanded': Bộ đặc trưng mở rộng toàn diện Tuần 5 (bao gồm toàn bộ 20 loại VLE,
                             activity_diversity, activity_entropy, weighted_avg_score, first_score, ...).
        """
        baseline_num_cols = [
            # 5.1 Click-based cơ sở
            "total_clicks",
            "active_days",
            "daily_click_mean",
            "daily_click_std",
            "clicks_per_active_day",
            "pre_course_clicks",
            "has_pre_course_clicks",
            # 5.2 Activity-based cơ sở (8 loại)
            "clicks_oucontent",
            "clicks_forumng",
            "clicks_quiz",
            "clicks_homepage",
            "clicks_subpage",
            "clicks_resource",
            "clicks_ouwiki",
            "clicks_url",
            "oucontent_ratio",
            "forumng_ratio",
            "quiz_ratio",
            # 5.3 Assessment-based cơ sở
            "total_course_assessments",
            "avg_score",
            "min_score",
            "max_score",
            "num_submissions",
            "avg_days_early",
            "late_submissions",
            "tma1_score",
            "tma1_days_early",
            "submission_rate",
            "submitted_tma1",
            "late_submission_rate",
            # 5.4 Demographics & Context số
            "num_of_prev_attempts",
            "studied_credits",
            "module_presentation_length",
            "date_registration",
        ]

        all_20_act_cols = [f"clicks_{act}" for act in cls.ALL_ACTIVITY_TYPES]

        expanded_num_cols = [
            # 5.1 Click-based đầy đủ
            "total_clicks",
            "active_days",
            "clicks_per_day",
            "clicks_per_active_day",
            "click_std",
            "click_cv",
            "active_days_in_course",
            "active_days_ratio",
            "pre_course_clicks",
            "has_pre_course_clicks",
            "active_span_days",
            # 5.2 Activity-based đầy đủ (20 loại + SEDL proxies + Diversity & Shannon Entropy)
            *all_20_act_cols,
            "cognitive_clicks",
            "social_clicks",
            "oucontent_ratio",
            "forumng_ratio",
            "quiz_ratio",
            "resource_ratio",
            "glossary_ratio",
            "cognitive_ratio",
            "social_ratio",
            "activity_diversity",
            "activity_entropy",
            # 5.3 Assessment-based đầy đủ (Coursework only)
            "total_course_assessments",
            "due_course_assessments",
            "avg_score",
            "weighted_avg_score",
            "earned_weight_score",
            "min_score",
            "max_score",
            "score_std",
            "first_score",
            "num_submissions",
            "num_submissions_raw",
            "submission_rate",
            "course_progress_submission_rate",
            "avg_days_before_deadline",
            "late_submissions",
            "late_submission_rate",
            "tma1_score",
            "tma1_days_early",
            "submitted_tma1",
            # 5.4 Demographics & Context đầy đủ
            "num_of_prev_attempts",
            "studied_credits",
            "module_presentation_length",
            "date_registration",
            "is_repeat_student",
            "credit_overload",
            "is_late_registration",
        ]

        advanced_num_cols = [
            ("days_before_start" if c == "date_registration" else c)
            for c in expanded_num_cols
        ] + [
            # 6.1 Temporal & Decay Features (8 đặc trưng)
            "clicks_week_1_to_4",
            "clicks_last_2_weeks",
            "clicks_trend",
            "engagement_decay",
            "days_since_last_activity",
            "inactivity_ratio",
            "max_inactivity_streak",
            "consistency_score",
            # 6.2 Regularity, Weekend & Burst Patterns (6 đặc trưng)
            "active_weeks",
            "login_regularity",
            "weekend_clicks",
            "weekend_ratio",
            "burst_count",
            "burst_ratio",
            # 6.3 Score Trajectory Early Warning (2 đặc trưng)
            "last_minus_first_score",
            "score_trend_slope",
        ]

        return {
            "week3_4_baseline": baseline_num_cols,
            "week5_expanded": expanded_num_cols,
            "week6_advanced": advanced_num_cols,
        }

    def prepare_ml_matrices(
        self,
        train_df: pd.DataFrame,
        val_df: pd.DataFrame,
        test_df: pd.DataFrame,
        feature_set: str = "week5_expanded",
    ) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        """
        Tiền xử lý tập Train, Validation và Test đảm bảo 100% Zero Data Leakage:
        - Học tham số Imputation, Outlier Capping (Q3 + 3*IQR), OneHotEncoder và RobustScaler CHỈ TỪ TẬP TRAIN.
        - Trả về 3 DataFrame số học sạch sẽ (0 NaN, 0 Inf) sẵn sàng cho huấn luyện mô hình.
        """
        num_cols = self.get_feature_sets()[feature_set]

        # Học tham số từ Train
        imd_mode = (
            train_df["imd_band"].mode().iloc[0]
            if not train_df["imd_band"].mode().empty
            else "20-30%"
        )
        reg_median = float(train_df["date_registration"].median())

        # Các cột lệch phải cần Capping + Log1p (loại trừ clicks_trend vì có thể âm)
        skewed_cols = [
            c
            for c in num_cols
            if (
                (c.startswith("clicks_") and c != "clicks_trend")
                or c
                in [
                    "total_clicks",
                    "active_days",
                    "active_days_in_course",
                    "daily_click_mean",
                    "daily_click_std",
                    "click_std",
                    "pre_course_clicks",
                    "cognitive_clicks",
                    "social_clicks",
                    "weekend_clicks",
                    "studied_credits",
                ]
            )
        ]

        bounds = {}
        for col in skewed_cols:
            s = train_df[col].dropna()
            q25, q75 = float(s.quantile(0.25)), float(s.quantile(0.75))
            iqr = q75 - q25
            if iqr > 0:
                bounds[col] = (0.0, q75 + 3.0 * iqr)

        ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
        ohe.fit(train_df[self.NOMINAL_COLS].fillna("Missing").astype(str))
        ohe_names = list(ohe.get_feature_names_out(self.NOMINAL_COLS))

        def _clean_subset(df_in: pd.DataFrame) -> pd.DataFrame:
            d = df_in.copy()
            d["imd_was_missing"] = d["imd_band"].isna().astype(float)
            d["has_no_submissions"] = (d["num_submissions"].fillna(0) == 0).astype(float)
            d["has_no_clicks"] = (d["total_clicks"].fillna(0) == 0).astype(float)

            d["imd_band"] = d["imd_band"].fillna(imd_mode)
            d["date_registration"] = d["date_registration"].fillna(reg_median)
            d["days_before_start"] = -d["date_registration"]

            score_cols = [
                "avg_score",
                "weighted_avg_score",
                "earned_weight_score",
                "min_score",
                "max_score",
                "score_std",
                "first_score",
                "tma1_score",
            ]
            for sc in score_cols:
                if sc in d.columns:
                    d[sc] = d[sc].fillna(0.0)

            early_cols = ["avg_days_early", "avg_days_before_deadline", "tma1_days_early"]
            for ec in early_cols:
                if ec in d.columns:
                    d[ec] = d[ec].fillna(-10.0)

            for c in num_cols:
                if c in d.columns:
                    d[c] = d[c].fillna(0.0)

            for col, mapping in self.ORDINAL_MAPPINGS.items():
                d[col] = d[col].map(mapping).fillna(mapping.get("Unknown", 2)).astype(float)

            for col, (low, high) in bounds.items():
                if col in d.columns:
                    d[col] = d[col].clip(lower=low, upper=high)

            for col in skewed_cols:
                if col in d.columns and "click" in col and "ratio" not in col and "per" not in col and "cv" not in col:
                    d[col] = np.log1p(np.maximum(0.0, d[col]))

            return d

        tr_clean = _clean_subset(train_df)
        va_clean = _clean_subset(val_df)
        te_clean = _clean_subset(test_df)

        cols_to_scale = num_cols + self.ORDINAL_COLS
        scaler = RobustScaler()
        scaler.fit(tr_clean[cols_to_scale])

        def _assemble(df_clean: pd.DataFrame, df_orig: pd.DataFrame) -> pd.DataFrame:
            scaled_arr = scaler.transform(df_clean[cols_to_scale])
            scaled_df = pd.DataFrame(scaled_arr, columns=cols_to_scale, index=df_orig.index)
            ohe_arr = ohe.transform(df_orig[self.NOMINAL_COLS].fillna("Missing").astype(str))
            ohe_df = pd.DataFrame(ohe_arr, columns=ohe_names, index=df_orig.index)
            flags_df = df_clean[["imd_was_missing", "has_no_submissions", "has_no_clicks"]].copy()
            out = pd.concat([scaled_df, ohe_df, flags_df], axis=1)
            return out.replace([np.inf, -np.inf], 0.0).fillna(0.0)

        return (
            _assemble(tr_clean, train_df),
            _assemble(va_clean, val_df),
            _assemble(te_clean, test_df),
        )


def cohens_d(group0: np.ndarray, group1: np.ndarray) -> float:
    """Tính độ lớn hiệu ứng Cohen's d giữa nhóm Not At-Risk (0) và At-Risk (1)."""
    g0 = group0[~np.isnan(group0)]
    g1 = group1[~np.isnan(group1)]
    n0, n1 = len(g0), len(g1)
    if n0 < 2 or n1 < 2:
        return 0.0
    v0, v1 = np.var(g0, ddof=1), np.var(g1, ddof=1)
    pooled = np.sqrt(((n0 - 1) * v0 + (n1 - 1) * v1) / (n0 + n1 - 2))
    if pooled == 0:
        return 0.0
    return float((np.mean(g0) - np.mean(g1)) / pooled)


def train_test_split_group(
    df: pd.DataFrame, split_strategy: str = "group_stratified", random_state: int = 42
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Chia tập dữ liệu 70/15/15 theo Stratified Random hoặc Group-Stratified (id_student)."""
    if split_strategy == "stratified":
        tr, temp = train_test_split(
            df, test_size=0.30, stratify=df["at_risk"], random_state=random_state
        )
        va, te = train_test_split(
            temp, test_size=0.50, stratify=temp["at_risk"], random_state=random_state
        )
        return (
            tr.copy().reset_index(drop=True),
            va.copy().reset_index(drop=True),
            te.copy().reset_index(drop=True),
        )
    elif split_strategy == "group_stratified":
        sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=random_state)
        splits = list(sgkf.split(df, df["at_risk"], groups=df["id_student"]))
        te_idx = splits[0][1]
        va_idx = splits[1][1]
        tr_idx = np.setdiff1d(np.arange(len(df)), np.concatenate([te_idx, va_idx]))
        return (
            df.iloc[tr_idx].copy().reset_index(drop=True),
            df.iloc[va_idx].copy().reset_index(drop=True),
            df.iloc[te_idx].copy().reset_index(drop=True),
        )
    else:
        raise ValueError(f"split_strategy không hợp lệ: {split_strategy}")

print('Đã khởi tạo thành công lớp OULADFeatureEngineer, cohens_d và train_test_split_group!')

Đã khởi tạo thành công lớp OULADFeatureEngineer, cohens_d và train_test_split_group!


## 2. Trích xuất 4 Lát cắt Thời gian (`25%`, `50%`, `75%`, `100%`)
Trích xuất toàn bộ 32,593 bản ghi đăng ký tại từng mốc thời lượng khóa học (`date <= cutoff_day` và `date_submitted <= cutoff_day`, loại trừ tuyệt đối 24 bài thi `Exam`) và lưu duy nhất 4 tệp snapshot chuẩn vào `data/processed/week5_6_snapshots/`.

In [3]:
engineer = OULADFeatureEngineer(raw_dir=RAW_DIR, random_state=42)
engineer.load_raw_tables()

cutoffs = [0.25, 0.50, 0.75, 1.00]
snapshots: Dict[float, pd.DataFrame] = {}

for pct in cutoffs:
    df_snap = engineer.extract_snapshot_features(cutoff_pct=pct)
    snapshots[pct] = df_snap
    out_csv = SNAPSHOTS_DIR / f"features_snapshot_{int(pct * 100)}pct.csv"
    df_snap.to_csv(out_csv, index=False)
    print(f"Đã lưu {out_csv.name}: {df_snap.shape[0]:,} dòng x {df_snap.shape[1]} cột")

2026-11-24 09:58:04,345 [INFO] Đang nạp 7 bảng dữ liệu gốc từ C:\Users\mrore\Project_I\data\raw...


2026-11-24 09:58:11,104 [INFO] Đã nạp xong! Enrollments: 32,593, VLE logs: 10,655,280, Coursework assessments: 182 (đã loại bỏ 24 bài Exam).


2026-11-24 09:58:11,105 [INFO] Đang trích xuất đặc trưng tại lát cắt thời gian cutoff_pct = 0.25 (25%)...


Đã lưu features_snapshot_25pct.csv: 32,593 dòng x 113 cột
2026-11-24 09:58:17,574 [INFO] Đang trích xuất đặc trưng tại lát cắt thời gian cutoff_pct = 0.50 (50%)...


Đã lưu features_snapshot_50pct.csv: 32,593 dòng x 113 cột
2026-11-24 09:58:25,857 [INFO] Đang trích xuất đặc trưng tại lát cắt thời gian cutoff_pct = 0.75 (75%)...


Đã lưu features_snapshot_75pct.csv: 32,593 dòng x 113 cột
2026-11-24 09:58:36,532 [INFO] Đang trích xuất đặc trưng tại lát cắt thời gian cutoff_pct = 1.00 (100%)...


Đã lưu features_snapshot_100pct.csv: 32,593 dòng x 113 cột


## 3. Pipeline Tuần 5: Danh mục 76 Đặc trưng, 92 Thực nghiệm Benchmark & Biểu đồ (Hình 21–25)
Đo lường Point-Biserial $r$, Spearman $\rho$, Cohen's $d$, Mutual Information và Random Forest Importance cho 76 đặc trưng Tuần 5, thực nghiệm 92 cấu hình mô hình qua 4 mốc snapshot và 3 Cohort, xuất `feature_catalog_week5.csv`, `feature_importance_week5.json`, `week5_experiment_metrics.json` vào `data/processed/week5_feature_engineering/` và xuất Hình 21–25.

In [4]:
def build_feature_catalog(
    df_100: pd.DataFrame,
    df_25: pd.DataFrame,
    df_50: pd.DataFrame,
    df_75: pd.DataFrame,
    engineer: OULADFeatureEngineer,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Xây dựng bảng danh mục đặc trưng toàn diện (feature_catalog_week5.csv) và
    phân tích Feature Importance tại cả 4 mốc 25%, 50%, 75%, 100%.
    """
    logger.info("Đang tính toán thống kê mô tả, tương quan và Feature Importance cho danh mục đặc trưng Tuần 5...")

    # Định nghĩa metadata cho từng đặc trưng
    feature_meta = [
        # Nhóm 5.1: Click-based
        ("total_clicks", "5.1 Click-based", "Behavioural", r"\sum_{t \le T} \text{sum\_click}_t", "studentVle", "float64", "Tổng số lượt click tính đến mốc cắt T_cutoff"),
        ("active_days", "5.1 Click-based", "Behavioural", r"|\{t \le T : \text{click}_t > 0\}|", "studentVle", "float64", "Tổng số ngày có đăng nhập và tương tác VLE"),
        ("clicks_per_day", "5.1 Click-based", "Behavioural", r"\text{total\_clicks} / \max(1, T_{\text{cutoff}})", "studentVle + courses", "float64", "Trung bình số lượt click trên mỗi ngày lịch khóa học"),
        ("clicks_per_active_day", "5.1 Click-based", "Behavioural", r"\text{total\_clicks} / \text{active\_days}", "studentVle", "float64", "Cường độ click trung bình trong những ngày có hoạt động (daily_click_mean)"),
        ("click_std", "5.1 Click-based", "Behavioural", r"\sqrt{\frac{1}{D-1}\sum_{d=1}^D (c_d - \bar{c})^2}", "studentVle", "float64", "Độ lệch chuẩn số lượt click mỗi ngày trên các ngày hoạt động (daily_click_std)"),
        ("click_cv", "5.1 Click-based", "Behavioural", r"\text{click\_std} / \text{clicks\_per\_active\_day}", "studentVle", "float64", "Hệ số biến thiên (Coefficient of Variation) phản ánh độ thất thường nhịp độ học"),
        ("active_days_in_course", "5.1 Click-based", "Behavioural", r"|\{0 \le t \le T : \text{click}_t > 0\}|", "studentVle", "float64", "Số ngày hoạt động tính từ ngày khai giảng chính thức (date >= 0)"),
        ("active_days_ratio", "5.1 Click-based", "Behavioural", r"\text{active\_days\_in\_course} / T_{\text{cutoff}}", "studentVle + courses", "float64", "Tỷ lệ ngày duy trì hoạt động trên tổng số ngày đã trôi qua của khóa học"),
        ("pre_course_clicks", "5.1 Click-based", "Behavioural / Self-Regulation", r"\sum_{t < 0} \text{sum\_click}_t", "studentVle", "float64", "Tổng số lượt click chuẩn bị trước ngày khai giảng (date < 0)"),
        ("has_pre_course_clicks", "5.1 Click-based", "Behavioural / Self-Regulation", r"\mathbb{I}(\text{pre\_course\_clicks} > 0)", "studentVle", "int64", "Cờ nhị phân: 1 nếu sinh viên có truy cập LMS trước ngày khai giảng"),
        ("first_click_day", "5.1 Click-based", "Behavioural", r"\min \{t \le T : \text{click}_t > 0\}", "studentVle", "float64", "Ngày truy cập hệ thống VLE đầu tiên"),
        ("last_click_day", "5.1 Click-based", "Behavioural", r"\max \{t \le T : \text{click}_t > 0\}", "studentVle", "float64", "Ngày truy cập hệ thống VLE gần nhất tính đến mốc cắt"),
        ("active_span_days", "5.1 Click-based", "Behavioural", r"\text{last\_click\_day} - \text{first\_click\_day} + 1", "studentVle", "float64", "Khoảng trải dài thời gian gắn kết (số ngày từ lần click đầu đến lần cuối)"),
        # Nhóm 5.2: Activity-based (20 loại VLE + SEDL proxies)
        ("clicks_oucontent", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{oucontent})", "studentVle + vle", "float64", "Lượt click đọc bài giảng và tài liệu cốt lõi OU Content"),
        ("clicks_forumng", "5.2 Activity-based", "Social (Proxy)", r"\sum \text{click}(\text{forumng})", "studentVle + vle", "float64", "Lượt click thảo luận trên diễn đàn ForumNG"),
        ("clicks_quiz", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{quiz})", "studentVle + vle", "float64", "Lượt click làm bài trắc nghiệm trực tuyến Quiz"),
        ("clicks_homepage", "5.2 Activity-based", "Behavioural (Navigation)", r"\sum \text{click}(\text{homepage})", "studentVle + vle", "float64", "Lượt click truy cập trang chủ môn học Homepage"),
        ("clicks_subpage", "5.2 Activity-based", "Behavioural (Navigation)", r"\sum \text{click}(\text{subpage})", "studentVle + vle", "float64", "Lượt click điều hướng trang con Subpage"),
        ("clicks_resource", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{resource})", "studentVle + vle", "float64", "Lượt click tải tài liệu tham khảo Resource (PDF/ebook)"),
        ("clicks_ouwiki", "5.2 Activity-based", "Social (Proxy)", r"\sum \text{click}(\text{ouwiki})", "studentVle + vle", "float64", "Lượt click biên tập từ điển chung OU Wiki"),
        ("clicks_url", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{url})", "studentVle + vle", "float64", "Lượt click truy cập liên kết học thuật bên ngoài URL"),
        ("clicks_oucollaborate", "5.2 Activity-based", "Social (Proxy)", r"\sum \text{click}(\text{oucollaborate})", "studentVle + vle", "float64", "Lượt click phòng học trực tuyến đồng bộ OU Collaborate"),
        ("clicks_glossary", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{glossary})", "studentVle + vle", "float64", "Lượt click tra cứu bảng thuật ngữ chuyên ngành Glossary"),
        ("clicks_questionnaire", "5.2 Activity-based", "Emotional / Feedback (Proxy)", r"\sum \text{click}(\text{questionnaire})", "studentVle + vle", "float64", "Lượt click phản hồi phiếu khảo sát học phần Questionnaire"),
        ("clicks_externalquiz", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{externalquiz})", "studentVle + vle", "float64", "Lượt click làm bài kiểm tra mở rộng External Quiz"),
        ("clicks_page", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{page})", "studentVle + vle", "float64", "Lượt click đọc trang thông tin tĩnh Page"),
        ("clicks_dataplus", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{dataplus})", "studentVle + vle", "float64", "Lượt click công cụ thu thập/phân tích dữ liệu DataPlus"),
        ("clicks_ouelluminate", "5.2 Activity-based", "Social (Proxy)", r"\sum \text{click}(\text{ouelluminate})", "studentVle + vle", "float64", "Lượt click hội thảo trực tuyến OU Elluminate"),
        ("clicks_dualpane", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{dualpane})", "studentVle + vle", "float64", "Lượt click giao diện đối chiếu song song DualPane"),
        ("clicks_htmlactivity", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{htmlactivity})", "studentVle + vle", "float64", "Lượt click bài tập tương tác HTML Activity"),
        ("clicks_folder", "5.2 Activity-based", "Behavioural (Navigation)", r"\sum \text{click}(\text{folder})", "studentVle + vle", "float64", "Lượt click mở thư mục tài liệu Folder"),
        ("clicks_sharedsubpage", "5.2 Activity-based", "Social (Proxy)", r"\sum \text{click}(\text{sharedsubpage})", "studentVle + vle", "float64", "Lượt click trang chia sẻ cộng tác Shared Subpage"),
        ("clicks_repeatactivity", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{repeatactivity})", "studentVle + vle", "float64", "Lượt click hoạt động ôn tập lặp lại Repeat Activity"),
        ("cognitive_clicks", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum_{a \in \mathcal{C}} \text{clicks}_a", "studentVle + vle", "float64", "Tổng số click vào nhóm học liệu Nhận thức (oucontent, quiz, glossary, resource, url...)"),
        ("social_clicks", "5.2 Activity-based", "Social (Proxy)", r"\sum_{a \in \mathcal{S}} \text{clicks}_a", "studentVle + vle", "float64", "Tổng số click vào nhóm tương tác Xã hội (forumng, ouwiki, oucollaborate, ouelluminate...)"),
        ("oucontent_ratio", "5.2 Activity-based", "Cognitive (Proxy)", r"\text{clicks\_oucontent} / \text{total\_clicks}", "studentVle + vle", "float64", "Tỷ trọng tương tác đọc nội dung bài giảng OU Content"),
        ("forumng_ratio", "5.2 Activity-based", "Social (Proxy)", r"\text{clicks\_forumng} / \text{total\_clicks}", "studentVle + vle", "float64", "Tỷ trọng tương tác thảo luận diễn đàn ForumNG"),
        ("quiz_ratio", "5.2 Activity-based", "Cognitive (Proxy)", r"\text{clicks\_quiz} / \text{total\_clicks}", "studentVle + vle", "float64", "Tỷ trọng tương tác làm bài kiểm tra trắc nghiệm Quiz"),
        ("resource_ratio", "5.2 Activity-based", "Cognitive (Proxy)", r"\text{clicks\_resource} / \text{total\_clicks}", "studentVle + vle", "float64", "Tỷ trọng tương tác tải tài nguyên tham khảo Resource"),
        ("glossary_ratio", "5.2 Activity-based", "Cognitive (Proxy)", r"\text{clicks\_glossary} / \text{total\_clicks}", "studentVle + vle", "float64", "Tỷ trọng tương tác tra cứu thuật ngữ Glossary"),
        ("cognitive_ratio", "5.2 Activity-based", "Cognitive (Proxy)", r"\text{cognitive\_clicks} / \text{total\_clicks}", "studentVle + vle", "float64", "Tỷ trọng tổng tương tác nhóm học liệu Nhận thức"),
        ("social_ratio", "5.2 Activity-based", "Social (Proxy)", r"\text{social\_clicks} / \text{total\_clicks}", "studentVle + vle", "float64", "Tỷ trọng tổng tương tác nhóm hoạt động Xã hội"),
        ("activity_diversity", "5.2 Activity-based", "Cognitive / Self-Regulation", r"\sum_{k=1}^{20} \mathbb{I}(\text{clicks}_k > 0)", "studentVle + vle", "float64", "Số lượng loại tài nguyên VLE khác nhau mà sinh viên đã khai thác (trên 20 loại)"),
        ("activity_entropy", "5.2 Activity-based", "Cognitive / Self-Regulation", r"-\sum_{k=1}^{20} p_k \ln(p_k)", "studentVle + vle", "float64", "Shannon Entropy đo độ cân bằng và đa dạng phân phối click qua 20 loại học liệu"),
        # Nhóm 5.3: Assessment-based (Coursework Only)
        ("avg_score", "5.3 Assessment-based", "Cognitive (Performance)", r"\frac{1}{M}\sum_{i=1}^M s_i", "studentAssessment + assessments", "float64", "Điểm trung bình cộng các bài Coursework (TMA/CMA) đã nộp tính đến T_cutoff"),
        ("weighted_avg_score", "5.3 Assessment-based", "Cognitive (Performance)", r"\frac{\sum s_i w_i}{\sum w_i}", "studentAssessment + assessments", "float64", "Điểm trung bình có trọng số các bài Coursework đã nộp tính đến T_cutoff"),
        ("earned_weight_score", "5.3 Assessment-based", "Cognitive (Achievement)", r"\sum_{i=1}^M \frac{s_i w_i}{100}", "studentAssessment + assessments", "float64", "Tổng điểm trọng số tích lũy thực tế trên thang 100 của học phần"),
        ("min_score", "5.3 Assessment-based", "Cognitive (Performance)", r"\min_{1 \le i \le M} s_i", "studentAssessment + assessments", "float64", "Điểm số bài Coursework thấp nhất tính đến T_cutoff"),
        ("max_score", "5.3 Assessment-based", "Cognitive (Performance)", r"\max_{1 \le i \le M} s_i", "studentAssessment + assessments", "float64", "Điểm số bài Coursework cao nhất tính đến T_cutoff"),
        ("score_std", "5.3 Assessment-based", "Cognitive (Consistency)", r"\sqrt{\frac{1}{M-1}\sum (s_i - \bar{s})^2}", "studentAssessment + assessments", "float64", "Độ lệch chuẩn điểm số giữa các bài kiểm tra đã nộp"),
        ("first_score", "5.3 Assessment-based", "Cognitive (Early Signal)", r"s_{\arg\min_i \text{date\_submitted}_i}", "studentAssessment + assessments", "float64", "Điểm của bài kiểm tra Coursework đầu tiên mà sinh viên thực tế nộp"),
        ("num_submissions", "5.3 Assessment-based", "Behavioural (Task Completion)", r"|\{i : \text{submitted}_i \le T \land s_i \neq \text{NaN}\}|", "studentAssessment + assessments", "int64", "Số bài Coursework đã nộp và có điểm hợp lệ tính đến T_cutoff (loại bỏ Exam)"),
        ("num_submissions_raw", "5.3 Assessment-based", "Behavioural (Task Completion)", r"|\{i : \text{submitted}_i \le T\}|", "studentAssessment + assessments", "int64", "Tổng số bài Coursework đã nộp tính cả các bài chưa kịp chấm điểm"),
        ("due_course_assessments", "5.3 Assessment-based", "Context (Course Structure)", r"|\{j \in \text{CW} : \text{deadline}_j \le T\}|", "assessments + courses", "int64", "Số bài đánh giá Coursework đã đến hạn nộp tính đến mốc T_cutoff"),
        ("total_course_assessments", "5.3 Assessment-based", "Context (Course Structure)", r"|\{j \in \text{CW}\}|", "assessments", "int64", "Tổng số bài đánh giá Coursework (TMA + CMA) của toàn khóa học"),
        ("submission_rate", "5.3 Assessment-based", "Behavioural (Task Completion)", r"\min(1, \text{num\_submissions} / \max(1, \text{due}))", "studentAssessment + assessments", "float64", "Tỷ lệ hoàn thành bài tập đã đến hạn tính đến T_cutoff (thuộc [0, 1])"),
        ("course_progress_submission_rate", "5.3 Assessment-based", "Behavioural (Task Completion)", r"\text{num\_submissions} / \text{total\_course\_assessments}", "studentAssessment + assessments", "float64", "Tỷ lệ bài tập đã hoàn thành so với tổng khối lượng bài tập cả khóa"),
        ("avg_days_before_deadline", "5.3 Assessment-based", "Behavioural / Self-Regulation", r"\frac{1}{M}\sum (\text{deadline}_i - \text{submitted}_i)", "studentAssessment + assessments", "float64", "Số ngày nộp bài trước hạn chót trung bình (dương = nộp sớm, âm = nộp trễ)"),
        ("late_submissions", "5.3 Assessment-based", "Behavioural / Disengagement", r"\sum \mathbb{I}(\text{submitted}_i > \text{deadline}_i)", "studentAssessment + assessments", "int64", "Tổng số bài Coursework nộp trễ hạn tính đến T_cutoff"),
        ("late_submission_rate", "5.3 Assessment-based", "Behavioural / Disengagement", r"\text{late\_submissions} / \text{num\_submissions\_raw}", "studentAssessment + assessments", "float64", "Tỷ lệ bài nộp trễ trên tổng số bài đã nộp"),
        ("tma1_score", "5.3 Assessment-based", "Cognitive (Early Signal)", r"s_{\text{TMA1}}", "studentAssessment + assessments", "float64", "Điểm số bài Tutor-Marked Assessment đầu tiên (TMA1)"),
        ("tma1_days_early", "5.3 Assessment-based", "Behavioural / Self-Regulation", r"\text{deadline}_{\text{TMA1}} - \text{submitted}_{\text{TMA1}}", "studentAssessment + assessments", "float64", "Số ngày nộp sớm/muộn của bài kiểm tra đầu tiên TMA1"),
        ("submitted_tma1", "5.3 Assessment-based", "Behavioural (Task Completion)", r"\mathbb{I}(s_{\text{TMA1}} \neq \text{NaN})", "studentAssessment + assessments", "int64", "Cờ nhị phân: 1 nếu sinh viên đã nộp và có điểm bài TMA1 tính đến T_cutoff"),
        # Nhóm 5.4: Demographics & Context
        ("gender", "5.4 Demographics & Context", "Demographic", r"\text{Nominal } \{M, F\}", "studentInfo", "category", "Giới tính của sinh viên"),
        ("region", "5.4 Demographics & Context", "Demographic", r"\text{Nominal (13 UK Regions)}", "studentInfo", "category", "Vùng địa lý cư trú tại Vương quốc Anh"),
        ("highest_education", "5.4 Demographics & Context", "Demographic (Prior Knowledge)", r"\text{Ordinal } \{0, 1, 2, 3, 4\}", "studentInfo", "ordinal", "Trình độ học vấn cao nhất khi nhập học"),
        ("imd_band", "5.4 Demographics & Context", "Demographic (Socio-Economic)", r"\text{Ordinal } \{0, \dots, 9\}", "studentInfo", "ordinal", "Chỉ số thiếu thốn đa chiều vùng cư trú (Index of Multiple Deprivation)"),
        ("age_band", "5.4 Demographics & Context", "Demographic", r"\text{Ordinal } \{0\text{-}35, 35\text{-}55, 55\le\}", "studentInfo", "ordinal", "Nhóm độ tuổi của sinh viên"),
        ("disability", "5.4 Demographics & Context", "Demographic", r"\text{Nominal } \{Y, N\}", "studentInfo", "category", "Tình trạng khuyết tật tự khai báo"),
        ("num_of_prev_attempts", "5.4 Demographics & Context", "Academic History", r"\mathbb{N}_0", "studentInfo", "int64", "Số lần đã đăng ký học lại học phần này trước đây"),
        ("studied_credits", "5.4 Demographics & Context", "Academic Load", r"\sum \text{credits}", "studentInfo", "int64", "Tổng số tín chỉ sinh viên đang theo học song song"),
        ("module_presentation_length", "5.4 Demographics & Context", "Course Context", r"L \in [234, 269]", "courses", "int64", "Tổng thời lượng chính thức của khóa học (số ngày từ khai giảng đến thi)"),
        ("date_registration", "5.4 Demographics & Context", "Behavioural (Planning)", r"t_{\text{reg}} < 0", "studentRegistration", "float64", "Ngày đăng ký môn học so với ngày khai giảng (âm = đăng ký sớm)"),
        ("is_repeat_student", "5.4 Demographics & Context", "Academic History", r"\mathbb{I}(\text{num\_of\_prev\_attempts} > 0)", "studentInfo", "int64", "Cờ nhị phân: 1 nếu sinh viên từng học lại môn học này"),
        ("credit_overload", "5.4 Demographics & Context", "Academic Load", r"\mathbb{I}(\text{studied\_credits} > 60)", "studentInfo", "int64", "Cờ nhị phân: 1 nếu sinh viên đăng ký trên 60 tín chỉ tiêu chuẩn"),
        ("is_late_registration", "5.4 Demographics & Context", "Behavioural (Planning)", r"\mathbb{I}(\text{date\_registration} > 0)", "studentRegistration", "int64", "Cờ nhị phân: 1 nếu sinh viên đăng ký muộn sau ngày khai giảng"),
    ]

    # Tính Random Forest Importance & Mutual Information trên các mốc 25%, 50%, 100%
    def _compute_ml_importances(df_snap: pd.DataFrame) -> Tuple[Dict[str, float], Dict[str, float], Dict[str, float]]:
        tr_df, va_df, te_df = train_test_split_group(df_snap, random_state=42)
        X_tr, X_va, _ = engineer.prepare_ml_matrices(tr_df, va_df, te_df, feature_set="week5_expanded")
        y_tr = tr_df["at_risk"].values
        y_va = va_df["at_risk"].values

        rf = RandomForestClassifier(
            n_estimators=150, max_depth=12, random_state=42, n_jobs=-1
        )
        rf.fit(X_tr, y_tr)
        rf_imp = dict(zip(X_tr.columns, rf.feature_importances_))

        # Permutation importance trên tập Validation
        perm = permutation_importance(
            rf, X_va, y_va, n_repeats=5, random_state=42, scoring="roc_auc", n_jobs=-1
        )
        perm_imp = dict(zip(X_va.columns, np.maximum(0.0, perm.importances_mean)))

        # Mutual Information trên mẫu đại diện tập Train (để tính nhanh và chính xác)
        rng = np.random.RandomState(42)
        sample_idx = rng.choice(len(X_tr), size=min(8000, len(X_tr)), replace=False)
        mi_vals = mutual_info_classif(
            X_tr.iloc[sample_idx], y_tr[sample_idx], random_state=42
        )
        mi_imp = dict(zip(X_tr.columns, mi_vals))
        return rf_imp, perm_imp, mi_imp

    rf_100, perm_100, mi_100 = _compute_ml_importances(df_100)
    rf_75, perm_75, mi_75 = _compute_ml_importances(df_75)
    rf_50, perm_50, mi_50 = _compute_ml_importances(df_50)
    rf_25, perm_25, mi_25 = _compute_ml_importances(df_25)

    # Chuẩn bị bảng số để tính tương quan với nhãn at_risk trên mốc 100% và 25%
    def _get_numeric_series(df_s: pd.DataFrame, col: str) -> pd.Series:
        if col in engineer.ORDINAL_MAPPINGS:
            m = engineer.ORDINAL_MAPPINGS[col]
            return df_s[col].map(m).fillna(m.get("Unknown", 2)).astype(float)
        elif col in ["gender", "disability", "region", "code_module"]:
            return pd.Series(pd.factorize(df_s[col])[0], index=df_s.index, dtype=float)
        elif col in ["avg_score", "weighted_avg_score", "min_score", "max_score", "first_score", "tma1_score"]:
            return df_s[col].fillna(0.0).astype(float)
        elif col in ["avg_days_before_deadline", "avg_days_early", "tma1_days_early"]:
            return df_s[col].fillna(-10.0).astype(float)
        else:
            return df_s[col].fillna(0.0).astype(float)

    catalog_rows = []
    y_full_100 = df_100["at_risk"].values
    y_full_75 = df_75["at_risk"].values
    y_full_50 = df_50["at_risk"].values
    y_full_25 = df_25["at_risk"].values

    for fname, fgroup, sedl_dim, formula, source_tbl, dtype_str, desc in feature_meta:
        s100 = _get_numeric_series(df_100, fname)
        s75 = _get_numeric_series(df_75, fname)
        s50 = _get_numeric_series(df_50, fname)
        s25 = _get_numeric_series(df_25, fname)

        vals_0 = s100[y_full_100 == 0].values
        vals_1 = s100[y_full_100 == 1].values

        r_100, p_100 = stats.pointbiserialr(y_full_100, s100.values)
        rho_100, _ = stats.spearmanr(y_full_100, s100.values)
        d_100 = cohens_d(vals_0, vals_1)

        r_75, _ = stats.pointbiserialr(y_full_75, s75.values)
        d_75 = cohens_d(s75[y_full_75 == 0].values, s75[y_full_75 == 1].values)

        r_50, _ = stats.pointbiserialr(y_full_50, s50.values)
        d_50 = cohens_d(s50[y_full_50 == 0].values, s50[y_full_50 == 1].values)

        r_25, _ = stats.pointbiserialr(y_full_25, s25.values)
        d_25 = cohens_d(s25[y_full_25 == 0].values, s25[y_full_25 == 1].values)

        # Ánh xạ importance (với biến nominal thì cộng tổng các cột One-Hot tương ứng)
        def _lookup_imp(imp_dict: Dict[str, float], col_name: str) -> float:
            if col_name in imp_dict:
                return float(imp_dict[col_name])
            prefix = f"{col_name}_"
            matched = [v for k, v in imp_dict.items() if k.startswith(prefix)]
            return float(sum(matched)) if matched else 0.0

        catalog_rows.append(
            {
                "feature_name": fname,
                "feature_group": fgroup,
                "sedl_dimension": sedl_dim,
                "formula_latex": formula,
                "source_table": source_tbl,
                "data_type": dtype_str,
                "description": desc,
                "missing_pct_raw_100": float(df_100[fname].isna().mean() * 100.0) if fname in df_100.columns else 0.0,
                "mean_not_at_risk_100": float(np.mean(vals_0)),
                "sd_not_at_risk_100": float(np.std(vals_0, ddof=1)),
                "median_not_at_risk_100": float(np.median(vals_0)),
                "iqr_not_at_risk_100": float(np.percentile(vals_0, 75) - np.percentile(vals_0, 25)),
                "mean_at_risk_100": float(np.mean(vals_1)),
                "sd_at_risk_100": float(np.std(vals_1, ddof=1)),
                "median_at_risk_100": float(np.median(vals_1)),
                "iqr_at_risk_100": float(np.percentile(vals_1, 75) - np.percentile(vals_1, 25)),
                "point_biserial_r_100": float(0.0 if np.isnan(r_100) else r_100),
                "spearman_rho_100": float(0.0 if np.isnan(rho_100) else rho_100),
                "cohens_d_100": float(d_100),
                "mutual_info_100": _lookup_imp(mi_100, fname),
                "rf_gini_importance_100": _lookup_imp(rf_100, fname),
                "rf_perm_importance_100": _lookup_imp(perm_100, fname),
                "point_biserial_r_25": float(0.0 if np.isnan(r_25) else r_25),
                "cohens_d_25": float(d_25),
                "mutual_info_25": _lookup_imp(mi_25, fname),
                "rf_gini_importance_25": _lookup_imp(rf_25, fname),
                "rf_perm_importance_25": _lookup_imp(perm_25, fname),
                "point_biserial_r_50": float(0.0 if np.isnan(r_50) else r_50),
                "cohens_d_50": float(d_50),
                "mutual_info_50": _lookup_imp(mi_50, fname),
                "rf_gini_importance_50": _lookup_imp(rf_50, fname),
                "rf_perm_importance_50": _lookup_imp(perm_50, fname),
                "point_biserial_r_75": float(0.0 if np.isnan(r_75) else r_75),
                "cohens_d_75": float(d_75),
                "mutual_info_75": _lookup_imp(mi_75, fname),
                "rf_gini_importance_75": _lookup_imp(rf_75, fname),
                "rf_perm_importance_75": _lookup_imp(perm_75, fname),
            }
        )

    catalog_df = pd.DataFrame(catalog_rows)

    importance_summary = {
        "top20_by_rf_100pct": catalog_df.sort_values("rf_gini_importance_100", ascending=False)[
            ["feature_name", "feature_group", "sedl_dimension", "point_biserial_r_100", "cohens_d_100", "mutual_info_100", "rf_gini_importance_100", "rf_perm_importance_100"]
        ].head(20).to_dict(orient="records"),
        "top20_by_rf_75pct": catalog_df.sort_values("rf_gini_importance_75", ascending=False)[
            ["feature_name", "feature_group", "sedl_dimension", "point_biserial_r_75", "cohens_d_75", "mutual_info_75", "rf_gini_importance_75", "rf_perm_importance_75"]
        ].head(20).to_dict(orient="records"),
        "top20_by_rf_50pct": catalog_df.sort_values("rf_gini_importance_50", ascending=False)[
            ["feature_name", "feature_group", "sedl_dimension", "point_biserial_r_50", "cohens_d_50", "mutual_info_50", "rf_gini_importance_50", "rf_perm_importance_50"]
        ].head(20).to_dict(orient="records"),
        "top20_by_rf_25pct": catalog_df.sort_values("rf_gini_importance_25", ascending=False)[
            ["feature_name", "feature_group", "sedl_dimension", "point_biserial_r_25", "cohens_d_25", "mutual_info_25", "rf_gini_importance_25", "rf_perm_importance_25"]
        ].head(20).to_dict(orient="records"),
        "group_importance_share_100pct": (
            catalog_df.groupby("feature_group")["rf_gini_importance_100"].sum().to_dict()
        ),
        "group_importance_share_75pct": (
            catalog_df.groupby("feature_group")["rf_gini_importance_75"].sum().to_dict()
        ),
        "group_importance_share_50pct": (
            catalog_df.groupby("feature_group")["rf_gini_importance_50"].sum().to_dict()
        ),
        "group_importance_share_25pct": (
            catalog_df.groupby("feature_group")["rf_gini_importance_25"].sum().to_dict()
        ),
        "sedl_importance_share_100pct": (
            catalog_df.groupby("sedl_dimension")["rf_gini_importance_100"].sum().to_dict()
        ),
        "sedl_importance_share_75pct": (
            catalog_df.groupby("sedl_dimension")["rf_gini_importance_75"].sum().to_dict()
        ),
        "sedl_importance_share_50pct": (
            catalog_df.groupby("sedl_dimension")["rf_gini_importance_50"].sum().to_dict()
        ),
        "sedl_importance_share_25pct": (
            catalog_df.groupby("sedl_dimension")["rf_gini_importance_25"].sum().to_dict()
        ),
    }

    return catalog_df, importance_summary


def run_snapshot_benchmark_experiments(
    snapshots: Dict[float, pd.DataFrame],
    engineer: OULADFeatureEngineer,
) -> Dict[str, Any]:
    """
    Thực hiện chuỗi thực nghiệm đối sánh toàn diện qua 4 lát cắt (25%, 50%, 75%, 100%):
    1. So sánh Bộ đặc trưng Cơ sở Tuần 3-4 (week3_4_baseline) vs Bộ đặc trưng Mở rộng Tuần 5 (week5_expanded).
    2. So sánh giữa 3 định nghĩa Cohort: Full Cohort (N=32,593), Pre-Course Active Cohort (N=29,915),
       và Dynamic Snapshot Active Cohort (lọc SV đã rút trước hoặc tại mốc cắt).
    3. So sánh dưới cả hai chiến lược chia tập: Group-Stratified Split (chống rò rỉ id_student) và Stratified Split.
    """
    logger.info("Đang chạy thực nghiệm đối sánh mô hình qua 4 mốc thời lượng (25%, 50%, 75%, 100%)...")
    results = []

    models_factory = {
        "Logistic Regression": lambda: LogisticRegression(max_iter=1000, C=1.0, random_state=42),
        "Random Forest": lambda: RandomForestClassifier(
            n_estimators=150, max_depth=12, random_state=42, n_jobs=-1
        ),
        "Gradient Boosting": lambda: GradientBoostingClassifier(
            n_estimators=120, max_depth=4, learning_rate=0.1, random_state=42
        ),
    }

    for pct, df_snap in snapshots.items():
        pct_label = f"{int(pct * 100)}%"
        for cohort_name in ["full", "pre_course_active", "dynamic_active"]:
            df_cohort = engineer.filter_cohort(df_snap, cohort_type=cohort_name)
            n_total = len(df_cohort)
            at_risk_rate = float(df_cohort["at_risk"].mean())

            for split_strat in ["group_stratified", "stratified"]:
                # Để tiết kiệm thời gian nhưng vẫn đầy đủ bảng đối sánh chính:
                # Chạy đầy đủ cả 2 feature sets trên group_stratified (chuẩn chính thức) và full cohort stratified
                feature_sets_to_run = (
                    ["week3_4_baseline", "week5_expanded"]
                    if (split_strat == "group_stratified" or cohort_name == "full")
                    else ["week5_expanded"]
                )

                tr_df, va_df, te_df = train_test_split_group(
                    df_cohort, split_strategy=split_strat, random_state=42
                )
                y_tr = tr_df["at_risk"].values
                y_va = va_df["at_risk"].values
                y_te = te_df["at_risk"].values

                for fset in feature_sets_to_run:
                    X_tr, X_va, X_te = engineer.prepare_ml_matrices(
                        tr_df, va_df, te_df, feature_set=fset
                    )

                    for m_name, m_builder in models_factory.items():
                        # Gradient Boosting chỉ chạy trên cấu hình trọng tâm để tối ưu thời gian
                        if m_name == "Gradient Boosting" and not (
                            split_strat == "group_stratified" and fset == "week5_expanded"
                        ):
                            continue

                        clf = m_builder()
                        clf.fit(X_tr, y_tr)

                        va_prob = clf.predict_proba(X_va)[:, 1]
                        va_pred = (va_prob >= 0.5).astype(int)

                        te_prob = clf.predict_proba(X_te)[:, 1]
                        te_pred = (te_prob >= 0.5).astype(int)

                        row_res = {
                            "cutoff_pct": float(pct),
                            "snapshot": pct_label,
                            "cohort": cohort_name,
                            "cohort_size": int(n_total),
                            "cohort_at_risk_rate": round(at_risk_rate, 4),
                            "split_strategy": split_strat,
                            "feature_set": fset,
                            "num_features_encoded": int(X_tr.shape[1]),
                            "model": m_name,
                            "val_f1": round(float(f1_score(y_va, va_pred)), 4),
                            "val_auc": round(float(roc_auc_score(y_va, va_prob)), 4),
                            "test_acc": round(float(accuracy_score(y_te, te_pred)), 4),
                            "test_bal_acc": round(float(balanced_accuracy_score(y_te, te_pred)), 4),
                            "test_prec": round(float(precision_score(y_te, te_pred, zero_division=0)), 4),
                            "test_rec": round(float(recall_score(y_te, te_pred, zero_division=0)), 4),
                            "test_f1": round(float(f1_score(y_te, te_pred, zero_division=0)), 4),
                            "test_auc": round(float(roc_auc_score(y_te, te_prob)), 4),
                            "test_pr_auc": round(float(average_precision_score(y_te, te_prob)), 4),
                        }
                        results.append(row_res)

    return {"experiments": results}


def generate_week5_figures(
    snapshots: Dict[float, pd.DataFrame],
    catalog_df: pd.DataFrame,
    exp_results: List[Dict[str, Any]],
    figures_dir: Path,
):
    """
    Tạo hệ thống biểu đồ trực quan hóa chất lượng xuất bản (300 DPI) cho Báo cáo Tuần 5:
    - Hình 21 / 21a / 21b: Tổng quan & biểu đồ tách riêng Nhóm 5.1 (21a) và Nhóm 5.4 (21b)
    - Hình 22: 22_activity_diversity_and_entropy_analysis.png (2x2 cho Nhóm 5.2)
    - Hình 23: 23_assessment_trajectory_and_weighted_scores.png (2x2 cho Nhóm 5.3)
    - Hình 24: 24_snapshot_progression_25_50_75_100.png
    - Hình 25: 25_preliminary_feature_importance_ranking.png
    """
    figures_dir.mkdir(parents=True, exist_ok=True)
    sns.set_theme(style="whitegrid", font="sans-serif")
    df_100 = snapshots[1.00]
    df_50 = snapshots[0.50]
    df_25 = snapshots[0.25]

    palette_bin = {"Not At-Risk (0)": "#2b83ba", "At-Risk (1)": "#d7191c"}
    order_4 = ["Distinction", "Pass", "Fail", "Withdrawn"]
    palette_4 = {
        "Distinction": "#27ae60",
        "Pass": "#2980b9",
        "Fail": "#e67e22",
        "Withdrawn": "#c0392b",
    }

    def _plot_g51_active_days_ratio(ax, title: str):
        sns.violinplot(
            data=df_50,
            x="final_result",
            y="active_days_ratio",
            order=order_4,
            palette=palette_4,
            inner="quartile",
            cut=0,
            ax=ax,
        )
        ax.set_title(title, fontsize=11.5, fontweight="bold")
        ax.set_xlabel("Kết quả học tập cuối kỳ (final_result)", fontsize=10.5, fontweight="bold")
        ax.set_ylabel("Tỷ lệ ngày hoạt động trong kỳ (0 -> 1)", fontsize=10.5, fontweight="bold")

    def _plot_g52_cognitive_clicks(ax, title: str):
        df_100_plot = df_100.copy()
        df_100_plot["log_cognitive_clicks"] = np.log1p(df_100_plot["cognitive_clicks"])
        sns.boxplot(
            data=df_100_plot,
            x="final_result",
            y="log_cognitive_clicks",
            order=order_4,
            palette=palette_4,
            showmeans=True,
            ax=ax,
        )
        ax.set_title(title, fontsize=11.5, fontweight="bold")
        ax.set_xlabel("Kết quả học tập cuối kỳ (final_result)", fontsize=10.5, fontweight="bold")
        ax.set_ylabel("log1p(cognitive_clicks)", fontsize=10.5, fontweight="bold")

    def _plot_g53_first_score_kde(ax, title: str):
        sub_25 = df_25[df_25["first_score"].notna()]
        sns.kdeplot(
            data=sub_25,
            x="first_score",
            hue="binary_label",
            palette=palette_bin,
            fill=True,
            common_norm=False,
            alpha=0.4,
            linewidth=2,
            ax=ax,
        )
        ax.set_title(title, fontsize=11.5, fontweight="bold")
        ax.set_xlabel("Điểm bài kiểm tra đầu tiên thực nộp (0 - 100)", fontsize=10.5, fontweight="bold")
        ax.set_ylabel("Mật độ phân phối (KDE)", fontsize=10.5, fontweight="bold")

    def _plot_g54_context_risk(ax, title: str):
        ctx_df = df_100.copy()
        ctx_df["context_group"] = np.select(
            [
                (ctx_df["is_repeat_student"] == 0) & (ctx_df["credit_overload"] == 0),
                (ctx_df["is_repeat_student"] == 0) & (ctx_df["credit_overload"] == 1),
                (ctx_df["is_repeat_student"] == 1) & (ctx_df["credit_overload"] == 0),
                (ctx_df["is_repeat_student"] == 1) & (ctx_df["credit_overload"] == 1),
            ],
            [
                "Học lần đầu &\n<=60 tín chỉ",
                "Học lần đầu &\n>60 tín chỉ (Quá tải)",
                "Học lại (>=1 lần) &\n<=60 tín chỉ",
                "Học lại (>=1 lần) &\n>60 tín chỉ (Kép)",
            ],
            default="Khác",
        )
        ctx_order = [
            "Học lần đầu &\n<=60 tín chỉ",
            "Học lần đầu &\n>60 tín chỉ (Quá tải)",
            "Học lại (>=1 lần) &\n<=60 tín chỉ",
            "Học lại (>=1 lần) &\n>60 tín chỉ (Kép)",
        ]
        ctx_rates = ctx_df.groupby("context_group")["at_risk"].agg(["mean", "count"]).reindex(ctx_order)
        bars = ax.bar(
            ctx_rates.index,
            ctx_rates["mean"] * 100.0,
            color=["#2ecc71", "#f39c12", "#e67e22", "#c0392b"],
            edgecolor="black",
            width=0.55,
        )
        ax.set_title(title, fontsize=11.5, fontweight="bold")
        ax.set_ylabel("Tỷ lệ sinh viên At-Risk (%)", fontsize=10.5, fontweight="bold")
        ax.set_ylim(0, 88)
        for bar, (_, row) in zip(bars, ctx_rates.iterrows()):
            h = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2.0,
                h + 1.8,
                f"{h:.1f}%\n(n={int(row['count']):,})",
                ha="center",
                fontsize=9.5,
                fontweight="bold",
            )

    # =========================================================================
    # HÌNH 21 (Tổng quan 4 nhóm) & HÌNH 21a/21b (Biểu đồ riêng cho Mục 2.1 và Mục 2.4)
    # =========================================================================
    logger.info("Đang vẽ Hình 21 (tổng quan) và 21a / 21b (tách theo từng mục)...")
    fig, axes = plt.subplots(2, 2, figsize=(16, 11))
    _plot_g51_active_days_ratio(axes[0, 0], "(A) Nhóm 5.1 Click-based: Tỷ lệ ngày hoạt động (active_days_ratio - Mốc 50%)")
    _plot_g52_cognitive_clicks(axes[0, 1], "(B) Nhóm 5.2 Activity-based: Tổng click Nhận thức log1p(cognitive_clicks)")
    _plot_g53_first_score_kde(axes[1, 0], "(C) Nhóm 5.3 Assessment-based: Phân phối Điểm bài đầu tiên (first_score - Mốc 25%)")
    _plot_g54_context_risk(axes[1, 1], "(D) Nhóm 5.4 Context: Tỷ lệ At-Risk theo Tiền sử Học lại & Quá tải Tín chỉ")
    plt.tight_layout()
    fig.savefig(figures_dir / "21_feature_engineering_taxonomy_and_distributions.png", dpi=300)
    plt.close(fig)

    fig_21a, ax_21a = plt.subplots(figsize=(8.0, 5.2))
    _plot_g51_active_days_ratio(ax_21a, "Nhóm 5.1 Click-based: Tỷ lệ ngày hoạt động (active_days_ratio - Mốc 50%)")
    plt.tight_layout()
    fig_21a.savefig(figures_dir / "21a_group51_active_days_ratio.png", dpi=300)
    plt.close(fig_21a)

    fig_21b, ax_21b = plt.subplots(figsize=(8.5, 5.2))
    _plot_g54_context_risk(ax_21b, "Nhóm 5.4 Context: Tỷ lệ At-Risk theo Tiền sử Học lại & Quá tải Tín chỉ")
    plt.tight_layout()
    fig_21b.savefig(figures_dir / "21b_group54_context_at_risk_rates.png", dpi=300)
    plt.close(fig_21b)

    # =========================================================================
    # HÌNH 22: Phân tích Nhóm 5.2 (2x2: Diversity, Shannon Entropy, Click Usage & Cognitive Clicks)
    # =========================================================================
    logger.info("Đang vẽ Hình 22: 22_activity_diversity_and_entropy_analysis.png (2x2)")
    fig, axes = plt.subplots(2, 2, figsize=(15.5, 10.8))

    # (a) Số loại activity_type truy cập theo 4 nhóm kết quả
    sns.boxplot(
        data=df_100,
        x="final_result",
        y="activity_diversity",
        order=order_4,
        palette=palette_4,
        showmeans=True,
        ax=axes[0, 0],
    )
    axes[0, 0].set_title(
        "(A) Độ đa dạng học liệu (activity_diversity / 20 loại)",
        fontsize=11.5,
        fontweight="bold",
    )
    axes[0, 0].set_xlabel("Kết quả cuối kỳ", fontsize=10.5, fontweight="bold")
    axes[0, 0].set_ylabel("Số loại activity_type đã truy cập", fontsize=10.5, fontweight="bold")

    # (b) Phân phối Shannon Entropy (activity_entropy) giữa 2 nhóm ở mốc 25% và 100%
    ent_comp = pd.concat(
        [
            df_25[["binary_label", "activity_entropy"]].assign(Snapshot="Mốc 25% (Cảnh báo sớm)"),
            df_100[["binary_label", "activity_entropy"]].assign(Snapshot="Mốc 100% (Toàn khóa)"),
        ],
        ignore_index=True,
    )
    sns.violinplot(
        data=ent_comp,
        x="Snapshot",
        y="activity_entropy",
        hue="binary_label",
        split=True,
        palette=palette_bin,
        inner="quartile",
        cut=0,
        ax=axes[0, 1],
    )
    axes[0, 1].set_title(
        "(B) Shannon Entropy (activity_entropy) ở Mốc 25% vs 100%",
        fontsize=11.5,
        fontweight="bold",
    )
    axes[0, 1].set_xlabel("Lát cắt thời gian (Snapshot)", fontsize=10.5, fontweight="bold")
    axes[0, 1].set_ylabel("Shannon Entropy H = -sum(p_k * ln(p_k))", fontsize=10.5, fontweight="bold")
    axes[0, 1].legend(title="Nhóm mục tiêu", loc="upper left")

    # (c) Tỷ lệ sử dụng (>0 click) của các loại học liệu
    supp_acts = ["oucontent", "forumng", "quiz", "resource", "url", "ouwiki", "glossary", "oucollaborate", "page", "externalquiz"]
    usage_rows = []
    for a in supp_acts:
        col = f"clicks_{a}"
        u0 = (df_100[df_100["at_risk"] == 0][col] > 0).mean() * 100.0
        u1 = (df_100[df_100["at_risk"] == 1][col] > 0).mean() * 100.0
        usage_rows.append({"Activity": a, "Not At-Risk (0)": u0, "At-Risk (1)": u1})
    usage_df = pd.DataFrame(usage_rows).set_index("Activity")
    usage_df.plot(
        kind="barh",
        color=["#2b83ba", "#d7191c"],
        edgecolor="black",
        width=0.75,
        ax=axes[1, 0],
    )
    axes[1, 0].invert_yaxis()
    axes[1, 0].set_title(
        "(C) Tỷ lệ Sinh viên có Truy cập (>0 click) theo Loại Học liệu",
        fontsize=11.5,
        fontweight="bold",
    )
    axes[1, 0].set_xlabel("Tỷ lệ sinh viên có sử dụng (%)", fontsize=10.5, fontweight="bold")
    axes[1, 0].set_ylabel("Loại tài nguyên VLE (activity_type)", fontsize=10.5, fontweight="bold")
    axes[1, 0].set_xlim(0, 108)
    axes[1, 0].legend(loc="lower right")

    # (d) Tổng click Nhận thức log1p(cognitive_clicks)
    _plot_g52_cognitive_clicks(
        axes[1, 1],
        "(D) Nhóm 5.2 Activity-based: Tổng click Nhận thức log1p(cognitive_clicks)",
    )

    plt.tight_layout()
    fig.savefig(figures_dir / "22_activity_diversity_and_entropy_analysis.png", dpi=300)
    plt.close(fig)

    # =========================================================================
    # HÌNH 23: Tiến trình Điểm số Coursework & Điểm bài đầu tiên (Nhóm 5.3 - 2x2)
    # =========================================================================
    logger.info("Đang vẽ Hình 23: 23_assessment_trajectory_and_weighted_scores.png (2x2)")
    fig, axes = plt.subplots(2, 2, figsize=(15.5, 10.8))

    # (a) Tiến trình earned_weight_score trung bình qua 4 mốc 25%, 50%, 75%, 100%
    traj_rows = []
    for p, d_s in snapshots.items():
        for res in order_4:
            sub_r = d_s[d_s["final_result"] == res]
            traj_rows.append(
                {
                    "Snapshot": f"{int(p*100)}%",
                    "final_result": res,
                    "earned_weight_score": sub_r["earned_weight_score"].mean(),
                    "submission_rate": sub_r["submission_rate"].mean() * 100.0,
                }
            )
    traj_df = pd.DataFrame(traj_rows)
    sns.lineplot(
        data=traj_df,
        x="Snapshot",
        y="earned_weight_score",
        hue="final_result",
        hue_order=order_4,
        palette=palette_4,
        marker="o",
        markersize=8,
        linewidth=2.5,
        ax=axes[0, 0],
    )
    axes[0, 0].set_title(
        "(A) Điểm Trọng số Tích lũy (earned_weight_score) qua 4 Mốc",
        fontsize=11.5,
        fontweight="bold",
    )
    axes[0, 0].set_xlabel("Lát cắt thời lượng khóa học", fontsize=10.5, fontweight="bold")
    axes[0, 0].set_ylabel("Điểm tích lũy Coursework trung bình (thang 100)", fontsize=10.5, fontweight="bold")
    axes[0, 0].legend(title="Kết quả cuối kỳ", loc="upper left")

    # (b) So sánh weighted_avg_score_filled qua 4 mốc
    w_comp = pd.concat(
        [
            df_25[["final_result", "weighted_avg_score_filled"]].assign(Snapshot="25%"),
            df_50[["final_result", "weighted_avg_score_filled"]].assign(Snapshot="50%"),
            snapshots[0.75][["final_result", "weighted_avg_score_filled"]].assign(Snapshot="75%"),
            df_100[["final_result", "weighted_avg_score_filled"]].assign(Snapshot="100%"),
        ],
        ignore_index=True,
    )
    sns.barplot(
        data=w_comp,
        x="Snapshot",
        y="weighted_avg_score_filled",
        hue="final_result",
        hue_order=order_4,
        palette=palette_4,
        edgecolor="black",
        errorbar=None,
        ax=axes[0, 1],
    )
    axes[0, 1].set_title(
        "(B) Điểm Trung bình Trọng số (weighted_avg_score) qua 4 Mốc",
        fontsize=11.5,
        fontweight="bold",
    )
    axes[0, 1].set_xlabel("Lát cắt thời lượng khóa học", fontsize=10.5, fontweight="bold")
    axes[0, 1].set_ylabel("Điểm trung bình trọng số (Impute 0 cho SV không nộp)", fontsize=10.5, fontweight="bold")
    axes[0, 1].set_ylim(0, 95)
    axes[0, 1].legend(title="Kết quả cuối kỳ", loc="upper right", fontsize=8.5)

    # (c) Tỷ lệ hoàn thành bài tập đến hạn (submission_rate) qua 4 mốc
    sns.lineplot(
        data=traj_df,
        x="Snapshot",
        y="submission_rate",
        hue="final_result",
        hue_order=order_4,
        palette=palette_4,
        marker="s",
        markersize=8,
        linewidth=2.5,
        ax=axes[1, 0],
    )
    axes[1, 0].set_title(
        "(C) Tỷ lệ Nộp bài Đúng tiến độ (submission_rate) qua 4 Mốc",
        fontsize=11.5,
        fontweight="bold",
    )
    axes[1, 0].set_xlabel("Lát cắt thời lượng khóa học", fontsize=10.5, fontweight="bold")
    axes[1, 0].set_ylabel("Tỷ lệ hoàn thành bài đã đến hạn (%)", fontsize=10.5, fontweight="bold")
    axes[1, 0].set_ylim(0, 105)
    axes[1, 0].legend(title="Kết quả cuối kỳ", loc="lower left")

    # (d) Phân phối Điểm bài kiểm tra đầu tiên (first_score - Mốc 25%)
    _plot_g53_first_score_kde(
        axes[1, 1],
        "(D) Nhóm 5.3 Assessment-based: Phân phối Điểm bài đầu tiên (first_score - Mốc 25%)",
    )

    plt.tight_layout()
    fig.savefig(figures_dir / "23_assessment_trajectory_and_weighted_scores.png", dpi=300)
    plt.close(fig)

    # =========================================================================
    # HÌNH 24: Tiến triển Hiệu năng Dự đoán qua 4 Mốc Snapshot (25%, 50%, 75%, 100%)
    # =========================================================================
    logger.info("Đang vẽ Hình 24: 24_snapshot_progression_25_50_75_100.png")
    exp_df = pd.DataFrame(exp_results)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6.2))

    # (a) Full Cohort (Group-Stratified): So sánh Bộ cơ sở Tuần 3-4 vs Bộ mở rộng Tuần 5 (Random Forest & Logistic Regression)
    sub_full_grp = exp_df[
        (exp_df["cohort"] == "full")
        & (exp_df["split_strategy"] == "group_stratified")
        & (exp_df["model"].isin(["Random Forest", "Logistic Regression"]))
    ].copy()
    sub_full_grp["Config"] = (
        sub_full_grp["model"]
        + " ("
        + sub_full_grp["feature_set"].map(
            {"week3_4_baseline": "Bộ cơ sở Tuần 3-4", "week5_expanded": "Bộ mở rộng Tuần 5"}
        )
        + ")"
    )
    sns.lineplot(
        data=sub_full_grp,
        x="snapshot",
        y="test_auc",
        hue="Config",
        style="Config",
        markers=True,
        markersize=9,
        linewidth=2.5,
        ax=axes[0],
    )
    axes[0].set_title(
        "(A) Test ROC-AUC qua 4 Mốc: Bộ Đặc trưng Tuần 3-4 vs Tuần 5 (Full Cohort, Group Split)",
        fontsize=11,
        fontweight="bold",
    )
    axes[0].set_xlabel("Lát cắt thời gian dự đoán (Snapshot Cutoff)", fontsize=10.5, fontweight="bold")
    axes[0].set_ylabel("Test ROC-AUC", fontsize=10.5, fontweight="bold")
    axes[0].set_ylim(0.86, 0.995)
    axes[0].legend(loc="lower right", fontsize=9)

    # (b) Dynamic Active Cohort vs Pre-Course Active vs Full Cohort (Bộ mở rộng Tuần 5, Random Forest, Group Split)
    sub_cohorts = exp_df[
        (exp_df["split_strategy"] == "group_stratified")
        & (exp_df["feature_set"] == "week5_expanded")
        & (exp_df["model"] == "Random Forest")
    ].copy()
    cohort_labels = {
        "full": "Full Cohort (N=32,593)",
        "pre_course_active": "Pre-Course Active (N=29,915)",
        "dynamic_active": "Dynamic Snapshot Active (SV chưa rút môn)",
    }
    sub_cohorts["Cohort_Label"] = sub_cohorts["cohort"].map(cohort_labels)

    sns.lineplot(
        data=sub_cohorts,
        x="snapshot",
        y="test_auc",
        hue="Cohort_Label",
        marker="o",
        markersize=9,
        linewidth=2.5,
        palette=["#c0392b", "#2980b9", "#27ae60"],
        ax=axes[1],
    )
    for _, r in sub_cohorts.iterrows():
        axes[1].text(
            r["snapshot"],
            r["test_auc"] + 0.004,
            f"{r['test_auc']:.4f}\n(N={r['cohort_size']:,})",
            ha="center",
            fontsize=8.2,
            fontweight="bold",
        )
    axes[1].set_title(
        "(B) Test ROC-AUC trên 3 Định nghĩa Cohort qua 4 Mốc (Random Forest, Group Split)",
        fontsize=11,
        fontweight="bold",
    )
    axes[1].set_xlabel("Lát cắt thời gian dự đoán (Snapshot Cutoff)", fontsize=10.5, fontweight="bold")
    axes[1].set_ylabel("Test ROC-AUC", fontsize=10.5, fontweight="bold")
    axes[1].set_ylim(0.83, 0.998)
    axes[1].legend(title="Định nghĩa Cohort", loc="lower right", fontsize=9)

    plt.tight_layout()
    fig.savefig(figures_dir / "24_snapshot_progression_25_50_75_100.png", dpi=300)
    plt.close(fig)

    # =========================================================================
    # HÌNH 25: Xếp hạng Top 20 Đặc trưng Quan trọng nhất (Mốc 25% vs Mốc 100%)
    # =========================================================================
    logger.info("Đang vẽ Hình 25: 25_preliminary_feature_importance_ranking.png")
    fig, axes = plt.subplots(1, 2, figsize=(17, 8.5))

    group_colors = {
        "5.1 Click-based": "#2980b9",
        "5.2 Activity-based": "#8e44ad",
        "5.3 Assessment-based": "#27ae60",
        "5.4 Demographics & Context": "#e67e22",
    }

    # (a) Top 20 tại mốc Cảnh báo sớm 25%
    top25 = catalog_df.sort_values("rf_gini_importance_25", ascending=False).head(20).iloc[::-1]
    c25 = [group_colors.get(g, "#34495e") for g in top25["feature_group"]]
    bars25 = axes[0].barh(
        top25["feature_name"],
        top25["rf_gini_importance_25"],
        color=c25,
        edgecolor="black",
        alpha=0.88,
    )
    axes[0].set_title(
        "(A) Top 20 Đặc trưng Quan trọng nhất tại Mốc Cảnh báo Sớm 25% (RF Gini & |r|)",
        fontsize=11.5,
        fontweight="bold",
    )
    axes[0].set_xlabel("Random Forest Feature Importance (Mốc 25%)", fontsize=10.5, fontweight="bold")
    axes[0].set_xlim(0, top25["rf_gini_importance_25"].max() * 1.28)
    for bar, (_, r) in zip(bars25, top25.iterrows()):
        w = bar.get_width()
        axes[0].text(
            w + 0.002,
            bar.get_y() + bar.get_height() / 2.0,
            f"{w:.3f} (r={r['point_biserial_r_25']:+.2f})",
            va="center",
            fontsize=8.5,
            fontweight="bold",
        )

    # (b) Top 20 tại mốc Toàn khóa 100%
    top100 = catalog_df.sort_values("rf_gini_importance_100", ascending=False).head(20).iloc[::-1]
    c100 = [group_colors.get(g, "#34495e") for g in top100["feature_group"]]
    bars100 = axes[1].barh(
        top100["feature_name"],
        top100["rf_gini_importance_100"],
        color=c100,
        edgecolor="black",
        alpha=0.88,
    )
    axes[1].set_title(
        "(B) Top 20 Đặc trưng Quan trọng nhất tại Mốc Toàn khóa 100% (RF Gini & |r|)",
        fontsize=11.5,
        fontweight="bold",
    )
    axes[1].set_xlabel("Random Forest Feature Importance (Mốc 100%)", fontsize=10.5, fontweight="bold")
    axes[1].set_xlim(0, top100["rf_gini_importance_100"].max() * 1.28)
    for bar, (_, r) in zip(bars100, top100.iterrows()):
        w = bar.get_width()
        axes[1].text(
            w + 0.002,
            bar.get_y() + bar.get_height() / 2.0,
            f"{w:.3f} (r={r['point_biserial_r_100']:+.2f})",
            va="center",
            fontsize=8.5,
            fontweight="bold",
        )

    # Chú giải màu theo nhóm đặc trưng
    from matplotlib.patches import Patch
    legend_handles = [Patch(facecolor=col, edgecolor="black", label=grp) for grp, col in group_colors.items()]
    fig.legend(
        handles=legend_handles,
        loc="upper center",
        ncol=4,
        frameon=True,
        fontsize=10.5,
        bbox_to_anchor=(0.5, 1.01),
    )

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(figures_dir / "25_preliminary_feature_importance_ranking.png", dpi=300, bbox_inches="tight")
    plt.close(fig)
    logger.info(f"Đã xuất thành công 5 biểu đồ Tuần 5 (Hình 21 -> Hình 25) tại {figures_dir}!")


def run_week5_notebook_stage(
    snapshots: Dict[float, pd.DataFrame],
    engineer: OULADFeatureEngineer,
    save_figures: bool = True,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    catalog_df, importance_summary = build_feature_catalog(
        df_100=snapshots[1.00],
        df_25=snapshots[0.25],
        df_50=snapshots[0.50],
        df_75=snapshots[0.75],
        engineer=engineer,
    )
    catalog_path = WEEK5_PROCESSED_DIR / "feature_catalog_week5.csv"
    catalog_df.to_csv(catalog_path, index=False)
    logger.info(f"Đã lưu bảng danh mục đặc trưng Tuần 5: {catalog_path} ({len(catalog_df)} đặc trưng)")

    imp_json_path = WEEK5_PROCESSED_DIR / "feature_importance_week5.json"
    with open(imp_json_path, "w", encoding="utf-8") as f:
        json.dump(importance_summary, f, indent=2, ensure_ascii=False)

    exp_dict = run_snapshot_benchmark_experiments(snapshots=snapshots, engineer=engineer)

    snapshot_progression_stats = {}
    for pct, d_s in snapshots.items():
        key = f"{int(pct * 100)}pct"
        d_dyn = engineer.filter_cohort(d_s, "dynamic_active")
        snapshot_progression_stats[key] = {
            "full_cohort_size": int(len(d_s)),
            "full_at_risk_rate": round(float(d_s["at_risk"].mean()), 4),
            "pre_course_active_size": int((d_s["is_pre_course_active"] == 1).sum()),
            "dynamic_active_size": int(len(d_dyn)),
            "dynamic_active_at_risk_rate": round(float(d_dyn["at_risk"].mean()), 4),
            "dynamic_active_fail_count": int((d_dyn["final_result"] == "Fail").sum()),
            "dynamic_active_withdrawn_count": int((d_dyn["final_result"] == "Withdrawn").sum()),
            "mean_total_clicks_not_at_risk": round(float(d_s[d_s["at_risk"] == 0]["total_clicks"].mean()), 2),
            "mean_total_clicks_at_risk": round(float(d_s[d_s["at_risk"] == 1]["total_clicks"].mean()), 2),
            "mean_active_days_not_at_risk": round(float(d_s[d_s["at_risk"] == 0]["active_days"].mean()), 2),
            "mean_active_days_at_risk": round(float(d_s[d_s["at_risk"] == 1]["active_days"].mean()), 2),
            "mean_activity_diversity_not_at_risk": round(float(d_s[d_s["at_risk"] == 0]["activity_diversity"].mean()), 2),
            "mean_activity_diversity_at_risk": round(float(d_s[d_s["at_risk"] == 1]["activity_diversity"].mean()), 2),
            "mean_activity_entropy_not_at_risk": round(float(d_s[d_s["at_risk"] == 0]["activity_entropy"].mean()), 4),
            "mean_activity_entropy_at_risk": round(float(d_s[d_s["at_risk"] == 1]["activity_entropy"].mean()), 4),
            "mean_submission_rate_not_at_risk": round(float(d_s[d_s["at_risk"] == 0]["submission_rate"].mean()), 4),
            "mean_submission_rate_at_risk": round(float(d_s[d_s["at_risk"] == 1]["submission_rate"].mean()), 4),
            "mean_earned_weight_not_at_risk": round(float(d_s[d_s["at_risk"] == 0]["earned_weight_score"].mean()), 2),
            "mean_earned_weight_at_risk": round(float(d_s[d_s["at_risk"] == 1]["earned_weight_score"].mean()), 2),
            "mean_weighted_score_filled_not_at_risk": round(float(d_s[d_s["at_risk"] == 0]["weighted_avg_score_filled"].mean()), 2),
            "mean_weighted_score_filled_at_risk": round(float(d_s[d_s["at_risk"] == 1]["weighted_avg_score_filled"].mean()), 2),
        }

    full_metrics_payload = {
        "snapshot_progression_stats": snapshot_progression_stats,
        "importance_summary": importance_summary,
        "experiments": exp_dict["experiments"],
    }

    metrics_path_proc = WEEK5_PROCESSED_DIR / "week5_experiment_metrics.json"
    with open(metrics_path_proc, "w", encoding="utf-8") as f:
        json.dump(full_metrics_payload, f, indent=2, ensure_ascii=False)
    logger.info(f"Đã lưu kết quả thực nghiệm Tuần 5 (chỉ tại data/processed/week5_feature_engineering/): {metrics_path_proc}")

    if save_figures:
        generate_week5_figures(
            snapshots=snapshots,
            catalog_df=catalog_df,
            exp_results=exp_dict["experiments"],
            figures_dir=FIGURES_DIR,
        )

    return catalog_df, full_metrics_payload


df_catalog_w5, w5_metrics = run_week5_notebook_stage(snapshots=snapshots, engineer=engineer, save_figures=True)

print(f"Tổng số đặc trưng Tuần 5: {len(df_catalog_w5)}")
print("\nPhân bổ theo nhóm đặc trưng:")
print(df_catalog_w5["feature_group"].value_counts())
display(df_catalog_w5[["feature_name", "feature_group", "sedl_dimension", "cohens_d_100", "mutual_info_100", "rf_gini_importance_100"]].head(10))

prog_df = pd.DataFrame(w5_metrics["snapshot_progression_stats"]).T
display(prog_df[["full_cohort_size", "dynamic_active_size", "dynamic_active_at_risk_rate", "mean_total_clicks_not_at_risk", "mean_total_clicks_at_risk", "mean_earned_weight_not_at_risk", "mean_earned_weight_at_risk"]])

w5_exp_df = pd.DataFrame(w5_metrics["experiments"])
filtered_w5 = w5_exp_df[(w5_exp_df["cohort"] == "full") & (w5_exp_df["split_strategy"] == "group_stratified") & (w5_exp_df["model"] == "Random Forest")]
print("\nĐối sánh Random Forest (Group-Stratified Split, Full Cohort - Tuần 5):")
display(filtered_w5[["snapshot", "feature_set", "num_features_encoded", "test_acc", "test_prec", "test_rec", "test_f1", "test_auc"]])


2026-11-24 09:58:48,288 [INFO] Đang tính toán thống kê mô tả, tương quan và Feature Importance cho danh mục đặc trưng Tuần 5...


2026-11-24 10:00:09,692 [INFO] Đã lưu bảng danh mục đặc trưng Tuần 5: C:\Users\mrore\Project_I\data\processed\week5_feature_engineering\feature_catalog_week5.csv (76 đặc trưng)


2026-11-24 10:00:09,695 [INFO] Đang chạy thực nghiệm đối sánh mô hình qua 4 mốc thời lượng (25%, 50%, 75%, 100%)...


2026-11-24 10:06:28,339 [INFO] Đã lưu kết quả thực nghiệm Tuần 5 (chỉ tại data/processed/week5_feature_engineering/): C:\Users\mrore\Project_I\data\processed\week5_feature_engineering\week5_experiment_metrics.json


2026-11-24 10:06:28,341 [INFO] Đang vẽ Hình 21 (tổng quan) và 21a / 21b (tách theo từng mục)...


2026-11-24 10:06:30,031 [INFO] Đang vẽ Hình 22: 22_activity_diversity_and_entropy_analysis.png (2x2)


... (5 outputs left out)

,feature_name,feature_group,sedl_dimension,cohens_d_100,mutual_info_100,rf_gini_importance_100
0,total_clicks,5.1 Click-based,Behavioural,1.0859,0.2530,0.0125
1,active_days,5.1 Click-based,Behavioural,1.6078,0.2907,0.0238
2,clicks_per_day,5.1 Click-based,Behavioural,1.0807,0.2490,0.0213
3,clicks_per_active_day,5.1 Click-based,Behavioural,0.5854,0.1295,0.0029
4,click_std,5.1 Click-based,Behavioural,0.5885,0.1156,0.0031
5,click_cv,5.1 Click-based,Behavioural,0.7835,0.1141,0.0029
6,active_days_in_course,5.1 Click-based,Behavioural,1.6480,0.3025,0.0333
7,active_days_ratio,5.1 Click-based,Behavioural,1.6364,0.3126,0.0422
8,pre_course_clicks,5.1 Click-based,Behavioural / Self-Regulation,0.4271,0.0506,0.0026
9,has_pre_course_clicks,5.1 Click-based,Behavioural / Self-Regulation,0.5580,0.0318,0.0006


,full_cohort_size,dynamic_active_size,dynamic_active_at_risk_rate,mean_total_clicks_not_at_risk,mean_total_clicks_at_risk,mean_earned_weight_not_at_risk,mean_earned_weight_at_risk
25pct,32593.0000,26248.0000,0.4139,755.5900,290.5000,15.7600,6.6400
50pct,32593.0000,24601.0000,0.3746,1175.4700,366.3700,36.8900,11.0400
75pct,32593.0000,23159.0000,0.3357,1670.0800,419.1300,57.9000,13.5700
100pct,32593.0000,22522.0000,0.3169,2068.2300,452.4300,65.5700,14.3300



Đối sánh Random Forest (Group-Stratified Split, Full Cohort - Tuần 5):


,snapshot,feature_set,num_features_encoded,test_acc,test_prec,test_rec,test_f1,test_auc
1,25%,week3_4_baseline,64,0.8106,0.8717,0.7518,0.8073,0.8887
3,25%,week5_expanded,98,0.8146,0.8771,0.7547,0.8113,0.8914
24,50%,week3_4_baseline,64,0.8681,0.9153,0.8267,0.8687,0.9368
26,50%,week5_expanded,98,0.8776,0.9233,0.8377,0.8784,0.9436
47,75%,week3_4_baseline,64,0.9053,0.9453,0.8710,0.9066,0.9672
49,75%,week5_expanded,98,0.9184,0.9581,0.8841,0.9196,0.9718
70,100%,week3_4_baseline,64,0.9246,0.9631,0.8914,0.9258,0.9780
72,100%,week5_expanded,98,0.9366,0.9754,0.9028,0.9377,0.9823


## 4. Pipeline Tuần 6: 16 Đặc trưng Động học (92 Features Catalog), Cảnh báo Sớm `LightGBM` & Biểu đồ (Hình 26–30)
Xây dựng danh mục 92 đặc trưng, huấn luyện đối kháng `week5_expanded` vs `week6_advanced` (114 biến mã hóa), tinh chỉnh ngưỡng quyết định $\tau^*$ **trên Validation** cho `LightGBM (Balanced)`, xuất `feature_catalog_week6.csv`, `feature_importance_week6.json`, `week6_experiment_metrics.json` vào `data/processed/week6_advanced_features/` và xuất Hình 26–30.

In [5]:
def get_week6_feature_metadata() -> List[Tuple[str, str, str, str, str, str, str]]:
    """Trả về metadata đầy đủ cho 92 đặc trưng (76 Tuần 5 cập nhật + 16 Tuần 6 mới)."""
    base_meta = [
        # Nhóm 5.1: Click-based (13)
        ("total_clicks", "5.1 Click-based", "Behavioural", r"\sum_{t \le T} \text{sum\_click}_t", "studentVle", "float64", "Tổng số lượt click tính đến mốc cắt T_cutoff"),
        ("active_days", "5.1 Click-based", "Behavioural", r"|\{t \le T : \text{click}_t > 0\}|", "studentVle", "float64", "Tổng số ngày có đăng nhập và tương tác VLE"),
        ("clicks_per_day", "5.1 Click-based", "Behavioural", r"\text{total\_clicks} / \max(1, T_{\text{cutoff}})", "studentVle + courses", "float64", "Trung bình số lượt click trên mỗi ngày lịch khóa học"),
        ("clicks_per_active_day", "5.1 Click-based", "Behavioural", r"\text{total\_clicks} / \text{active\_days}", "studentVle", "float64", "Cường độ click trung bình trong những ngày có hoạt động"),
        ("click_std", "5.1 Click-based", "Behavioural", r"\sqrt{\frac{1}{D-1}\sum_{d=1}^D (c_d - \bar{c})^2}", "studentVle", "float64", "Độ lệch chuẩn số lượt click mỗi ngày"),
        ("click_cv", "5.1 Click-based", "Behavioural", r"\text{click\_std} / \text{clicks\_per\_active\_day}", "studentVle", "float64", "Hệ số biến thiên phản ánh độ thất thường nhịp độ học"),
        ("active_days_in_course", "5.1 Click-based", "Behavioural", r"|\{0 \le t \le T : \text{click}_t > 0\}|", "studentVle", "float64", "Số ngày hoạt động tính từ ngày khai giảng (date >= 0)"),
        ("active_days_ratio", "5.1 Click-based", "Behavioural", r"\text{active\_days\_in\_course} / T_{\text{cutoff}}", "studentVle + courses", "float64", "Tỷ lệ ngày duy trì hoạt động trên tổng số ngày đã qua"),
        ("pre_course_clicks", "5.1 Click-based", "Behavioural / Self-Regulation", r"\sum_{t < 0} \text{sum\_click}_t", "studentVle", "float64", "Tổng số lượt click chuẩn bị trước khai giảng (date < 0)"),
        ("has_pre_course_clicks", "5.1 Click-based", "Behavioural / Self-Regulation", r"\mathbb{I}(\text{pre\_course\_clicks} > 0)", "studentVle", "int64", "Cờ nhị phân có truy cập LMS trước ngày khai giảng"),
        ("first_click_day", "5.1 Click-based", "Behavioural", r"\min \{t \le T : \text{click}_t > 0\}", "studentVle", "float64", "Ngày truy cập hệ thống VLE đầu tiên"),
        ("last_click_day", "5.1 Click-based", "Behavioural", r"\max \{t \le T : \text{click}_t > 0\}", "studentVle", "float64", "Ngày truy cập hệ thống VLE gần nhất đến mốc cắt"),
        ("active_span_days", "5.1 Click-based", "Behavioural", r"\text{last\_click\_day} - \text{first\_click\_day} + 1", "studentVle", "float64", "Khoảng trải dài thời gian gắn kết từ lần đầu đến cuối"),
        # Nhóm 5.2: Activity-based (31)
        ("clicks_oucontent", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{oucontent})", "studentVle + vle", "float64", "Lượt click đọc bài giảng cốt lõi OU Content"),
        ("clicks_forumng", "5.2 Activity-based", "Social (Proxy)", r"\sum \text{click}(\text{forumng})", "studentVle + vle", "float64", "Lượt click thảo luận trên diễn đàn ForumNG"),
        ("clicks_quiz", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{quiz})", "studentVle + vle", "float64", "Lượt click làm bài trắc nghiệm trực tuyến Quiz"),
        ("clicks_homepage", "5.2 Activity-based", "Behavioural (Navigation)", r"\sum \text{click}(\text{homepage})", "studentVle + vle", "float64", "Lượt click truy cập trang chủ môn học Homepage"),
        ("clicks_subpage", "5.2 Activity-based", "Behavioural (Navigation)", r"\sum \text{click}(\text{subpage})", "studentVle + vle", "float64", "Lượt click điều hướng trang con Subpage"),
        ("clicks_resource", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{resource})", "studentVle + vle", "float64", "Lượt click tải tài liệu tham khảo Resource"),
        ("clicks_ouwiki", "5.2 Activity-based", "Social (Proxy)", r"\sum \text{click}(\text{ouwiki})", "studentVle + vle", "float64", "Lượt click biên tập từ điển chung OU Wiki"),
        ("clicks_url", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{url})", "studentVle + vle", "float64", "Lượt click liên kết học thuật bên ngoài URL"),
        ("clicks_oucollaborate", "5.2 Activity-based", "Social (Proxy)", r"\sum \text{click}(\text{oucollaborate})", "studentVle + vle", "float64", "Lượt click phòng học trực tuyến đồng bộ"),
        ("clicks_glossary", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{glossary})", "studentVle + vle", "float64", "Lượt click tra cứu thuật ngữ Glossary"),
        ("clicks_questionnaire", "5.2 Activity-based", "Emotional / Feedback (Proxy)", r"\sum \text{click}(\text{questionnaire})", "studentVle + vle", "float64", "Lượt click phiếu khảo sát học phần"),
        ("clicks_externalquiz", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{externalquiz})", "studentVle + vle", "float64", "Lượt click làm bài kiểm tra mở rộng External Quiz"),
        ("clicks_page", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{page})", "studentVle + vle", "float64", "Lượt click đọc trang thông tin tĩnh Page"),
        ("clicks_dataplus", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{dataplus})", "studentVle + vle", "float64", "Lượt click công cụ dữ liệu DataPlus"),
        ("clicks_ouelluminate", "5.2 Activity-based", "Social (Proxy)", r"\sum \text{click}(\text{ouelluminate})", "studentVle + vle", "float64", "Lượt click hội thảo trực tuyến OU Elluminate"),
        ("clicks_dualpane", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{dualpane})", "studentVle + vle", "float64", "Lượt click giao diện đối chiếu song song DualPane"),
        ("clicks_htmlactivity", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{htmlactivity})", "studentVle + vle", "float64", "Lượt click bài tập tương tác HTML Activity"),
        ("clicks_folder", "5.2 Activity-based", "Behavioural (Navigation)", r"\sum \text{click}(\text{folder})", "studentVle + vle", "float64", "Lượt click mở thư mục tài liệu Folder"),
        ("clicks_sharedsubpage", "5.2 Activity-based", "Social (Proxy)", r"\sum \text{click}(\text{sharedsubpage})", "studentVle + vle", "float64", "Lượt click trang chia sẻ cộng tác Shared Subpage"),
        ("clicks_repeatactivity", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum \text{click}(\text{repeatactivity})", "studentVle + vle", "float64", "Lượt click hoạt động ôn tập lặp lại"),
        ("cognitive_clicks", "5.2 Activity-based", "Cognitive (Proxy)", r"\sum_{a \in \mathcal{C}} \text{clicks}_a", "studentVle + vle", "float64", "Tổng lượt click học liệu Nhận thức"),
        ("social_clicks", "5.2 Activity-based", "Social (Proxy)", r"\sum_{a \in \mathcal{S}} \text{clicks}_a", "studentVle + vle", "float64", "Tổng lượt click tương tác Xã hội"),
        ("oucontent_ratio", "5.2 Activity-based", "Cognitive (Proxy)", r"\text{clicks\_oucontent} / \text{total\_clicks}", "studentVle + vle", "float64", "Tỷ trọng tương tác đọc nội dung bài giảng"),
        ("forumng_ratio", "5.2 Activity-based", "Social (Proxy)", r"\text{clicks\_forumng} / \text{total\_clicks}", "studentVle + vle", "float64", "Tỷ trọng tương tác diễn đàn thảo luận"),
        ("quiz_ratio", "5.2 Activity-based", "Cognitive (Proxy)", r"\text{clicks\_quiz} / \text{total\_clicks}", "studentVle + vle", "float64", "Tỷ trọng tương tác làm trắc nghiệm Quiz"),
        ("resource_ratio", "5.2 Activity-based", "Cognitive (Proxy)", r"\text{clicks\_resource} / \text{total\_clicks}", "studentVle + vle", "float64", "Tỷ trọng tương tác tải tài nguyên tham khảo"),
        ("glossary_ratio", "5.2 Activity-based", "Cognitive (Proxy)", r"\text{clicks\_glossary} / \text{total\_clicks}", "studentVle + vle", "float64", "Tỷ trọng tương tác tra cứu thuật ngữ"),
        ("cognitive_ratio", "5.2 Activity-based", "Cognitive (Proxy)", r"\text{cognitive\_clicks} / \text{total\_clicks}", "studentVle + vle", "float64", "Tỷ trọng tổng tương tác nhóm Nhận thức"),
        ("social_ratio", "5.2 Activity-based", "Social (Proxy)", r"\text{social\_clicks} / \text{total\_clicks}", "studentVle + vle", "float64", "Tỷ trọng tổng tương tác nhóm Xã hội"),
        ("activity_diversity", "5.2 Activity-based", "Cognitive / Self-Regulation", r"\sum_{k=1}^{20} \mathbb{I}(\text{clicks}_k > 0)", "studentVle + vle", "float64", "Số loại tài nguyên VLE khác nhau đã khai thác"),
        ("activity_entropy", "5.2 Activity-based", "Cognitive / Self-Regulation", r"-\sum_{k=1}^{20} p_k \ln(p_k)", "studentVle + vle", "float64", "Shannon Entropy đo độ cân bằng và đa dạng tài nguyên"),
        # Nhóm 5.3: Assessment-based (19)
        ("avg_score", "5.3 Assessment-based", "Cognitive (Performance)", r"\frac{1}{M}\sum_{i=1}^M s_i", "studentAssessment + assessments", "float64", "Điểm trung bình các bài Coursework đã nộp đến T_cutoff"),
        ("weighted_avg_score", "5.3 Assessment-based", "Cognitive (Performance)", r"\frac{\sum s_i w_i}{\sum w_i}", "studentAssessment + assessments", "float64", "Điểm trung bình có trọng số các bài Coursework đã nộp"),
        ("earned_weight_score", "5.3 Assessment-based", "Cognitive (Achievement)", r"\sum_{i=1}^M \frac{s_i w_i}{100}", "studentAssessment + assessments", "float64", "Tổng điểm trọng số tích lũy trên thang 100"),
        ("min_score", "5.3 Assessment-based", "Cognitive (Performance)", r"\min_{1 \le i \le M} s_i", "studentAssessment + assessments", "float64", "Điểm bài Coursework thấp nhất đến T_cutoff"),
        ("max_score", "5.3 Assessment-based", "Cognitive (Performance)", r"\max_{1 \le i \le M} s_i", "studentAssessment + assessments", "float64", "Điểm bài Coursework cao nhất đến T_cutoff"),
        ("score_std", "5.3 Assessment-based", "Cognitive (Consistency)", r"\sqrt{\frac{1}{M-1}\sum (s_i - \bar{s})^2}", "studentAssessment + assessments", "float64", "Độ lệch chuẩn điểm số giữa các bài đã nộp"),
        ("first_score", "5.3 Assessment-based", "Cognitive (Early Signal)", r"s_{\arg\min_i \text{date\_submitted}_i}", "studentAssessment + assessments", "float64", "Điểm bài Coursework đầu tiên thực tế nộp"),
        ("num_submissions", "5.3 Assessment-based", "Behavioural (Task Completion)", r"|\{i : \text{submitted}_i \le T \land s_i \neq \text{NaN}\}|", "studentAssessment + assessments", "int64", "Số bài Coursework đã nộp và có điểm đến T_cutoff"),
        ("num_submissions_raw", "5.3 Assessment-based", "Behavioural (Task Completion)", r"|\{i : \text{submitted}_i \le T\}|", "studentAssessment + assessments", "int64", "Tổng số bài Coursework đã nộp đến T_cutoff"),
        ("due_course_assessments", "5.3 Assessment-based", "Context (Course Structure)", r"|\{j \in \text{CW} : \text{deadline}_j \le T\}|", "assessments + courses", "int64", "Số bài Coursework đã đến hạn tại T_cutoff"),
        ("total_course_assessments", "5.3 Assessment-based", "Context (Course Structure)", r"|\{j \in \text{CW}\}|", "assessments", "int64", "Tổng số bài Coursework toàn khóa học"),
        ("submission_rate", "5.3 Assessment-based", "Behavioural (Task Completion)", r"\min(1, \text{num\_submissions} / \max(1, \text{due}))", "studentAssessment + assessments", "float64", "Tỷ lệ hoàn thành bài tập đã đến hạn đến T_cutoff"),
        ("course_progress_submission_rate", "5.3 Assessment-based", "Behavioural (Task Completion)", r"\text{num\_submissions} / \text{total\_course\_assessments}", "studentAssessment + assessments", "float64", "Tỷ lệ hoàn thành so với tổng bài tập cả khóa"),
        ("avg_days_before_deadline", "5.3 Assessment-based", "Behavioural / Self-Regulation", r"\frac{1}{M}\sum (\text{deadline}_i - \text{submitted}_i)", "studentAssessment + assessments", "float64", "Số ngày nộp trước hạn trung bình"),
        ("late_submissions", "5.3 Assessment-based", "Behavioural / Disengagement", r"\sum \mathbb{I}(\text{submitted}_i > \text{deadline}_i)", "studentAssessment + assessments", "int64", "Số bài Coursework nộp trễ hạn"),
        ("late_submission_rate", "5.3 Assessment-based", "Behavioural / Disengagement", r"\text{late\_submissions} / \text{num\_submissions\_raw}", "studentAssessment + assessments", "float64", "Tỷ lệ bài nộp trễ trên tổng bài nộp"),
        ("tma1_score", "5.3 Assessment-based", "Cognitive (Early Signal)", r"s_{\text{TMA1}}", "studentAssessment + assessments", "float64", "Điểm số bài TMA đầu tiên (TMA1)"),
        ("tma1_days_early", "5.3 Assessment-based", "Behavioural / Self-Regulation", r"\text{deadline}_{\text{TMA1}} - \text{submitted}_{\text{TMA1}}", "studentAssessment + assessments", "float64", "Số ngày nộp sớm/muộn bài TMA1"),
        ("submitted_tma1", "5.3 Assessment-based", "Behavioural (Task Completion)", r"\mathbb{I}(s_{\text{TMA1}} \neq \text{NaN})", "studentAssessment + assessments", "int64", "Cờ nhị phân đã nộp bài TMA1 đến T_cutoff"),
        # Nhóm 5.4: Demographics & Context (13) - dùng days_before_start chuẩn hóa
        ("gender", "5.4 Demographics & Context", "Demographic", r"\text{Nominal } \{M, F\}", "studentInfo", "category", "Giới tính của sinh viên"),
        ("region", "5.4 Demographics & Context", "Demographic", r"\text{Nominal (13 UK Regions)}", "studentInfo", "category", "Vùng địa lý cư trú tại Vương quốc Anh"),
        ("highest_education", "5.4 Demographics & Context", "Demographic (Prior Knowledge)", r"\text{Ordinal } \{0, 1, 2, 3, 4\}", "studentInfo", "ordinal", "Trình độ học vấn cao nhất khi nhập học"),
        ("imd_band", "5.4 Demographics & Context", "Demographic (Socio-Economic)", r"\text{Ordinal } \{0, \dots, 9\}", "studentInfo", "ordinal", "Chỉ số thiếu thốn đa chiều vùng cư trú (IMD)"),
        ("age_band", "5.4 Demographics & Context", "Demographic", r"\text{Ordinal } \{0\text{-}35, 35\text{-}55, 55\le\}", "studentInfo", "ordinal", "Nhóm độ tuổi của sinh viên"),
        ("disability", "5.4 Demographics & Context", "Demographic", r"\text{Nominal } \{Y, N\}", "studentInfo", "category", "Tình trạng khuyết tật tự khai báo"),
        ("num_of_prev_attempts", "5.4 Demographics & Context", "Academic History", r"\mathbb{N}_0", "studentInfo", "int64", "Số lần đã đăng ký học lại học phần trước đây"),
        ("studied_credits", "5.4 Demographics & Context", "Academic Load", r"\sum \text{credits}", "studentInfo", "int64", "Tổng số tín chỉ đang học song song"),
        ("module_presentation_length", "5.4 Demographics & Context", "Course Context", r"L \in [234, 269]", "courses", "int64", "Tổng thời lượng chính thức của khóa học (ngày)"),
        ("days_before_start", "5.4 Demographics & Context", "Behavioural (Planning)", r"-t_{\text{reg}}", "studentRegistration", "float64", "Số ngày đăng ký sớm trước ngày khai giảng (-date_registration)"),
        ("is_repeat_student", "5.4 Demographics & Context", "Academic History", r"\mathbb{I}(\text{num\_of\_prev\_attempts} > 0)", "studentInfo", "int64", "Cờ nhị phân sinh viên từng học lại môn học"),
        ("credit_overload", "5.4 Demographics & Context", "Academic Load", r"\mathbb{I}(\text{studied\_credits} > 60)", "studentInfo", "int64", "Cờ nhị phân đăng ký trên 60 tín chỉ tiêu chuẩn"),
        ("is_late_registration", "5.4 Demographics & Context", "Behavioural (Planning)", r"\mathbb{I}(\text{date\_registration} > 0)", "studentRegistration", "int64", "Cờ nhị phân đăng ký muộn sau khai giảng"),
    ]

    week6_meta = [
        # Nhóm 6.1: Temporal & Decay Features (8)
        ("clicks_week_1_to_4", "6.1 Temporal & Decay", "Behavioural (Early Engagement)", r"\sum_{0 \le t \le \min(27, T)} \text{click}_t", "studentVle", "float64", "Tổng số lượt click trong 4 tuần đầu khóa học (ngày 0-27)"),
        ("clicks_last_2_weeks", "6.1 Temporal & Decay", "Behavioural (Recency)", r"\sum_{T-14 < t \le T} \text{click}_t", "studentVle", "float64", "Tổng số lượt click trong 14 ngày gần nhất trước T_cutoff"),
        ("clicks_trend", "6.1 Temporal & Decay", "Behavioural (Trajectory)", r"\beta_{\text{OLS}}(\text{weekly\_clicks}_{0..W})", "studentVle", "float64", "Hệ số góc OLS của chuỗi click theo từng tuần (dương = tăng, âm = giảm)"),
        ("engagement_decay", "6.1 Temporal & Decay", "Behavioural (Decay)", r"\frac{\text{clicks\_last\_2\_weeks}}{\max(1, \text{clicks\_week\_1\_to\_4})}", "studentVle", "float64", "Tỷ lệ suy giảm/tăng trưởng tương tác cuối cửa sổ so với 4 tuần đầu"),
        ("days_since_last_activity", "6.1 Temporal & Decay", "Behavioural (Disengagement)", r"\max(0, T_{\text{cutoff}} - \text{last\_click\_day})", "studentVle", "float64", "Số ngày vắng mặt tính từ lần click cuối cùng đến mốc T_cutoff"),
        ("inactivity_ratio", "6.1 Temporal & Decay", "Behavioural (Disengagement)", r"\frac{\lfloor T_{\text{cutoff}} \rfloor + 1 - \text{active\_days\_in\_course}}{\lfloor T_{\text{cutoff}} \rfloor + 1}", "studentVle", "float64", "Tỷ lệ ngày hoàn toàn không hoạt động trong kỳ chính thức"),
        ("max_inactivity_streak", "6.1 Temporal & Decay", "Behavioural (Disengagement)", r"\max_i (\Delta t_{\text{inactive}, i})", "studentVle", "float64", "Chuỗi ngày vắng mặt liên tiếp dài nhất từ ngày 0 đến T_cutoff"),
        ("consistency_score", "6.1 Temporal & Decay", "Behavioural / Self-Regulation", r"\frac{-\sum_{w=0}^{W} p_w \ln(p_w)}{\ln(W+1)}", "studentVle", "float64", "Độ đều đặn phân bố click qua các tuần theo Shannon Entropy chuẩn hóa [0,1]"),
        # Nhóm 6.2: Regularity, Weekend & Burst Patterns (6)
        ("active_weeks", "6.2 Regularity & Burst", "Behavioural (Persistence)", r"|\{w \in [0, W] : \text{weekly\_click}_w > 0\}|", "studentVle", "int64", "Số tuần có ít nhất 1 lượt click từ khai giảng đến T_cutoff"),
        ("login_regularity", "6.2 Regularity & Burst", "Behavioural (Persistence)", r"\frac{\text{active\_weeks}}{W_{\text{cutoff}} + 1}", "studentVle", "float64", "Tỷ lệ tuần duy trì đăng nhập trên tổng số tuần đã qua [0,1]"),
        ("weekend_clicks", "6.2 Regularity & Burst", "Behavioural (Study Habit)", r"\sum_{t \bmod 7 \in \{5,6\}} \text{click}_t", "studentVle", "float64", "Tổng số lượt click vào cuối tuần (thứ 7 và chủ nhật theo lịch khóa)"),
        ("weekend_ratio", "6.2 Regularity & Burst", "Behavioural (Study Habit)", r"\text{weekend\_clicks} / \text{total\_clicks}", "studentVle", "float64", "Tỷ trọng hoạt động học tập diễn ra vào cuối tuần [0,1]"),
        ("burst_count", "6.2 Regularity & Burst", "Behavioural / Self-Regulation", r"|\{d : c_d > \bar{c} + 2 s_c\}|", "studentVle", "int64", "Số ngày học dồn bất thường (vượt mean + 2*std, active_days >= 3)"),
        ("burst_ratio", "6.2 Regularity & Burst", "Behavioural / Self-Regulation", r"\text{burst\_count} / \text{active\_days}", "studentVle", "float64", "Tỷ lệ ngày học bùng nổ trên tổng số ngày có hoạt động [0,1]"),
        # Nhóm 6.3: Early Warning Score Trajectory (2)
        ("last_minus_first_score", "6.3 Score Trajectory", "Cognitive (Trajectory)", r"s_{\text{last}} - s_{\text{first}}", "studentAssessment + assessments", "float64", "Chênh lệch điểm bài nộp gần nhất trừ bài đầu tiên (0 nếu < 2 bài)"),
        ("score_trend_slope", "6.3 Score Trajectory", "Cognitive (Trajectory)", r"\beta_{\text{OLS}}(s_0, \dots, s_{m-1})", "studentAssessment + assessments", "float64", "Hệ số góc OLS xu hướng điểm số theo thứ tự bài nộp (0 nếu < 2 bài)"),
    ]
    return base_meta + week6_meta


def _get_numeric_series(df_s: pd.DataFrame, engineer: OULADFeatureEngineer, col: str) -> pd.Series:
    """Chuyển đổi một cột đặc trưng (số hoặc phân loại) sang dạng số phục vụ thống kê tương quan."""
    if col in engineer.ORDINAL_MAPPINGS:
        m = engineer.ORDINAL_MAPPINGS[col]
        return df_s[col].map(m).fillna(m.get("Unknown", 2)).astype(float)
    if col in ["gender", "disability", "region", "code_module"]:
        return pd.Series(pd.factorize(df_s[col])[0], index=df_s.index, dtype=float)
    if col in ["avg_score", "weighted_avg_score", "min_score", "max_score", "first_score", "tma1_score"]:
        return df_s[col].fillna(0.0).astype(float)
    if col in ["avg_days_before_deadline", "avg_days_early", "tma1_days_early"]:
        return df_s[col].fillna(-10.0).astype(float)
    return df_s[col].fillna(0.0).astype(float)


def build_week6_feature_catalog(
    df_100: pd.DataFrame,
    df_25: pd.DataFrame,
    df_50: pd.DataFrame,
    df_75: pd.DataFrame,
    engineer: OULADFeatureEngineer,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """Xây dựng bảng danh mục 92 đặc trưng Tuần 6 và phân tích Feature Importance tại cả 4 mốc 25%, 50%, 75%, 100%."""
    logger.info("Đang xây dựng Danh mục 92 Đặc trưng Tuần 6 và tính Importance...")
    feature_meta = get_week6_feature_metadata()

    def _compute_ml_importances(df_snap: pd.DataFrame) -> Tuple[Dict[str, float], Dict[str, float], Dict[str, float]]:
        tr_df, va_df, te_df = train_test_split_group(df_snap, random_state=42)
        X_tr, X_va, _ = engineer.prepare_ml_matrices(tr_df, va_df, te_df, feature_set="week6_advanced")
        y_tr = tr_df["at_risk"].values
        y_va = va_df["at_risk"].values

        rf = RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1)
        rf.fit(X_tr, y_tr)
        rf_imp = dict(zip(X_tr.columns, rf.feature_importances_))

        perm = permutation_importance(rf, X_va, y_va, n_repeats=5, random_state=42, scoring="roc_auc", n_jobs=-1)
        perm_imp = dict(zip(X_va.columns, np.maximum(0.0, perm.importances_mean)))

        rng = np.random.RandomState(42)
        sample_idx = rng.choice(len(X_tr), size=min(8000, len(X_tr)), replace=False)
        mi_vals = mutual_info_classif(X_tr.iloc[sample_idx], y_tr[sample_idx], random_state=42)
        mi_imp = dict(zip(X_tr.columns, mi_vals))
        return rf_imp, perm_imp, mi_imp

    rf_100, perm_100, mi_100 = _compute_ml_importances(df_100)
    rf_75, perm_75, mi_75 = _compute_ml_importances(df_75)
    rf_50, perm_50, mi_50 = _compute_ml_importances(df_50)
    rf_25, perm_25, mi_25 = _compute_ml_importances(df_25)

    def _lookup_imp(imp_dict: Dict[str, float], col_name: str) -> float:
        if col_name in imp_dict:
            return float(imp_dict[col_name])
        prefix = f"{col_name}_"
        matched = [v for k, v in imp_dict.items() if k.startswith(prefix)]
        return float(sum(matched)) if matched else 0.0

    y_full_100 = df_100["at_risk"].values
    y_full_75 = df_75["at_risk"].values
    y_full_50 = df_50["at_risk"].values
    y_full_25 = df_25["at_risk"].values
    rows = []

    for fname, fgroup, sedl_dim, formula, source_tbl, dtype_str, desc in feature_meta:
        s100 = _get_numeric_series(df_100, engineer, fname)
        s75 = _get_numeric_series(df_75, engineer, fname)
        s50 = _get_numeric_series(df_50, engineer, fname)
        s25 = _get_numeric_series(df_25, engineer, fname)

        v0_100 = s100[y_full_100 == 0].values
        v1_100 = s100[y_full_100 == 1].values
        r_100, _ = stats.pointbiserialr(y_full_100, s100.values)
        rho_100, _ = stats.spearmanr(y_full_100, s100.values)
        d_100 = cohens_d(v0_100, v1_100)

        r_75, _ = stats.pointbiserialr(y_full_75, s75.values)
        d_75 = cohens_d(s75[y_full_75 == 0].values, s75[y_full_75 == 1].values)

        r_50, _ = stats.pointbiserialr(y_full_50, s50.values)
        d_50 = cohens_d(s50[y_full_50 == 0].values, s50[y_full_50 == 1].values)

        r_25, _ = stats.pointbiserialr(y_full_25, s25.values)
        d_25 = cohens_d(s25[y_full_25 == 0].values, s25[y_full_25 == 1].values)

        rows.append(
            {
                "feature_name": fname,
                "feature_group": fgroup,
                "sedl_dimension": sedl_dim,
                "formula_latex": formula,
                "source_table": source_tbl,
                "data_type": dtype_str,
                "description": desc,
                "is_week6_new": int(fgroup.startswith("6.")),
                "missing_pct_raw_100": float(df_100[fname].isna().mean() * 100.0) if fname in df_100.columns else 0.0,
                "mean_not_at_risk_100": float(np.mean(v0_100)),
                "sd_not_at_risk_100": float(np.std(v0_100, ddof=1)),
                "median_not_at_risk_100": float(np.median(v0_100)),
                "iqr_not_at_risk_100": float(np.percentile(v0_100, 75) - np.percentile(v0_100, 25)),
                "mean_at_risk_100": float(np.mean(v1_100)),
                "sd_at_risk_100": float(np.std(v1_100, ddof=1)),
                "median_at_risk_100": float(np.median(v1_100)),
                "iqr_at_risk_100": float(np.percentile(v1_100, 75) - np.percentile(v1_100, 25)),
                "point_biserial_r_100": float(0.0 if np.isnan(r_100) else r_100),
                "spearman_rho_100": float(0.0 if np.isnan(rho_100) else rho_100),
                "cohens_d_100": float(d_100),
                "mutual_info_100": _lookup_imp(mi_100, fname),
                "rf_gini_importance_100": _lookup_imp(rf_100, fname),
                "rf_perm_importance_100": _lookup_imp(perm_100, fname),
                "point_biserial_r_25": float(0.0 if np.isnan(r_25) else r_25),
                "cohens_d_25": float(d_25),
                "mutual_info_25": _lookup_imp(mi_25, fname),
                "rf_gini_importance_25": _lookup_imp(rf_25, fname),
                "rf_perm_importance_25": _lookup_imp(perm_25, fname),
                "point_biserial_r_50": float(0.0 if np.isnan(r_50) else r_50),
                "cohens_d_50": float(d_50),
                "mutual_info_50": _lookup_imp(mi_50, fname),
                "rf_gini_importance_50": _lookup_imp(rf_50, fname),
                "rf_perm_importance_50": _lookup_imp(perm_50, fname),
                "point_biserial_r_75": float(0.0 if np.isnan(r_75) else r_75),
                "cohens_d_75": float(d_75),
                "mutual_info_75": _lookup_imp(mi_75, fname),
                "rf_gini_importance_75": _lookup_imp(rf_75, fname),
                "rf_perm_importance_75": _lookup_imp(perm_75, fname),
            }
        )

    catalog_df = pd.DataFrame(rows)
    for p in [25, 50, 75, 100]:
        catalog_df[f"rank_overall_{p}"] = (
            catalog_df[f"rf_gini_importance_{p}"].rank(ascending=False, method="min").astype(int)
        )

    importance_summary = {
        "total_catalog_features": int(len(catalog_df)),
        "new_week6_features_count": int(catalog_df["is_week6_new"].sum()),
        "top20_by_rf_100pct": catalog_df.sort_values("rf_gini_importance_100", ascending=False)[
            ["feature_name", "feature_group", "is_week6_new", "point_biserial_r_100", "cohens_d_100", "mutual_info_100", "rf_gini_importance_100", "rf_perm_importance_100", "rank_overall_100"]
        ].head(20).to_dict(orient="records"),
        "top20_by_rf_75pct": catalog_df.sort_values("rf_gini_importance_75", ascending=False)[
            ["feature_name", "feature_group", "is_week6_new", "point_biserial_r_75", "cohens_d_75", "mutual_info_75", "rf_gini_importance_75", "rf_perm_importance_75", "rank_overall_75"]
        ].head(20).to_dict(orient="records"),
        "top20_by_rf_50pct": catalog_df.sort_values("rf_gini_importance_50", ascending=False)[
            ["feature_name", "feature_group", "is_week6_new", "point_biserial_r_50", "cohens_d_50", "mutual_info_50", "rf_gini_importance_50", "rf_perm_importance_50", "rank_overall_50"]
        ].head(20).to_dict(orient="records"),
        "top20_by_rf_25pct": catalog_df.sort_values("rf_gini_importance_25", ascending=False)[
            ["feature_name", "feature_group", "is_week6_new", "point_biserial_r_25", "cohens_d_25", "mutual_info_25", "rf_gini_importance_25", "rf_perm_importance_25", "rank_overall_25"]
        ].head(20).to_dict(orient="records"),
        "week6_new_features_importance_25pct": catalog_df[catalog_df["is_week6_new"] == 1].sort_values(
            "rf_gini_importance_25", ascending=False
        )[
            ["feature_name", "feature_group", "point_biserial_r_25", "cohens_d_25", "mutual_info_25", "rf_gini_importance_25", "rf_perm_importance_25", "rank_overall_25", "rf_gini_importance_100"]
        ].to_dict(orient="records"),
        "week6_new_features_importance_50pct": catalog_df[catalog_df["is_week6_new"] == 1].sort_values(
            "rf_gini_importance_50", ascending=False
        )[
            ["feature_name", "feature_group", "point_biserial_r_50", "cohens_d_50", "mutual_info_50", "rf_gini_importance_50", "rf_perm_importance_50", "rank_overall_50"]
        ].to_dict(orient="records"),
        "week6_new_features_importance_75pct": catalog_df[catalog_df["is_week6_new"] == 1].sort_values(
            "rf_gini_importance_75", ascending=False
        )[
            ["feature_name", "feature_group", "point_biserial_r_75", "cohens_d_75", "mutual_info_75", "rf_gini_importance_75", "rf_perm_importance_75", "rank_overall_75"]
        ].to_dict(orient="records"),
        "week6_new_features_importance_100pct": catalog_df[catalog_df["is_week6_new"] == 1].sort_values(
            "rf_gini_importance_100", ascending=False
        )[
            ["feature_name", "feature_group", "point_biserial_r_100", "cohens_d_100", "mutual_info_100", "rf_gini_importance_100", "rf_perm_importance_100", "rank_overall_100"]
        ].to_dict(orient="records"),
        "group_importance_share_25pct": catalog_df.groupby("feature_group")["rf_gini_importance_25"].sum().to_dict(),
        "group_importance_share_50pct": catalog_df.groupby("feature_group")["rf_gini_importance_50"].sum().to_dict(),
        "group_importance_share_75pct": catalog_df.groupby("feature_group")["rf_gini_importance_75"].sum().to_dict(),
        "group_importance_share_100pct": catalog_df.groupby("feature_group")["rf_gini_importance_100"].sum().to_dict(),
    }
    return catalog_df, importance_summary


def tune_threshold_on_validation(
    y_val: np.ndarray,
    val_prob: np.ndarray,
    min_recall_target: float = 0.76,
) -> Tuple[float, float, float]:
    """
    Tìm ngưỡng quyết định tau* CHỈ TRÊN TẬP VALIDATION (Zero Test Leakage) sao cho:
    - Tối đa hóa F1-Score trong tập các ngưỡng đạt Validation Recall >= min_recall_target.
    - Nếu không có ngưỡng nào đạt min_recall_target, chọn ngưỡng tối đa hóa Recall.
    """
    best_tau = 0.50
    best_f1 = -1.0
    best_rec = -1.0

    candidates = np.arange(0.15, 0.71, 0.01)
    feasible_found = False
    for tau in candidates:
        pred = (val_prob >= tau).astype(int)
        rec = float(recall_score(y_val, pred, zero_division=0))
        f1 = float(f1_score(y_val, pred, zero_division=0))
        if rec >= min_recall_target:
            feasible_found = True
            if f1 > best_f1 or (abs(f1 - best_f1) < 1e-9 and rec > best_rec):
                best_f1 = f1
                best_rec = rec
                best_tau = float(round(tau, 2))

    if not feasible_found:
        for tau in candidates:
            pred = (val_prob >= tau).astype(int)
            rec = float(recall_score(y_val, pred, zero_division=0))
            f1 = float(f1_score(y_val, pred, zero_division=0))
            if rec > best_rec or (abs(rec - best_rec) < 1e-9 and f1 > best_f1):
                best_rec = rec
                best_f1 = f1
                best_tau = float(round(tau, 2))

    return best_tau, best_rec, best_f1


def _evaluate_predictions(y_true: np.ndarray, prob: np.ndarray, threshold: float = 0.50) -> Dict[str, float]:
    """Tính toàn bộ thước đo đánh giá phân loại nhị phân tại ngưỡng cho trước."""
    pred = (prob >= threshold).astype(int)
    return {
        "accuracy": float(np.mean(pred == y_true)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, prob)),
        "pr_auc": float(average_precision_score(y_true, prob)),
        "brier": float(brier_score_loss(y_true, prob)),
    }


def run_week6_benchmark_experiments(
    snapshots: Dict[float, pd.DataFrame],
    engineer: OULADFeatureEngineer,
) -> Dict[str, Any]:
    """
    Thực nghiệm đối sánh Tuần 6 qua 4 lát cắt (25%, 50%, 75%, 100%) x 3 Cohorts (group_stratified):
    1. So sánh week5_expanded (98 biến mã hóa) vs week6_advanced (114 biến mã hóa).
    2. Mô hình chuẩn (tau=0.50): Logistic Regression, Random Forest, Gradient Boosting, LightGBM.
    3. Cấu hình Early Warning Cost-Sensitive (class_weight='balanced' + Validation-tuned tau*)
       nhằm đạt Test Recall >= 75% ngay từ mốc 25% trên dynamic_active.
    """
    logger.info("Đang chạy thực nghiệm đối sánh Tuần 6 (4 snapshots x 3 cohorts x 4 models + Early Warning)...")
    standard_results: List[Dict[str, Any]] = []
    early_warning_results: List[Dict[str, Any]] = []

    standard_models = {
        "Logistic Regression": lambda: LogisticRegression(max_iter=1000, C=1.0, random_state=42),
        "Random Forest": lambda: RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42, n_jobs=-1),
        "Gradient Boosting": lambda: GradientBoostingClassifier(
            n_estimators=120, max_depth=4, learning_rate=0.1, random_state=42
        ),
        "LightGBM": lambda: LGBMClassifier(
            n_estimators=250,
            learning_rate=0.05,
            num_leaves=31,
            colsample_bytree=0.8,
            subsample=0.8,
            subsample_freq=1,
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        ),
    }

    balanced_models = {
        "Logistic Regression (Balanced)": lambda: LogisticRegression(
            max_iter=1000, C=1.0, class_weight="balanced", random_state=42
        ),
        "Random Forest (Balanced)": lambda: RandomForestClassifier(
            n_estimators=180, max_depth=12, class_weight="balanced", random_state=42, n_jobs=-1
        ),
        "LightGBM (Balanced)": lambda: LGBMClassifier(
            n_estimators=250,
            learning_rate=0.05,
            num_leaves=31,
            colsample_bytree=0.8,
            subsample=0.8,
            subsample_freq=1,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
            verbose=-1,
        ),
    }

    for pct, df_snap in snapshots.items():
        pct_label = f"{int(pct * 100)}%"
        for cohort_name in ["full", "pre_course_active", "dynamic_active"]:
            df_cohort = engineer.filter_cohort(df_snap, cohort_type=cohort_name)
            tr_df, va_df, te_df = train_test_split_group(df_cohort, split_strategy="group_stratified", random_state=42)
            y_tr, y_va, y_te = tr_df["at_risk"].values, va_df["at_risk"].values, te_df["at_risk"].values
            n_total = len(df_cohort)
            at_risk_rate = float(df_cohort["at_risk"].mean())

            for fset in ["week5_expanded", "week6_advanced"]:
                X_tr, X_va, X_te = engineer.prepare_ml_matrices(tr_df, va_df, te_df, feature_set=fset)

                for m_name, m_builder in standard_models.items():
                    clf = m_builder()
                    clf.fit(X_tr, y_tr)
                    va_m = _evaluate_predictions(y_va, clf.predict_proba(X_va)[:, 1], threshold=0.50)
                    te_m = _evaluate_predictions(y_te, clf.predict_proba(X_te)[:, 1], threshold=0.50)

                    standard_results.append(
                        {
                            "snapshot": pct_label,
                            "cutoff_pct": pct,
                            "cohort": cohort_name,
                            "n_cohort": int(n_total),
                            "at_risk_rate": round(at_risk_rate, 4),
                            "split_strategy": "group_stratified",
                            "feature_set": fset,
                            "n_features_encoded": int(X_tr.shape[1]),
                            "model": m_name,
                            "threshold": 0.50,
                            **{f"val_{k}": round(v, 4) for k, v in va_m.items()},
                            **{f"test_{k}": round(v, 4) for k, v in te_m.items()},
                        }
                    )

            # Cost-sensitive + Validation-tuned threshold trên week6_advanced cho toàn bộ 4 mốc x 3 cohort (36 cấu hình)
            X_tr_w6, X_va_w6, X_te_w6 = engineer.prepare_ml_matrices(
                tr_df, va_df, te_df, feature_set="week6_advanced"
            )
            for m_name, m_builder in balanced_models.items():
                clf = m_builder()
                clf.fit(X_tr_w6, y_tr)
                va_prob = clf.predict_proba(X_va_w6)[:, 1]
                te_prob = clf.predict_proba(X_te_w6)[:, 1]

                # Standard tau=0.50 on balanced model
                te_m_05 = _evaluate_predictions(y_te, te_prob, threshold=0.50)
                # Tuned tau* purely on validation (Zero Test Leakage)
                tau_star, va_rec_tuned, va_f1_tuned = tune_threshold_on_validation(
                    y_va, va_prob, min_recall_target=0.76
                )
                va_m_tuned = _evaluate_predictions(y_va, va_prob, threshold=tau_star)
                te_m_tuned = _evaluate_predictions(y_te, te_prob, threshold=tau_star)

                early_warning_results.append(
                    {
                        "snapshot": pct_label,
                        "cutoff_pct": pct,
                        "cohort": cohort_name,
                        "n_cohort": int(n_total),
                        "at_risk_rate": round(at_risk_rate, 4),
                        "feature_set": "week6_advanced",
                        "n_features_encoded": int(X_tr_w6.shape[1]),
                        "model": m_name,
                        "default_05_test_recall": round(te_m_05["recall"], 4),
                        "default_05_test_f1": round(te_m_05["f1"], 4),
                        "tuned_threshold_val": tau_star,
                        "val_recall_at_tau": round(va_m_tuned["recall"], 4),
                        "val_f1_at_tau": round(va_m_tuned["f1"], 4),
                        **{f"test_{k}": round(v, 4) for k, v in te_m_tuned.items()},
                    }
                )

        logger.info(f"Hoàn tất benchmark snapshot {pct_label}")

    return {
        "experiments": standard_results,
        "early_warning_tuned": early_warning_results,
    }


def generate_week6_figures(
    snapshots: Dict[float, pd.DataFrame],
    catalog_df: pd.DataFrame,
    exp_results: List[Dict[str, Any]],
    ew_results: List[Dict[str, Any]],
    figures_dir: Path,
) -> None:
    """Sinh 5 biểu đồ Tuần 6 chuẩn học thuật 300 DPI tại reports/figures/ (Hình 26 - Hình 30)."""
    figures_dir.mkdir(parents=True, exist_ok=True)
    df_25 = snapshots[0.25].copy()
    df_50 = snapshots[0.50].copy()
    df_100 = snapshots[1.00].copy()

    order_4 = ["Distinction", "Pass", "Fail", "Withdrawn"]
    palette_4 = {"Distinction": "#27ae60", "Pass": "#2980b9", "Fail": "#e67e22", "Withdrawn": "#c0392b"}
    palette_bin = {0: "#2980b9", 1: "#c0392b"}

    # =========================================================================
    # HÌNH 26: 26_temporal_trend_and_engagement_decay.png
    # =========================================================================
    logger.info("Đang vẽ Hình 26: 26_temporal_trend_and_engagement_decay.png")
    fig, axes = plt.subplots(2, 2, figsize=(16, 11.5))

    # (A) So sánh Click 4 tuần đầu vs 2 tuần cuối tại Mốc 50% (Cohort Dynamic Active)
    dyn_50 = df_50[df_50["is_snapshot_active"] == 1].copy()
    melt_w = dyn_50[["final_result", "clicks_week_1_to_4", "clicks_last_2_weeks"]].melt(
        id_vars="final_result", var_name="Window", value_name="Clicks"
    )
    melt_w["LogClicks"] = np.log1p(melt_w["Clicks"])
    melt_w["WindowLabel"] = melt_w["Window"].map(
        {"clicks_week_1_to_4": "4 Tuần Đầu (Ngày 0-27)", "clicks_last_2_weeks": "2 Tuần Gần Nhất (Trước 50%)"}
    )
    sns.boxplot(
        data=melt_w,
        x="final_result",
        y="LogClicks",
        hue="WindowLabel",
        order=order_4,
        palette={"4 Tuần Đầu (Ngày 0-27)": "#3498db", "2 Tuần Gần Nhất (Trước 50%)": "#e67e22"},
        showfliers=False,
        ax=axes[0, 0],
    )
    axes[0, 0].set_title("(A) Cường độ Tương tác Đầu kỳ vs Cận mốc 50% (Dynamic Active)", fontsize=11.5, fontweight="bold")
    axes[0, 0].set_xlabel("Kết quả học tập cuối kỳ", fontsize=10.5, fontweight="bold")
    axes[0, 0].set_ylabel("log1p(Số lượt Click)", fontsize=10.5, fontweight="bold")
    axes[0, 0].legend(title="Cửa sổ thời gian", loc="upper right", fontsize=9)
    axes[0, 0].grid(axis="y", alpha=0.3)

    # (B) Phân phối Engagement Decay (capped at 4.0 cho trực quan hóa) tại Mốc 50%
    dyn_50["decay_viz"] = np.clip(dyn_50["engagement_decay"], 0.0, 3.0)
    sns.violinplot(
        data=dyn_50[dyn_50["clicks_week_1_to_4"] > 0],
        x="final_result",
        y="decay_viz",
        hue="final_result",
        order=order_4,
        palette=palette_4,
        legend=False,
        inner="quartile",
        cut=0,
        ax=axes[0, 1],
    )
    axes[0, 1].axhline(1.0, color="black", linestyle="--", linewidth=1.2, label="Mốc duy trì ngang bằng (1.0)")
    axes[0, 1].set_title("(B) Tỷ lệ Suy giảm Tương tác (engagement_decay - Mốc 50%)", fontsize=11.5, fontweight="bold")
    axes[0, 1].set_xlabel("Kết quả học tập cuối kỳ", fontsize=10.5, fontweight="bold")
    axes[0, 1].set_ylabel("Tỷ lệ Cận mốc / Đầu kỳ (Cắt tại 3.0)", fontsize=10.5, fontweight="bold")
    axes[0, 1].legend(loc="upper right", fontsize=9)
    axes[0, 1].grid(axis="y", alpha=0.3)

    # (C) Phân phối Clicks Trend (OLS slope theo tuần) tại Mốc 50%
    q_low, q_high = np.percentile(dyn_50["clicks_trend"], [2, 98])
    trend_viz = dyn_50[(dyn_50["clicks_trend"] >= q_low) & (dyn_50["clicks_trend"] <= q_high)].copy()
    sns.boxplot(
        data=trend_viz,
        x="final_result",
        y="clicks_trend",
        hue="final_result",
        order=order_4,
        palette=palette_4,
        legend=False,
        showmeans=True,
        showfliers=False,
        ax=axes[1, 0],
    )
    axes[1, 0].axhline(0.0, color="#c0392b", linestyle="--", linewidth=1.4, label="Độ dốc = 0 (Không đổi)")
    axes[1, 0].set_title("(C) Độ dốc Xu hướng Click theo Tuần (clicks_trend - Mốc 50%)", fontsize=11.5, fontweight="bold")
    axes[1, 0].set_xlabel("Kết quả học tập cuối kỳ", fontsize=10.5, fontweight="bold")
    axes[1, 0].set_ylabel("Độ dốc OLS (clicks / tuần)", fontsize=10.5, fontweight="bold")
    axes[1, 0].legend(loc="lower left", fontsize=9)
    axes[1, 0].grid(axis="y", alpha=0.3)

    # (D) Tỷ lệ At-Risk theo Chiều hướng Xu hướng Tương tác ở Mốc 25% & 50% (Dynamic Active)
    dyn_25 = df_25[df_25["is_snapshot_active"] == 1].copy()
    bucket_rows = []
    for lbl, d_sub in [("Mốc 25% (Dynamic)", dyn_25), ("Mốc 50% (Dynamic)", dyn_50)]:
        conds = [
            d_sub["clicks_trend"] < -5.0,
            (d_sub["clicks_trend"] >= -5.0) & (d_sub["clicks_trend"] < 0.0),
            (d_sub["clicks_trend"] >= 0.0) & (d_sub["clicks_trend"] < 5.0),
            d_sub["clicks_trend"] >= 5.0,
        ]
        labels = ["Giảm mạnh\n(<-5/tuần)", "Giảm nhẹ\n([-5, 0))", "Tăng nhẹ\n([0, 5))", "Tăng trưởng\n(>=5/tuần)"]
        d_sub = d_sub.copy()
        d_sub["trend_tier"] = np.select(conds, labels, default="Giảm nhẹ\n([-5, 0))")
        for tier in labels:
            sub_t = d_sub[d_sub["trend_tier"] == tier]
            if len(sub_t) > 0:
                bucket_rows.append(
                    {"Snapshot": lbl, "Tier": tier, "AtRiskPct": float(sub_t["at_risk"].mean() * 100.0), "N": int(len(sub_t))}
                )
    bucket_df = pd.DataFrame(bucket_rows)
    sns.barplot(
        data=bucket_df,
        x="Tier",
        y="AtRiskPct",
        hue="Snapshot",
        palette={"Mốc 25% (Dynamic)": "#1f77b4", "Mốc 50% (Dynamic)": "#ff7f0e"},
        edgecolor="black",
        ax=axes[1, 1],
    )
    axes[1, 1].set_title("(D) Tỷ lệ Nguy cơ At-Risk theo Nhóm Xu hướng Click (25% vs 50%)", fontsize=11.5, fontweight="bold")
    axes[1, 1].set_xlabel("Nhóm độ dốc clicks_trend", fontsize=10.5, fontweight="bold")
    axes[1, 1].set_ylabel("Tỷ lệ sinh viên At-Risk (%)", fontsize=10.5, fontweight="bold")
    axes[1, 1].set_ylim(0, 82)
    for p in axes[1, 1].patches:
        h = p.get_height()
        if h > 0:
            axes[1, 1].text(p.get_x() + p.get_width() / 2.0, h + 1.2, f"{h:.1f}%", ha="center", fontsize=8.8, fontweight="bold")
    axes[1, 1].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    fig.savefig(figures_dir / "26_temporal_trend_and_engagement_decay.png", dpi=300)
    plt.close(fig)

    # =========================================================================
    # HÌNH 27: 27_inactivity_gap_and_consistency_analysis.png
    # =========================================================================
    logger.info("Đang vẽ Hình 27: 27_inactivity_gap_and_consistency_analysis.png")
    fig, axes = plt.subplots(2, 2, figsize=(16, 11.5))

    # (A) KDE days_since_last_activity tại Mốc 25% (Dynamic Active)
    sns.kdeplot(
        data=dyn_25,
        x="days_since_last_activity",
        hue="at_risk",
        hue_order=[0, 1],
        palette=palette_bin,
        fill=True,
        alpha=0.35,
        linewidth=2,
        common_norm=False,
        legend=False,
        ax=axes[0, 0],
    )
    from matplotlib.patches import Patch
    axes[0, 0].set_title("(A) Số ngày Vắng mặt Kể từ Lần Click cuối (Mốc 25% - Dynamic Active)", fontsize=11.5, fontweight="bold")
    axes[0, 0].set_xlabel("days_since_last_activity (ngày)", fontsize=10.5, fontweight="bold")
    axes[0, 0].set_ylabel("Mật độ phân phối (KDE)", fontsize=10.5, fontweight="bold")
    axes[0, 0].legend(
        handles=[
            Patch(facecolor="#2980b9", edgecolor="#2980b9", alpha=0.45, label="Not At-Risk (0)"),
            Patch(facecolor="#c0392b", edgecolor="#c0392b", alpha=0.45, label="At-Risk (1)"),
        ],
        loc="upper right",
        fontsize=9.5,
    )
    axes[0, 0].grid(alpha=0.3)

    # (B) max_inactivity_streak theo 4 nhóm kết quả tại Mốc 50%
    sns.boxplot(
        data=dyn_50,
        x="final_result",
        y="max_inactivity_streak",
        hue="final_result",
        order=order_4,
        palette=palette_4,
        legend=False,
        showmeans=True,
        ax=axes[0, 1],
    )
    axes[0, 1].set_title("(B) Chuỗi Vắng mặt Liên tiếp Dài nhất (max_inactivity_streak - Mốc 50%)", fontsize=11.5, fontweight="bold")
    axes[0, 1].set_xlabel("Kết quả học tập cuối kỳ", fontsize=10.5, fontweight="bold")
    axes[0, 1].set_ylabel("Số ngày vắng liên tiếp dài nhất", fontsize=10.5, fontweight="bold")
    axes[0, 1].grid(axis="y", alpha=0.3)

    # (C) consistency_score theo 4 nhóm kết quả tại Mốc 50%
    sns.violinplot(
        data=dyn_50,
        x="final_result",
        y="consistency_score",
        hue="final_result",
        order=order_4,
        palette=palette_4,
        legend=False,
        inner="quartile",
        cut=0,
        ax=axes[1, 0],
    )
    axes[1, 0].set_title("(C) Điểm Độ đều đặn Hàng tuần (consistency_score - Mốc 50%)", fontsize=11.5, fontweight="bold")
    axes[1, 0].set_xlabel("Kết quả học tập cuối kỳ", fontsize=10.5, fontweight="bold")
    axes[1, 0].set_ylabel("Shannon Entropy Chuẩn hóa theo Tuần [0, 1]", fontsize=10.5, fontweight="bold")
    axes[1, 0].grid(axis="y", alpha=0.3)

    # (D) Thang rủi ro At-Risk theo ngưỡng max_inactivity_streak ở Mốc 25% & 50%
    streak_bins = [-1, 3, 7, 14, 30, 500]
    streak_labels = ["0-3 ngày", "4-7 ngày", "8-14 ngày", "15-30 ngày", ">30 ngày"]
    streak_rows = []
    for lbl, d_sub in [("Mốc 25% (Dynamic)", dyn_25), ("Mốc 50% (Dynamic)", dyn_50)]:
        d_s = d_sub.copy()
        d_s["gap_bin"] = pd.cut(d_s["max_inactivity_streak"], bins=streak_bins, labels=streak_labels)
        grp = d_s.groupby("gap_bin", observed=False)["at_risk"].agg(["mean", "count"]).reset_index()
        for _, r in grp.iterrows():
            streak_rows.append(
                {
                    "Snapshot": lbl,
                    "GapBin": str(r["gap_bin"]),
                    "AtRiskPct": float(r["mean"] * 100.0) if pd.notna(r["mean"]) else 0.0,
                    "N": int(r["count"]),
                }
            )
    streak_df = pd.DataFrame(streak_rows)
    sns.barplot(
        data=streak_df,
        x="GapBin",
        y="AtRiskPct",
        hue="Snapshot",
        palette={"Mốc 25% (Dynamic)": "#2ecc71", "Mốc 50% (Dynamic)": "#c0392b"},
        edgecolor="black",
        ax=axes[1, 1],
    )
    axes[1, 1].set_title("(D) Thang Nguy cơ At-Risk theo Ngưỡng Vắng mặt Liên tiếp (Early Warning)", fontsize=11.5, fontweight="bold")
    axes[1, 1].set_xlabel("Khoảng chuỗi ngày vắng mặt dài nhất (max_inactivity_streak)", fontsize=10.5, fontweight="bold")
    axes[1, 1].set_ylabel("Tỷ lệ sinh viên At-Risk (%)", fontsize=10.5, fontweight="bold")
    axes[1, 1].set_ylim(0, 102)
    for p in axes[1, 1].patches:
        h = p.get_height()
        if h > 0:
            axes[1, 1].text(p.get_x() + p.get_width() / 2.0, h + 1.4, f"{h:.1f}%", ha="center", fontsize=8.5, fontweight="bold")
    axes[1, 1].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    fig.savefig(figures_dir / "27_inactivity_gap_and_consistency_analysis.png", dpi=300)
    plt.close(fig)

    # =========================================================================
    # HÌNH 28: 28_regularity_weekend_and_burst_patterns.png
    # =========================================================================
    logger.info("Đang vẽ Hình 28: 28_regularity_weekend_and_burst_patterns.png")
    fig, axes = plt.subplots(2, 2, figsize=(16, 11.5))

    # (A) login_regularity ở Mốc 50%
    sns.boxplot(
        data=dyn_50,
        x="final_result",
        y="login_regularity",
        hue="final_result",
        order=order_4,
        palette=palette_4,
        legend=False,
        showmeans=True,
        ax=axes[0, 0],
    )
    axes[0, 0].set_title("(A) Tỷ lệ Tuần Hoạt động (login_regularity - Mốc 50%)", fontsize=11.5, fontweight="bold")
    axes[0, 0].set_xlabel("Kết quả học tập cuối kỳ", fontsize=10.5, fontweight="bold")
    axes[0, 0].set_ylabel("Tỷ lệ tuần có truy cập LMS [0, 1]", fontsize=10.5, fontweight="bold")
    axes[0, 0].grid(axis="y", alpha=0.3)

    # (B) weekend_ratio theo 4 nhóm kết quả tại Mốc 100% (chỉ SV có click > 0)
    act_100 = df_100[df_100["total_clicks"] > 0].copy()
    sns.boxplot(
        data=act_100,
        x="final_result",
        y="weekend_ratio",
        hue="final_result",
        order=order_4,
        palette=palette_4,
        legend=False,
        showmeans=True,
        showfliers=False,
        ax=axes[0, 1],
    )
    axes[0, 1].axhline(2.0 / 7.0, color="gray", linestyle="--", linewidth=1.2, label="Tỷ lệ đều đặn lý thuyết (2/7 ≈ 0.286)")
    axes[0, 1].set_title("(B) Tỷ trọng Học tập Cuối tuần (weekend_ratio - Mốc 100%)", fontsize=11.5, fontweight="bold")
    axes[0, 1].set_xlabel("Kết quả học tập cuối kỳ", fontsize=10.5, fontweight="bold")
    axes[0, 1].set_ylabel("weekend_clicks / total_clicks", fontsize=10.5, fontweight="bold")
    axes[0, 1].legend(loc="upper right", fontsize=9)
    axes[0, 1].grid(axis="y", alpha=0.3)

    # (C) Burst Ratio vs Consistency Score tại Mốc 50% (phân nhóm Cramming vs Steady)
    burst_bins = pd.cut(
        dyn_50["burst_ratio"], bins=[-0.01, 0.0, 0.06, 0.12, 1.0], labels=["0 (Không dồn)", "(0, 0.06]", "(0.06, 0.12]", ">0.12 (Học dồn)"]
    )
    b_df = dyn_50.copy()
    b_df["burst_tier"] = burst_bins
    sns.boxplot(
        data=b_df,
        x="burst_tier",
        y="consistency_score",
        hue="at_risk",
        palette=palette_bin,
        showfliers=False,
        ax=axes[1, 0],
    )
    axes[1, 0].set_title("(C) Tương quan Học dồn (burst_ratio) & Độ đều đặn (Mốc 50%)", fontsize=11.5, fontweight="bold")
    axes[1, 0].set_xlabel("Mức độ tỷ lệ ngày học dồn (burst_ratio)", fontsize=10.5, fontweight="bold")
    axes[1, 0].set_ylabel("consistency_score [0, 1]", fontsize=10.5, fontweight="bold")
    handles, _ = axes[1, 0].get_legend_handles_labels()
    axes[1, 0].legend(handles, ["Not At-Risk (0)", "At-Risk (1)"], loc="lower left", fontsize=9)
    axes[1, 0].grid(axis="y", alpha=0.3)

    # (D) Early Warning Score Trajectory (last_minus_first_score) cho SV nộp >= 2 bài ở Mốc 50% & 100%
    sub2_50 = df_50[df_50["num_submissions"] >= 2].copy()
    sub2_100 = df_100[df_100["num_submissions"] >= 2].copy()
    sub2_50["Mốc"] = "Mốc 50% (>=2 bài)"
    sub2_100["Mốc"] = "Mốc 100% (>=2 bài)"
    comb_sc = pd.concat([sub2_50[["final_result", "last_minus_first_score", "Mốc"]], sub2_100[["final_result", "last_minus_first_score", "Mốc"]]])
    sns.boxplot(
        data=comb_sc,
        x="final_result",
        y="last_minus_first_score",
        hue="Mốc",
        order=order_4,
        palette={"Mốc 50% (>=2 bài)": "#3498db", "Mốc 100% (>=2 bài)": "#9b59b6"},
        showfliers=False,
        ax=axes[1, 1],
    )
    axes[1, 1].axhline(0.0, color="#c0392b", linestyle="--", linewidth=1.3, label="Không đổi điểm (0)")
    axes[1, 1].set_title("(D) Quỹ đạo Biến thiên Điểm (last_minus_first_score, SV nộp >=2 bài)", fontsize=11.5, fontweight="bold")
    axes[1, 1].set_xlabel("Kết quả học tập cuối kỳ", fontsize=10.5, fontweight="bold")
    axes[1, 1].set_ylabel("Điểm bài sau cùng - Bài đầu tiên", fontsize=10.5, fontweight="bold")
    axes[1, 1].legend(loc="lower left", fontsize=9)
    axes[1, 1].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    fig.savefig(figures_dir / "28_regularity_weekend_and_burst_patterns.png", dpi=300)
    plt.close(fig)

    # =========================================================================
    # HÌNH 29: 29_week6_early_warning_breakthrough_25_50_75_100.png
    # =========================================================================
    logger.info("Đang vẽ Hình 29: 29_week6_early_warning_breakthrough_25_50_75_100.png")
    exp_df = pd.DataFrame(exp_results)
    ew_df = pd.DataFrame(ew_results)
    fig, axes = plt.subplots(1, 2, figsize=(16.5, 6.4))

    # (A) ROC-AUC trên Dynamic Active qua 4 mốc: week5 vs week6 (RF, GBDT, LightGBM)
    dyn_exp = exp_df[exp_df["cohort"] == "dynamic_active"].copy()
    plot_specs = [
        ("week5_expanded", "Random Forest", "--", "#7f8c8d", "o", "W5: Random Forest"),
        ("week6_advanced", "Random Forest", "-", "#2980b9", "s", "W6: Random Forest"),
        ("week6_advanced", "Gradient Boosting", "-", "#27ae60", "^", "W6: Gradient Boosting"),
        ("week6_advanced", "LightGBM", "-", "#8e44ad", "D", "W6: LightGBM"),
    ]
    x_pcts = [25, 50, 75, 100]
    for fset, mname, ls, col, mk, lbl in plot_specs:
        sub = dyn_exp[(dyn_exp["feature_set"] == fset) & (dyn_exp["model"] == mname)].sort_values("cutoff_pct")
        if not sub.empty:
            axes[0].plot(x_pcts, sub["test_roc_auc"].values, linestyle=ls, color=col, marker=mk, linewidth=2.3, markersize=7, label=lbl)
            if mname == "LightGBM":
                for xp, yv in zip(x_pcts, sub["test_roc_auc"].values):
                    axes[0].text(xp, yv + 0.006, f"{yv:.4f}", ha="center", fontsize=8.8, fontweight="bold", color=col)
    axes[0].axhline(0.80, color="#c0392b", linestyle=":", linewidth=1.6, label="Mục tiêu Dự án AUC = 0.80")
    axes[0].set_title("(A) Tiến hóa ROC-AUC Test trên Dynamic Active: Tuần 5 vs Tuần 6", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Mốc thời gian cắt Snapshot (% Thời lượng khóa)", fontsize=10.5, fontweight="bold")
    axes[0].set_ylabel("Test ROC-AUC", fontsize=10.5, fontweight="bold")
    axes[0].set_xticks(x_pcts)
    axes[0].set_xticklabels(["25%", "50%", "75%", "100%"])
    axes[0].set_ylim(0.75, 0.99)
    axes[0].legend(loc="lower right", fontsize=9.2)
    axes[0].grid(alpha=0.3)

    # (B) Đột phá Recall & F1 trên Dynamic Active với LightGBM Standard vs Balanced Tuned
    lgb_std = dyn_exp[(dyn_exp["feature_set"] == "week6_advanced") & (dyn_exp["model"] == "LightGBM")].sort_values("cutoff_pct")
    lgb_ew = ew_df[(ew_df["cohort"] == "dynamic_active") & (ew_df["model"] == "LightGBM (Balanced)")].sort_values("cutoff_pct")

    axes[1].plot(x_pcts, lgb_std["test_recall"].values, linestyle="--", color="#95a5a6", marker="o", linewidth=2.0, label="LGBM Chuẩn (tau=0.50) - Test Recall")
    axes[1].plot(x_pcts, lgb_ew["test_recall"].values, linestyle="-", color="#c0392b", marker="D", linewidth=2.6, markersize=7.5, label="LGBM Early Warning (Val-tuned tau*) - Test Recall")
    axes[1].plot(x_pcts, lgb_ew["test_f1"].values, linestyle="-", color="#e67e22", marker="s", linewidth=2.3, markersize=6.8, label="LGBM Early Warning (Val-tuned tau*) - Test F1")
    axes[1].axhline(0.75, color="#27ae60", linestyle=":", linewidth=1.8, label="Ngưỡng DoD Tuần 6 Recall >= 0.75")
    axes[1].axhline(0.70, color="#16a085", linestyle="-.", linewidth=1.4, label="Mục tiêu Đề cương F1 >= 0.70")

    label_box = dict(boxstyle="round,pad=0.15", facecolor="white", edgecolor="none", alpha=0.88)
    pos_std = {25: (25, -0.026), 50: (50, -0.026), 75: (75, -0.028), 100: (100 - 4.2, +0.022)}
    pos_rec = {25: (25, +0.020), 50: (50, +0.022), 75: (75 + 5.8, -0.008), 100: (100, -0.044)}
    pos_f1 = {25: (25 + 3.2, -0.006), 50: (50 + 4.8, -0.012), 75: (75, +0.018), 100: (100, +0.018)}

    for xp, r_std, r_ew, f_ew, tau_v in zip(
        x_pcts,
        lgb_std["test_recall"].values,
        lgb_ew["test_recall"].values,
        lgb_ew["test_f1"].values,
        lgb_ew["tuned_threshold_val"].values,
    ):
        sx, sy = pos_std[xp]
        rx, ry = pos_rec[xp]
        fx, fy = pos_f1[xp]
        axes[1].text(sx, r_std + sy, f"{r_std:.3f}", ha="center", va="center", fontsize=8.4, color="#555555", bbox=label_box)
        axes[1].text(rx, r_ew + ry, f"{r_ew:.3f} (tau={tau_v:.2f})" if xp < 75 else f"{r_ew:.3f}\n(tau={tau_v:.2f})", ha="center", va="center", fontsize=8.2, fontweight="bold", color="#c0392b", bbox=label_box)
        axes[1].text(fx, f_ew + fy, f"{f_ew:.3f}", ha="center", va="center", fontsize=8.4, fontweight="bold", color="#d35400", bbox=label_box)

    axes[1].set_title("(B) Đột phá Cảnh báo Sớm: Chuẩn vs Balanced + Val-Tuned Threshold", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Mốc thời gian cắt Snapshot (% Thời lượng khóa)", fontsize=10.5, fontweight="bold")
    axes[1].set_ylabel("Test Recall / F1-Score", fontsize=10.5, fontweight="bold")
    axes[1].set_xticks(x_pcts)
    axes[1].set_xticklabels(["25%", "50%", "75%", "100%"])
    axes[1].set_xlim(18, 108)
    axes[1].set_ylim(0.50, 1.00)
    axes[1].legend(loc="lower right", fontsize=8.8)
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    fig.savefig(figures_dir / "29_week6_early_warning_breakthrough_25_50_75_100.png", dpi=300)
    plt.close(fig)

    # =========================================================================
    # HÌNH 30: 30_week6_feature_importance_evolution.png
    # =========================================================================
    logger.info("Đang vẽ Hình 30: 30_week6_feature_importance_evolution.png")
    fig, axes = plt.subplots(1, 2, figsize=(18.5, 7.2))

    # (A) Top 20 Đặc trưng quan trọng nhất tại Mốc 25% (RF Gini Importance)
    top20_25 = catalog_df.sort_values("rf_gini_importance_25", ascending=False).head(20).iloc[::-1]
    colors_20 = [
        "#e67e22" if int(w6) == 1 else ("#27ae60" if fg.startswith("5.3") else "#2980b9")
        for w6, fg in zip(top20_25["is_week6_new"], top20_25["feature_group"])
    ]
    bars = axes[0].barh(top20_25["feature_name"], top20_25["rf_gini_importance_25"], color=colors_20, edgecolor="black")
    axes[0].set_title("(A) Top 20 Đặc trưng tại Mốc Sớm 25% (Màu cam = Mới Tuần 6)", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Random Forest Gini Importance (Mốc 25%)", fontsize=10.5, fontweight="bold")
    for b, val in zip(bars, top20_25["rf_gini_importance_25"]):
        axes[0].text(val + 0.001, b.get_y() + b.get_height() / 2.0, f"{val:.4f}", va="center", fontsize=8.3, fontweight="bold")
    axes[0].set_xlim(0, top20_25["rf_gini_importance_25"].max() * 1.18)
    axes[0].grid(axis="x", alpha=0.3)

    # Custom legend for (A)
    from matplotlib.patches import Patch
    legend_elems = [
        Patch(facecolor="#e67e22", edgecolor="black", label="Đặc trưng Mới Tuần 6 (Temporal/Regularity/Trajectory)"),
        Patch(facecolor="#27ae60", edgecolor="black", label="Nhóm 5.3 Assessment (Tuần 5)"),
        Patch(facecolor="#2980b9", edgecolor="black", label="Nhóm 5.1 / 5.2 / 5.4 Click & Context (Tuần 5)"),
    ]
    axes[0].legend(handles=legend_elems, loc="lower right", fontsize=8.5)

    # (B) Tỷ trọng Importance theo 7 Nhóm đặc trưng qua cả 4 mốc 25%, 50%, 75%, 100%
    group_order = [
        "5.1 Click-based",
        "5.2 Activity-based",
        "5.3 Assessment-based",
        "5.4 Demographics & Context",
        "6.1 Temporal & Decay",
        "6.2 Regularity & Burst",
        "6.3 Score Trajectory",
    ]
    grp_share = pd.DataFrame(
        {
            "25% Snapshot": catalog_df.groupby("feature_group")["rf_gini_importance_25"].sum(),
            "50% Snapshot": catalog_df.groupby("feature_group")["rf_gini_importance_50"].sum(),
            "75% Snapshot": catalog_df.groupby("feature_group")["rf_gini_importance_75"].sum(),
            "100% Snapshot": catalog_df.groupby("feature_group")["rf_gini_importance_100"].sum(),
        }
    ).reindex(group_order).fillna(0.0) * 100.0
    grp_share_melt = grp_share.reset_index().melt(id_vars="feature_group", var_name="Snapshot", value_name="ImportanceShare")
    sns.barplot(
        data=grp_share_melt,
        x="feature_group",
        y="ImportanceShare",
        hue="Snapshot",
        palette={"25% Snapshot": "#3498db", "50% Snapshot": "#f39c12", "75% Snapshot": "#27ae60", "100% Snapshot": "#2c3e50"},
        edgecolor="black",
        ax=axes[1],
    )
    axes[1].set_title("(B) Phân bổ Tỷ trọng Importance theo 7 Nhóm Đặc trưng (25% -> 50% -> 75% -> 100%)", fontsize=11.5, fontweight="bold")
    axes[1].set_xlabel("Nhóm Đặc trưng", fontsize=10.5, fontweight="bold")
    axes[1].set_ylabel("Tổng Tỷ trọng Importance (%)", fontsize=10.5, fontweight="bold")
    axes[1].tick_params(axis="x", rotation=25)
    for p in axes[1].patches:
        h = p.get_height()
        if h >= 1.0:
            axes[1].text(p.get_x() + p.get_width() / 2.0, h + 0.5, f"{h:.1f}", ha="center", fontsize=7.5, fontweight="bold")
    axes[1].set_ylim(0, grp_share.values.max() * 1.18)
    axes[1].legend(loc="upper right", fontsize=9)
    axes[1].grid(axis="y", alpha=0.3)

    plt.tight_layout()
    fig.savefig(figures_dir / "30_week6_feature_importance_evolution.png", dpi=300)
    plt.close(fig)


def run_week6_notebook_stage(
    snapshots: Dict[float, pd.DataFrame],
    engineer: OULADFeatureEngineer,
    save_figures: bool = True,
) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    catalog_df, importance_summary = build_week6_feature_catalog(
        df_100=snapshots[1.00],
        df_25=snapshots[0.25],
        df_50=snapshots[0.50],
        df_75=snapshots[0.75],
        engineer=engineer,
    )
    catalog_path = WEEK6_PROCESSED_DIR / "feature_catalog_week6.csv"
    catalog_df.to_csv(catalog_path, index=False)
    logger.info(f"Đã lưu bảng danh mục đặc trưng Tuần 6: {catalog_path} ({len(catalog_df)} đặc trưng)")

    imp_json_path = WEEK6_PROCESSED_DIR / "feature_importance_week6.json"
    with open(imp_json_path, "w", encoding="utf-8") as f:
        json.dump(importance_summary, f, indent=2, ensure_ascii=False)

    exp_payload = run_week6_benchmark_experiments(snapshots=snapshots, engineer=engineer)

    w6_stats = {}
    for pct, d_s in snapshots.items():
        key = f"{int(pct * 100)}pct"
        d_dyn = engineer.filter_cohort(d_s, "dynamic_active")
        w6_stats[key] = {
            "full_cohort_size": int(len(d_s)),
            "dynamic_active_size": int(len(d_dyn)),
            "dynamic_active_at_risk_rate": round(float(d_dyn["at_risk"].mean()), 4),
            "mean_clicks_trend_not_at_risk": round(float(d_dyn[d_dyn["at_risk"] == 0]["clicks_trend"].mean()), 3),
            "mean_clicks_trend_at_risk": round(float(d_dyn[d_dyn["at_risk"] == 1]["clicks_trend"].mean()), 3),
            "mean_days_since_last_not_at_risk": round(float(d_dyn[d_dyn["at_risk"] == 0]["days_since_last_activity"].mean()), 2),
            "mean_days_since_last_at_risk": round(float(d_dyn[d_dyn["at_risk"] == 1]["days_since_last_activity"].mean()), 2),
            "mean_max_streak_not_at_risk": round(float(d_dyn[d_dyn["at_risk"] == 0]["max_inactivity_streak"].mean()), 2),
            "mean_max_streak_at_risk": round(float(d_dyn[d_dyn["at_risk"] == 1]["max_inactivity_streak"].mean()), 2),
            "mean_consistency_not_at_risk": round(float(d_dyn[d_dyn["at_risk"] == 0]["consistency_score"].mean()), 4),
            "mean_consistency_at_risk": round(float(d_dyn[d_dyn["at_risk"] == 1]["consistency_score"].mean()), 4),
            "mean_login_regularity_not_at_risk": round(float(d_dyn[d_dyn["at_risk"] == 0]["login_regularity"].mean()), 4),
            "mean_login_regularity_at_risk": round(float(d_dyn[d_dyn["at_risk"] == 1]["login_regularity"].mean()), 4),
        }

    full_payload = {
        "week6_progression_stats": w6_stats,
        "importance_summary": importance_summary,
        "experiments": exp_payload["experiments"],
        "early_warning_tuned": exp_payload["early_warning_tuned"],
    }

    metrics_w6_path = WEEK6_PROCESSED_DIR / "week6_experiment_metrics.json"
    with open(metrics_w6_path, "w", encoding="utf-8") as f:
        json.dump(full_payload, f, indent=2, ensure_ascii=False)
    logger.info(f"Đã lưu week6_experiment_metrics.json (chỉ tại data/processed/week6_advanced_features/): {metrics_w6_path}")

    if save_figures:
        generate_week6_figures(
            snapshots=snapshots,
            catalog_df=catalog_df,
            exp_results=exp_payload["experiments"],
            ew_results=exp_payload["early_warning_tuned"],
            figures_dir=FIGURES_DIR,
        )

    return catalog_df, full_payload


catalog_w6, w6_metrics = run_week6_notebook_stage(snapshots=snapshots, engineer=engineer, save_figures=True)

print(f"Tổng số đặc trưng Danh mục Tuần 6: {len(catalog_w6)} (trong đó {int(catalog_w6['is_week6_new'].sum())} đặc trưng mới Tuần 6)")
top15_w6 = catalog_w6.sort_values("rf_gini_importance_25", ascending=False).head(15)
display(top15_w6[["feature_name", "feature_group", "is_week6_new", "point_biserial_r_25", "cohens_d_25", "rf_gini_importance_25", "rf_gini_importance_100"]])

ew_df = pd.DataFrame(w6_metrics["early_warning_tuned"])
print("\nHiệu năng Early Warning (Dynamic Active Cohort, LightGBM Balanced + Val-Tuned Threshold):")
display(ew_df[(ew_df["cohort"] == "dynamic_active") & (ew_df["model"] == "LightGBM (Balanced)")][[
    "snapshot", "n_cohort", "default_05_test_recall", "tuned_threshold_val", "test_recall", "test_precision", "test_f1", "test_roc_auc", "test_pr_auc"
]])

w6_exp = pd.DataFrame(w6_metrics["experiments"])
dyn_lgb = w6_exp[(w6_exp["cohort"] == "dynamic_active") & (w6_exp["model"] == "LightGBM")]
print("\nĐối sánh LightGBM Chuẩn (week5_expanded vs week6_advanced) trên Dynamic Active Cohort:")
display(dyn_lgb[["snapshot", "feature_set", "n_features_encoded", "test_roc_auc", "test_f1", "test_recall", "test_precision"]])


2026-11-24 10:06:33,719 [INFO] Đang xây dựng Danh mục 92 Đặc trưng Tuần 6 và tính Importance...


2026-11-24 10:08:02,106 [INFO] Đã lưu bảng danh mục đặc trưng Tuần 6: C:\Users\mrore\Project_I\data\processed\week6_advanced_features\feature_catalog_week6.csv (92 đặc trưng)


2026-11-24 10:08:02,111 [INFO] Đang chạy thực nghiệm đối sánh Tuần 6 (4 snapshots x 3 cohorts x 4 models + Early Warning)...


2026-11-24 10:11:08,163 [INFO] Hoàn tất benchmark snapshot 25%


2026-11-24 10:14:25,014 [INFO] Hoàn tất benchmark snapshot 50%


2026-11-24 10:18:01,225 [INFO] Hoàn tất benchmark snapshot 75%


... (9 outputs left out)


Hiệu năng Early Warning (Dynamic Active Cohort, LightGBM Balanced + Val-Tuned Threshold):


,snapshot,n_cohort,default_05_test_recall,tuned_threshold_val,test_recall,test_precision,test_f1,test_roc_auc,test_pr_auc
8,25%,26248,0.7146,0.4300,0.7713,0.6879,0.7272,0.8492,0.8291
17,50%,24601,0.7699,0.4800,0.7798,0.7751,0.7774,0.9070,0.8804
26,75%,23159,0.8020,0.5000,0.8020,0.8430,0.8220,0.9429,0.9208
35,100%,22522,0.8294,0.6600,0.7765,0.9307,0.8466,0.9623,0.9435



Đối sánh LightGBM Chuẩn (week5_expanded vs week6_advanced) trên Dynamic Active Cohort:


,snapshot,feature_set,n_features_encoded,test_roc_auc,test_f1,test_recall,test_precision
19,25%,week5_expanded,98,0.8473,0.7140,0.6572,0.7816
23,25%,week6_advanced,114,0.8515,0.7172,0.6617,0.7828
43,50%,week5_expanded,98,0.9002,0.7612,0.7031,0.8297
47,50%,week6_advanced,114,0.9070,0.7688,0.7145,0.8320
67,75%,week5_expanded,98,0.9392,0.8147,0.7480,0.8945
71,75%,week6_advanced,114,0.9431,0.8266,0.7597,0.9066
91,100%,week5_expanded,98,0.9577,0.8488,0.7814,0.9289
95,100%,week6_advanced,114,0.9626,0.8541,0.7863,0.9347


## 5. Bộ Kiểm Thử Tự Động Pipeline Feature Engineering Tuần 5–6 (18/18 In-Notebook Unit Tests)
Thực thi toàn bộ 18 Unit Tests kiểm tra Zero Temporal Leakage, loại bỏ tuyệt đối `Exam`, tính đơn điệu tích lũy, khớp tổng 20 loại hoạt động, giới hạn Shannon Entropy, kích thước Cohort, Zero Statistical Leakage ở ma trận 114 biến `week6_advanced`, sự tồn tại của các thư mục tuần trong `data/processed/` & 10 biểu đồ `21_*.png`–`30_*.png`, và ngưỡng đột phá Cảnh báo Sớm $\text{Recall} \ge 0.75$ tại mốc 25%.

In [6]:
class TestOULADFeatureEngineering(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        """Tái sử dụng engineer đã nạp dữ liệu thô và đọc 4 file snapshot CSV đã xuất trên disk để kiểm chứng toàn vẹn."""
        cls.engineer = engineer
        cls.snapshots = {}
        for pct in [0.25, 0.50, 0.75, 1.00]:
            csv_path = SNAPSHOTS_DIR / f"features_snapshot_{int(pct * 100)}pct.csv"
            assert csv_path.exists(), f"Thiếu file snapshot {csv_path}"
            cls.snapshots[pct] = pd.read_csv(csv_path)

        # Kiểm tra không còn file metrics JSON trong reports/ và không còn tệp rời tại gốc data/processed/
        assert not (REPORTS_DIR / "week5_experiment_metrics.json").exists(), "Vẫn còn bản sao JSON trong reports/!"
        assert not (REPORTS_DIR / "week6_experiment_metrics.json").exists(), "Vẫn còn bản sao JSON trong reports/!"
        assert not any(p.is_file() for p in PROCESSED_DIR.iterdir()), "Còn tệp rời ngoài thư mục tuần tại data/processed/!"

    def test_01_snapshot_files_and_row_counts(self):
        """Test 1: Kiểm tra cả 4 lát cắt (25%, 50%, 75%, 100%) đều có đúng 32,593 lượt đăng ký."""
        for pct, df in self.snapshots.items():
            self.assertEqual(
                len(df),
                32593,
                f"Snapshot {int(pct*100)}% có số dòng {len(df)} != 32,593!",
            )
            self.assertEqual(df["id_student"].nunique(), 28785)

    def test_02_zero_temporal_leakage_vle(self):
        """Test 2: Đảm bảo không có tương tác VLE nào vượt quá mốc cắt (last_click_day <= cutoff_day)."""
        for pct, df in self.snapshots.items():
            valid_last = df[df["last_click_day"].notna()]
            violations = (valid_last["last_click_day"] > valid_last["cutoff_day"] + 1e-9).sum()
            self.assertEqual(
                violations,
                0,
                f"Phát hiện {violations} bản ghi rò rỉ VLE tương lai tại mốc {int(pct*100)}%!",
            )
            # active_days_in_course không được vượt quá cutoff_day + 1
            in_course_violations = (df["active_days_in_course"] > (df["cutoff_day"] + 1.0)).sum()
            self.assertEqual(in_course_violations, 0)

    def test_03_zero_temporal_leakage_assessments(self):
        """Test 3: Đảm bảo chỉ các bài Coursework nộp tại hoặc trước cutoff_day được tính."""
        # Kiểm tra trực tiếp trên khóa học GGG (nơi TMA1 có hạn nộp ngày 61 > cutoff 25% của GGG_2014B là 60.25)
        df_25 = self.snapshots[0.25]
        df_100 = self.snapshots[1.00]
        self.assertLess(
            df_25["num_submissions"].sum(),
            df_100["num_submissions"].sum(),
            "Tổng số bài nộp ở mốc 25% phải nhỏ hơn mốc 100%!",
        )
        # Kiểm tra khớp tuyệt đối với phép lọc độc lập từ bảng raw
        cw_sub = self.engineer._coursework_submissions
        sub_25_raw = cw_sub[
            (cw_sub["date_submitted"] <= cw_sub["module_presentation_length"] * 0.25)
            & (cw_sub["score"].notna())
        ]
        self.assertEqual(int(df_25["num_submissions"].sum()), len(sub_25_raw))

    def test_04_exam_strictly_excluded(self):
        """Test 4: Đảm bảo bài thi cuối kỳ (assessment_type == 'Exam') bị loại bỏ hoàn toàn khỏi đặc trưng."""
        df_100 = self.snapshots[1.00]
        # Tổng số bài nộp đã chấm ở mốc 100% không được vượt quá tổng số bài coursework của khóa học
        exceed_cw = (df_100["num_submissions"] > df_100["total_course_assessments"]).sum()
        self.assertEqual(
            exceed_cw,
            0,
            f"Có {exceed_cw} sinh viên có số bài nộp vượt quá tổng số bài coursework (nghi ngờ lọt Exam)!",
        )
        # Kiểm tra tổng số dòng coursework assessments là 182 (206 - 24 Exam)
        self.assertEqual(len(self.engineer._coursework_assessments), 182)

    def test_05_monotonic_cumulative_features(self):
        """Test 5: Kiểm tra tính đơn điệu không giảm (25% <= 50% <= 75% <= 100%) của các biến tích lũy."""
        cumulative_cols = [
            "total_clicks",
            "active_days",
            "active_days_in_course",
            "activity_diversity",
            "cognitive_clicks",
            "social_clicks",
            "num_submissions",
            "num_submissions_raw",
            "late_submissions",
            "earned_weight_score",
        ]
        pairs = [(0.25, 0.50), (0.50, 0.75), (0.75, 1.00)]
        for c in cumulative_cols:
            for p_early, p_late in pairs:
                v_early = self.snapshots[p_early][c].values
                v_late = self.snapshots[p_late][c].values
                diff = v_late - v_early
                self.assertTrue(
                    np.all(diff >= -1e-9),
                    f"Biến tích lũy {c} bị giảm giữa mốc {int(p_early*100)}% và {int(p_late*100)}%!",
                )

    def test_06_activity_types_and_ratios_sum_to_one(self):
        """Test 6: Kiểm tra đủ 20 loại activity_type (gồm clicks_glossary) và tổng bằng total_clicks."""
        df_100 = self.snapshots[1.00]
        act_cols = [f"clicks_{a}" for a in OULADFeatureEngineer.ALL_ACTIVITY_TYPES]
        self.assertEqual(len(act_cols), 20)
        self.assertIn("clicks_glossary", df_100.columns)

        for col in act_cols:
            self.assertIn(col, df_100.columns, f"Thiếu cột {col}!")

        sum_20 = df_100[act_cols].sum(axis=1).values
        np.testing.assert_allclose(
            sum_20,
            df_100["total_clicks"].values,
            rtol=1e-6,
            err_msg="Tổng click của 20 loại activity_type không khớp với total_clicks!",
        )

        # Kiểm tra tổng tỷ trọng của 20 loại bằng 1.0 đối với sinh viên có total_clicks > 0
        active_mask = df_100["total_clicks"] > 0
        ratios_sum = sum_20[active_mask] / df_100.loc[active_mask, "total_clicks"].values
        np.testing.assert_allclose(ratios_sum, np.ones_like(ratios_sum), rtol=1e-6)

    def test_07_shannon_entropy_and_diversity_bounds(self):
        """Test 7: Kiểm tra miền giá trị toán học của activity_diversity và activity_entropy."""
        max_entropy = np.log(20.0)
        for pct, df in self.snapshots.items():
            self.assertTrue((df["activity_diversity"] >= 0).all())
            self.assertTrue((df["activity_diversity"] <= 20).all())
            self.assertTrue((df["activity_entropy"] >= 0.0).all())
            self.assertTrue((df["activity_entropy"] <= max_entropy + 1e-6).all())

            # Nếu chỉ truy cập <= 1 loại tài nguyên thì Shannon Entropy phải bằng 0
            single_or_zero = df["activity_diversity"] <= 1
            np.testing.assert_allclose(
                df.loc[single_or_zero, "activity_entropy"].values,
                0.0,
                atol=1e-9,
            )

    def test_08_submission_rate_and_weighted_score_invariants(self):
        """Test 8: Kiểm tra bất biến tỷ lệ nộp bài in [0, 1] và min_score <= weighted_avg_score <= max_score."""
        for pct, df in self.snapshots.items():
            self.assertTrue((df["submission_rate"] >= 0.0).all())
            self.assertTrue((df["submission_rate"] <= 1.0).all())
            self.assertTrue((df["late_submission_rate"] >= 0.0).all())
            self.assertTrue((df["late_submission_rate"] <= 1.0).all())

            has_score = df["avg_score"].notna()
            sub = df[has_score]
            self.assertTrue(
                (sub["weighted_avg_score"] >= sub["min_score"] - 1e-6).all(),
                "weighted_avg_score nhỏ hơn min_score!",
            )
            self.assertTrue(
                (sub["weighted_avg_score"] <= sub["max_score"] + 1e-6).all(),
                "weighted_avg_score lớn hơn max_score!",
            )

    def test_09_cohort_filtering_definitions(self):
        """Test 9: Kiểm tra kích thước của 3 định nghĩa Cohort tại các mốc dự đoán."""
        df_25 = self.snapshots[0.25]
        df_50 = self.snapshots[0.50]
        df_75 = self.snapshots[0.75]
        df_100 = self.snapshots[1.00]

        full_c = self.engineer.filter_cohort(df_100, "full")
        pre_c = self.engineer.filter_cohort(df_100, "pre_course_active")
        self.assertEqual(len(full_c), 32593)
        self.assertEqual(len(pre_c), 29915)

        dyn_25 = self.engineer.filter_cohort(df_25, "dynamic_active")
        dyn_50 = self.engineer.filter_cohort(df_50, "dynamic_active")
        dyn_75 = self.engineer.filter_cohort(df_75, "dynamic_active")
        dyn_100 = self.engineer.filter_cohort(df_100, "dynamic_active")

        self.assertEqual(len(dyn_25), 26248)
        self.assertEqual(len(dyn_50), 24601)
        self.assertEqual(len(dyn_75), 23159)
        self.assertEqual(len(dyn_100), 22522)

    def test_10_ml_matrix_zero_nan_inf_and_leakage_free(self):
        """Test 10: Kiểm tra ma trận đầu ra cho ML không chứa NaN/Inf và không bị rò rỉ từ tập Test."""
        df_25 = self.snapshots[0.25]
        tr, va, te = train_test_split_group(df_25, split_strategy="group_stratified", random_state=42)
        X_tr, X_va, X_te = self.engineer.prepare_ml_matrices(tr, va, te, feature_set="week5_expanded")

        for name, mat in [("Train", X_tr), ("Val", X_va), ("Test", X_te)]:
            self.assertEqual(mat.isna().sum().sum(), 0, f"Ma trận {name} còn chứa NaN!")
            self.assertFalse(np.isinf(mat.values).any(), f"Ma trận {name} còn chứa Inf!")

        # Thay đổi cực đoan tập Test và kiểm tra X_tr không hề thay đổi (Zero Statistical Leakage)
        te_corrupted = te.copy()
        te_corrupted["total_clicks"] = 9999999.0
        te_corrupted["avg_score"] = 100.0
        X_tr_2, _, _ = self.engineer.prepare_ml_matrices(tr, va, te_corrupted, feature_set="week5_expanded")
        np.testing.assert_allclose(X_tr.values, X_tr_2.values, err_msg="Tập Train bị rò rỉ thống kê từ tập Test!")

    def test_11_group_stratified_split_zero_overlap(self):
        """Test 11: Đảm bảo Group-Stratified Split có đúng 0 sinh viên trùng lặp giữa Train, Val, Test."""
        tr, va, te = train_test_split_group(
            self.snapshots[1.00], split_strategy="group_stratified", random_state=42
        )
        s_tr, s_va, s_te = set(tr["id_student"]), set(va["id_student"]), set(te["id_student"])
        self.assertEqual(len(s_tr.intersection(s_va)), 0)
        self.assertEqual(len(s_tr.intersection(s_te)), 0)
        self.assertEqual(len(s_va.intersection(s_te)), 0)

    def test_12_catalog_and_metrics_artifacts_exist(self):
        """Test 12: Kiểm tra file danh mục đặc trưng, feature importance và JSON kết quả thực nghiệm Tuần 5 tồn tại và hợp lệ."""
        cat_path = WEEK5_PROCESSED_DIR / "feature_catalog_week5.csv"
        imp_path = WEEK5_PROCESSED_DIR / "feature_importance_week5.json"
        met_path = WEEK5_PROCESSED_DIR / "week5_experiment_metrics.json"
        self.assertTrue(cat_path.exists(), "Thiếu feature_catalog_week5.csv")
        self.assertTrue(imp_path.exists(), "Thiếu feature_importance_week5.json")
        self.assertTrue(met_path.exists(), "Thiếu week5_experiment_metrics.json")
        cat_df = pd.read_csv(cat_path)
        self.assertEqual(len(cat_df), 76)
        for col in ["point_biserial_r_50", "point_biserial_r_75", "rf_gini_importance_75", "mutual_info_75"]:
            self.assertIn(col, cat_df.columns, f"Thiếu cột {col} trong feature_catalog_week5.csv")
        with open(imp_path, "r", encoding="utf-8") as f:
            imp_payload = json.load(f)
        for key in ["top20_by_rf_25pct", "top20_by_rf_50pct", "top20_by_rf_75pct", "top20_by_rf_100pct"]:
            self.assertIn(key, imp_payload)
        with open(met_path, "r", encoding="utf-8") as f:
            payload = json.load(f)
        self.assertIn("experiments", payload)
        self.assertGreater(len(payload["experiments"]), 20)

    def test_13_week6_temporal_regularity_bounds(self):
        """Test 13 (Tuần 6): Kiểm tra đủ 16 đặc trưng mới, 0 NaN và miền giá trị toán học hợp lệ."""
        expected_w6_cols = [
            "clicks_week_1_to_4",
            "clicks_last_2_weeks",
            "clicks_trend",
            "engagement_decay",
            "days_since_last_activity",
            "inactivity_ratio",
            "max_inactivity_streak",
            "consistency_score",
            "active_weeks",
            "login_regularity",
            "weekend_clicks",
            "weekend_ratio",
            "burst_count",
            "burst_ratio",
            "last_minus_first_score",
            "score_trend_slope",
            "days_before_start",
        ]
        unit_bounded = ["consistency_score", "login_regularity", "weekend_ratio", "burst_ratio", "inactivity_ratio"]
        non_negative = ["clicks_week_1_to_4", "clicks_last_2_weeks", "engagement_decay", "active_weeks", "burst_count"]

        for pct, df in self.snapshots.items():
            for col in expected_w6_cols:
                self.assertIn(col, df.columns, f"Thiếu cột Tuần 6 {col} ở snapshot {pct}")
                self.assertEqual(df[col].isna().sum(), 0, f"Cột {col} còn NaN ở snapshot {pct}")
            for col in unit_bounded:
                self.assertTrue((df[col] >= -1e-9).all() and (df[col] <= 1.0 + 1e-9).all(), f"{col} ngoài [0, 1] ở {pct}")
            for col in non_negative:
                self.assertTrue((df[col] >= 0).all(), f"{col} có giá trị âm ở {pct}")
            self.assertTrue((df["days_since_last_activity"] >= -1e-9).all())
            has_click = df["last_click_day"].notna()
            np.testing.assert_allclose(
                df.loc[has_click, "days_since_last_activity"].values,
                np.maximum(0.0, df.loc[has_click, "cutoff_day"].values - df.loc[has_click, "last_click_day"].values),
                atol=1e-6,
            )
            np.testing.assert_allclose(
                df.loc[~has_click, "days_since_last_activity"].values,
                df.loc[~has_click, "cutoff_day"].values + 1.0,
                atol=1e-6,
            )
            self.assertTrue((df["max_inactivity_streak"] >= 0.0).all())
            self.assertTrue((df["max_inactivity_streak"] <= np.floor(df["cutoff_day"]) + 1.0 + 1e-6).all())

    def test_14_week6_zero_temporal_leakage(self):
        """Test 14 (Tuần 6): Kiểm chứng Zero Temporal Leakage cho clicks_week_1_to_4, clicks_last_2_weeks & weekend_clicks."""
        vle = self.engineer._vle_enriched
        for pct, df in self.snapshots.items():
            # Tổng weekend_clicks không bao giờ vượt quá total_clicks
            self.assertTrue((df["weekend_clicks"] <= df["total_clicks"] + 1e-6).all())
            # clicks_week_1_to_4 trên 25% và 50% phải bất biến vì đều chứa trọn ngày 0..27
            if pct >= 0.50:
                np.testing.assert_allclose(
                    df["clicks_week_1_to_4"].values,
                    self.snapshots[0.25]["clicks_week_1_to_4"].values,
                    atol=1e-6,
                    err_msg="clicks_week_1_to_4 bị thay đổi giữa mốc 25% và mốc sau -> dấu hiệu rò rỉ!",
                )
            # Kiểm tra đối chiếu clicks_last_2_weeks với bảng vle gốc lọc nghiêm ngặt <= cutoff_day
            cut = vle["module_presentation_length"] * pct
            v_sub = vle[(vle["date"] <= cut) & (vle["date"] > cut - 14.0)]
            expected = v_sub.groupby(self.engineer.KEYS)["sum_click"].sum().reindex(
                pd.MultiIndex.from_frame(df[self.engineer.KEYS])
            ).fillna(0.0).values
            np.testing.assert_allclose(df["clicks_last_2_weeks"].values, expected, atol=1e-6)

    def test_15_week6_score_trajectory_consistency(self):
        """Test 15 (Tuần 6): Kiểm tra last_minus_first_score & score_trend_slope bằng 0 khi < 2 bài nộp và chuẩn OLS."""
        for pct, df in self.snapshots.items():
            lt2 = df["num_submissions"] < 2
            np.testing.assert_allclose(df.loc[lt2, "last_minus_first_score"].values, 0.0, atol=1e-9)
            np.testing.assert_allclose(df.loc[lt2, "score_trend_slope"].values, 0.0, atol=1e-9)
            # days_before_start phải đối xứng với date_registration khi không NaN
            has_reg = df["date_registration"].notna()
            np.testing.assert_allclose(
                df.loc[has_reg, "days_before_start"].values,
                -df.loc[has_reg, "date_registration"].values,
                atol=1e-9,
            )

    def test_16_week6_advanced_matrix_shape_and_zero_leakage(self):
        """Test 16 (Tuần 6): Kiểm tra ma trận week6_advanced đúng 114 biến, bảo toàn dấu clicks_trend và 0 leakage."""
        df_25 = self.snapshots[0.25]
        tr, va, te = train_test_split_group(df_25, split_strategy="group_stratified", random_state=42)
        X_tr, X_va, X_te = self.engineer.prepare_ml_matrices(tr, va, te, feature_set="week6_advanced")

        self.assertEqual(X_tr.shape[1], 114, f"Kỳ vọng 114 cột cho week6_advanced, nhận được {X_tr.shape[1]}")
        for name, mat in [("Train", X_tr), ("Val", X_va), ("Test", X_te)]:
            self.assertEqual(mat.isna().sum().sum(), 0, f"Ma trận {name} week6_advanced còn NaN!")
            self.assertFalse(np.isinf(mat.values).any(), f"Ma trận {name} week6_advanced còn Inf!")

        # clicks_trend phải có giá trị âm và dương (không bị clip [0, high] như total_clicks)
        self.assertTrue((X_tr["clicks_trend"] < 0).any(), "clicks_trend không còn giá trị âm sau chuẩn hóa!")

        # Test Zero Leakage: làm nhiễu Test mạnh và kiểm tra X_tr bất biến
        te_dirty = te.copy()
        te_dirty["clicks_trend"] = 99999.0
        te_dirty["max_inactivity_streak"] = 999.0
        X_tr_2, _, _ = self.engineer.prepare_ml_matrices(tr, va, te_dirty, feature_set="week6_advanced")
        np.testing.assert_allclose(X_tr.values, X_tr_2.values)

    def test_17_week6_catalog_and_early_warning_artifacts(self):
        """Test 17 (Tuần 6): Kiểm tra feature_catalog_week6.csv (92 features), JSON thực nghiệm và 5 biểu đồ Tuần 6."""
        cat6_path = WEEK6_PROCESSED_DIR / "feature_catalog_week6.csv"
        imp6_path = WEEK6_PROCESSED_DIR / "feature_importance_week6.json"
        met6_path = WEEK6_PROCESSED_DIR / "week6_experiment_metrics.json"
        self.assertTrue(cat6_path.exists(), "Thiếu feature_catalog_week6.csv")
        self.assertTrue(imp6_path.exists(), "Thiếu feature_importance_week6.json")
        self.assertTrue(met6_path.exists(), "Thiếu week6_experiment_metrics.json")

        cat6 = pd.read_csv(cat6_path)
        self.assertEqual(len(cat6), 92, f"Kỳ vọng 92 đặc trưng trong catalog Tuần 6, thấy {len(cat6)}")
        self.assertEqual(int(cat6["is_week6_new"].sum()), 16)
        for col in ["point_biserial_r_50", "point_biserial_r_75", "rf_gini_importance_75", "mutual_info_75", "rank_overall_75"]:
            self.assertIn(col, cat6.columns, f"Thiếu cột {col} trong feature_catalog_week6.csv")

        with open(met6_path, "r", encoding="utf-8") as f:
            payload = json.load(f)
        self.assertIn("experiments", payload)
        self.assertIn("early_warning_tuned", payload)
        self.assertEqual(len(payload["early_warning_tuned"]), 36, "Kỳ vọng đủ 36 cấu hình early_warning_tuned (4 mốc x 3 cohort x 3 mô hình)")

        expected_figs = [
            "26_temporal_trend_and_engagement_decay.png",
            "27_inactivity_gap_and_consistency_analysis.png",
            "28_regularity_weekend_and_burst_patterns.png",
            "29_week6_early_warning_breakthrough_25_50_75_100.png",
            "30_week6_feature_importance_evolution.png",
        ]
        for fname in expected_figs:
            self.assertTrue((FIGURES_DIR / fname).exists(), f"Thiếu biểu đồ Tuần 6: {fname}")
        expected_w5_figs = [
            "21_feature_engineering_taxonomy_and_distributions.png",
            "21a_group51_active_days_ratio.png",
            "21b_group54_context_at_risk_rates.png",
            "22_activity_diversity_and_entropy_analysis.png",
            "23_assessment_trajectory_and_weighted_scores.png",
            "24_snapshot_progression_25_50_75_100.png",
            "25_preliminary_feature_importance_ranking.png",
        ]
        for fname in expected_w5_figs:
            self.assertTrue((FIGURES_DIR / fname).exists(), f"Thiếu biểu đồ Tuần 5: {fname}")

    def test_18_week6_early_warning_recall_threshold_breakthrough(self):
        """Test 18 (Tuần 6): Kiểm tra hệ thống Cảnh báo Sớm đạt Test Recall >= 0.75 ngay tại mốc 25% Dynamic Active."""
        met6_path = WEEK6_PROCESSED_DIR / "week6_experiment_metrics.json"
        with open(met6_path, "r", encoding="utf-8") as f:
            payload = json.load(f)
        ew_df = pd.DataFrame(payload["early_warning_tuned"])
        row_25_lgbm = ew_df[
            (ew_df["snapshot"] == "25%")
            & (ew_df["cohort"] == "dynamic_active")
            & (ew_df["model"] == "LightGBM (Balanced)")
        ].iloc[0]
        self.assertGreaterEqual(
            float(row_25_lgbm["test_recall"]),
            0.75,
            f"Test Recall ở mốc 25% Dynamic Active chỉ đạt {row_25_lgbm['test_recall']} (< 0.75)!",
        )
        self.assertGreaterEqual(
            float(row_25_lgbm["test_roc_auc"]),
            0.78,
            f"Test ROC-AUC ở mốc 25% Dynamic Active quá thấp: {row_25_lgbm['test_roc_auc']}",
        )


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestOULADFeatureEngineering)
result = unittest.TextTestRunner(stream=sys.stdout, verbosity=2).run(suite)
assert result.wasSuccessful(), f"Có test thất bại: failures={len(result.failures)}, errors={len(result.errors)}"
print(f"[PASSED] {result.testsRun}/{result.testsRun} kiểm thử Feature Engineering Tuần 5-6 hoàn tất không lỗi!")


test_01_snapshot_files_and_row_counts (__main__.TestOULADFeatureEngineering.test_01_snapshot_files_and_row_counts)
Test 1: Kiểm tra cả 4 lát cắt (25%, 50%, 75%, 100%) đều có đúng 32,593 lượt đăng ký. ... 

ok


test_02_zero_temporal_leakage_vle (__main__.TestOULADFeatureEngineering.test_02_zero_temporal_leakage_vle)
Test 2: Đảm bảo không có tương tác VLE nào vượt quá mốc cắt (last_click_day <= cutoff_day). ... 

ok


test_03_zero_temporal_leakage_assessments (__main__.TestOULADFeatureEngineering.test_03_zero_temporal_leakage_assessments)
Test 3: Đảm bảo chỉ các bài Coursework nộp tại hoặc trước cutoff_day được tính. ... 

ok


... (28 outputs left out)

test_18_week6_early_warning_recall_threshold_breakthrough (__main__.TestOULADFeatureEngineering.test_18_week6_early_warning_recall_threshold_breakthrough)
Test 18 (Tuần 6): Kiểm tra hệ thống Cảnh báo Sớm đạt Test Recall >= 0.75 ngay tại mốc 25% Dynamic Active. ... 

ok


----------------------------------------------------------------------
Ran 18 tests in 19.464s

OK


[PASSED] 18/18 kiểm thử Feature Engineering Tuần 5-6 hoàn tất không lỗi!
